# Pléyade · Eos · cara v3 (SMIRK + HuBERT + boca por regresión + no verbal generativo) — A100

Entorno: **A100, High-RAM**, Drive montado (funciona en T4, más lento y con menos capas de voz).
Todo corre en `/content/v3.sh`, independiente del kernel. Salidas en Drive
`literalevel_cara/voz2cara/v3/`: `fiabilidad_piloto.json`, `smirk/`, `mapa_arkit.json`, `voz/`,
`art/`, `boca/`, `noverbal/` y `estado.txt` (se reescribe cada minuto).

Para Eos en el PC: copiar `v3/boca/{modelo.pt,normalizacion.json}` y `v3/noverbal/{...}` a
`D:/literalevel_cara/v3_modelo/{boca,noverbal}/` y `generar.py --gestos v3`.

In [ ]:
# --- 1. Drive (VS Code pide el permiso en un diálogo)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ARCHIVOS = {"tools/cara_parlante/voz3/smirk_codificador.py": "IiIiRWwgY29kaWZpY2Fkb3IgZGUgU01JUksgKFJldHNpbmFzIGV0IGFsLiwgQ1ZQUiAyMDI0OyBjw7NkaWdvIE1JVCkgc2luIGVsIHJlc3RvIGRlbCByZXBvc2l0b3Jpby4NCg0KU01JUksgcmVjb25zdHJ1eWUgdW5hIGNhcmEgRkxBTUUgYSBwYXJ0aXIgZGUgdW5hIGltYWdlbiByZWNvcnRhZGEgZGUgMjI0w5cyMjQgY29uIHRyZXMNCk1vYmlsZU5ldFYzIChwb3NlK2PDoW1hcmEsIGZvcm1hLCBleHByZXNpw7NuKS4gUGFyYSBFVElRVUVUQVIgc29sbyBoYWNlIGZhbHRhIGVsIGNvZGlmaWNhZG9yOg0KbG9zIHBlc29zIHDDumJsaWNvcyBTTUlSS19lbTEucHQgKEdvb2dsZSBEcml2ZSBkZWwgYXV0b3IsIHNpbiByZWdpc3RybykgeSBgdGltbWAuIEVsIG1vZGVsbw0KRkxBTUUgZGVsIE1QSSAoY29uIHJlZ2lzdHJvKSDDum5pY2FtZW50ZSBoYWNlIGZhbHRhIHBhcmEgcGFzYXIgZGUgcGFyw6FtZXRyb3MgYSBtYWxsYSB5DQpyZW5kZXJpemFyLCB5IGFxdcOtIG5vIHNlIHJlbmRlcml6YSBuYWRhOiBsYSByZWQgdjMgYXByZW5kZSBzb2JyZSBsb3MgcGFyw6FtZXRyb3MgeSBlbCBtYXBhDQphIEFSS2l0IChtYXBhX2Fya2l0LnB5KSBzZSBhanVzdGEgY29uIGxvcyBwYXJlcyBTTUlSS+KGlE1lZGlhUGlwZS4NCg0KU2FsaWRhcyBwb3IgaW1hZ2VuIChsb3MgbWlzbW9zIG5vbWJyZXMgcXVlIGVuIGVsIHJlcG8sIHRyYWR1Y2lkb3MpOg0KICBleHByZXNpb24gWzUwXSAgIGNvZWZpY2llbnRlcyBkZSBleHByZXNpw7NuIEZMQU1FDQogIG1hbmRpYnVsYSBbM10gICAgcm90YWNpw7NuIGRlIGxhIG1hbmTDrWJ1bGEgKGFwZXJ0dXJhIOKJpSAwIHkgZG9zIGdpcm9zIHBlcXVlw7FvcykNCiAgcGFycGFkb3MgIFsyXSAgICBjaWVycmUgZGUgY2FkYSBww6FycGFkbyBlbiBbMCwgMV0NCiAgcG9zZSAgICAgIFszXSAgICByb3RhY2nDs24gZ2xvYmFsIChlamUtw6FuZ3Vsbyk7IGdyYWRvcygpIGxhIHBhc2EgYSBndWnDsWFkYSwgY2FiZWNlbyB5IGFsYWJlbw0KICBjYW0gICAgICAgWzNdICAgIGVzY2FsYSB5IHRyYXNsYWNpw7NuIGVuIGxhIGltYWdlbg0KICBmb3JtYSAgICAgWzMwMF0gIGlkZW50aWRhZCAoY29uc3RhbnRlIGVuIHVuIGNsaXA7IHNlIGd1YXJkYSBsYSBtZWRpYSkNCg0KVXNvOg0KICAgIHJlZCA9IGNhcmdhcignU01JUktfZW0xLnB0JywgJ2N1ZGEnKQ0KICAgIHRmb3JtID0gcmVjb3J0ZShwdW50b3NfeHkpICAgICAgICAgICAgIyBtYXRyaXogMsOXMyBwYXJhIGN2Mi53YXJwQWZmaW5lIGEgMjI0w5cyMjQNCiAgICBzYWxpZGEgPSByZWQobG90ZV9kZV9pbWFnZW5lcykgICAgICAgICMgW0IsIDMsIDIyNCwgMjI0XSBSR0IgZW4gWzAsIDFdDQoiIiINCmltcG9ydCBjdjINCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHRvcmNoDQppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGDQpmcm9tIHRvcmNoIGltcG9ydCBubg0KDQpUQU1BTk8gPSAyMjQNCkVTQ0FMQV9SRUNPUlRFID0gMS40ICAgICAjIGxhIGRlbCBkZW1vIGRlIFNNSVJLDQoNCg0KZGVmIF90cm9uY28obm9tYnJlKToNCiAgICBpbXBvcnQgdGltbQ0KICAgIHJlZCA9IHRpbW0uY3JlYXRlX21vZGVsKG5vbWJyZSwgcHJldHJhaW5lZD1GYWxzZSwgZmVhdHVyZXNfb25seT1UcnVlKQ0KICAgIHJldHVybiByZWQsIHJlZC5mZWF0dXJlX2luZm9bLTFdWydudW1fY2hzJ10NCg0KDQpjbGFzcyBfQ29kaWZpY2Fkb3Iobm4uTW9kdWxlKToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgdHJvbmNvKToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNlbGYuZW5jb2Rlciwgc2VsZi5kID0gX3Ryb25jbyh0cm9uY28pDQoNCiAgICBkZWYgcmFzZ29zKHNlbGYsIGltZyk6DQogICAgICAgIGggPSBzZWxmLmVuY29kZXIoaW1nKVstMV0NCiAgICAgICAgcmV0dXJuIEYuYWRhcHRpdmVfYXZnX3Bvb2wyZChoLCAoMSwgMSkpLmZsYXR0ZW4oMSkNCg0KDQpjbGFzcyBDb2RpZmljYWRvclBvc2UoX0NvZGlmaWNhZG9yKToNCiAgICBkZWYgX19pbml0X18oc2VsZik6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18oJ3RmX21vYmlsZW5ldHYzX3NtYWxsX21pbmltYWxfMTAwJykNCiAgICAgICAgc2VsZi5wb3NlX2NhbV9sYXllcnMgPSBubi5TZXF1ZW50aWFsKG5uLkxpbmVhcihzZWxmLmQsIDYpKQ0KDQogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1nKToNCiAgICAgICAgcGMgPSBzZWxmLnBvc2VfY2FtX2xheWVycyhzZWxmLnJhc2dvcyhpbWcpKQ0KICAgICAgICByZXR1cm4geydwb3NlJzogcGNbOiwgOjNdLCAnY2FtJzogcGNbOiwgMzpdfQ0KDQoNCmNsYXNzIENvZGlmaWNhZG9yRm9ybWEoX0NvZGlmaWNhZG9yKToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgbj0zMDApOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCd0Zl9tb2JpbGVuZXR2M19sYXJnZV9taW5pbWFsXzEwMCcpDQogICAgICAgIHNlbGYuc2hhcGVfbGF5ZXJzID0gbm4uU2VxdWVudGlhbChubi5MaW5lYXIoc2VsZi5kLCBuKSkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIGltZyk6DQogICAgICAgIHJldHVybiB7J2Zvcm1hJzogc2VsZi5zaGFwZV9sYXllcnMoc2VsZi5yYXNnb3MoaW1nKSl9DQoNCg0KY2xhc3MgQ29kaWZpY2Fkb3JFeHByZXNpb24oX0NvZGlmaWNhZG9yKToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgbj01MCk6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18oJ3RmX21vYmlsZW5ldHYzX2xhcmdlX21pbmltYWxfMTAwJykNCiAgICAgICAgc2VsZi5leHByZXNzaW9uX2xheWVycyA9IG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKHNlbGYuZCwgbiArIDIgKyAzKSkNCiAgICAgICAgc2VsZi5uID0gbg0KDQogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1nKToNCiAgICAgICAgcCA9IHNlbGYuZXhwcmVzc2lvbl9sYXllcnMoc2VsZi5yYXNnb3MoaW1nKSkNCiAgICAgICAgbiA9IHNlbGYubg0KICAgICAgICByZXR1cm4geydleHByZXNpb24nOiBwWzosIDpuXSwgJ3BhcnBhZG9zJzogdG9yY2guY2xhbXAocFs6LCBuOm4gKyAyXSwgMCwgMSksDQogICAgICAgICAgICAgICAgJ21hbmRpYnVsYSc6IHRvcmNoLmNhdChbRi5yZWx1KHBbOiwgbiArIDI6biArIDNdKSwgdG9yY2guY2xhbXAocFs6LCBuICsgMzpuICsgNV0sIC0uMiwgLjIpXSwgMSl9DQoNCg0KY2xhc3MgU21pcmsobm4uTW9kdWxlKToNCiAgICBkZWYgX19pbml0X18oc2VsZik6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQ0KICAgICAgICBzZWxmLnBvc2VfZW5jb2RlciA9IENvZGlmaWNhZG9yUG9zZSgpDQogICAgICAgIHNlbGYuc2hhcGVfZW5jb2RlciA9IENvZGlmaWNhZG9yRm9ybWEoKQ0KICAgICAgICBzZWxmLmV4cHJlc3Npb25fZW5jb2RlciA9IENvZGlmaWNhZG9yRXhwcmVzaW9uKCkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIGltZyk6DQogICAgICAgIG91dCA9IHt9DQogICAgICAgIGZvciBtIGluIChzZWxmLnBvc2VfZW5jb2Rlciwgc2VsZi5zaGFwZV9lbmNvZGVyLCBzZWxmLmV4cHJlc3Npb25fZW5jb2Rlcik6DQogICAgICAgICAgICBvdXQudXBkYXRlKG0oaW1nKSkNCiAgICAgICAgcmV0dXJuIG91dA0KDQoNCmRlZiBjYXJnYXIoY2hlY2twb2ludCwgZGlzcG9zaXRpdm89J2NwdScpOg0KICAgICIiIkxlZSBTTUlSS19lbTEucHQgKGd1YXJkYSBjb2RpZmljYWRvciB5IGdlbmVyYWRvcikgeSBkZXZ1ZWx2ZSBzb2xvIGVsIGNvZGlmaWNhZG9yLiIiIg0KICAgIGNrID0gdG9yY2gubG9hZChjaGVja3BvaW50LCBtYXBfbG9jYXRpb249J2NwdScsIHdlaWdodHNfb25seT1UcnVlKQ0KICAgIHBlc29zID0ge2sucmVwbGFjZSgnc21pcmtfZW5jb2Rlci4nLCAnJyk6IHYgZm9yIGssIHYgaW4gY2suaXRlbXMoKSBpZiBrLnN0YXJ0c3dpdGgoJ3NtaXJrX2VuY29kZXIuJyl9DQogICAgcmVkID0gU21pcmsoKQ0KICAgIHJlZC5sb2FkX3N0YXRlX2RpY3QocGVzb3MsIHN0cmljdD1UcnVlKQ0KICAgIHJldHVybiByZWQudG8oZGlzcG9zaXRpdm8pLmV2YWwoKQ0KDQoNCmRlZiBjYWphKHB1bnRvcyk6DQogICAgIiIiQ2VudHJvICh4LCB5KSB5IGxhZG8gZGVsIHJlY29ydGUgY3VhZHJhZG8gYSBwYXJ0aXIgZGUgbG9zIHB1bnRvcyBNZWRpYVBpcGUgWzQ3OCwgMl0gZW4gcMOteGVsZXMuIiIiDQogICAgaXpxLCBkZXIgPSBwdW50b3NbOiwgMF0ubWluKCksIHB1bnRvc1s6LCAwXS5tYXgoKQ0KICAgIGFyciwgYWJhID0gcHVudG9zWzosIDFdLm1pbigpLCBwdW50b3NbOiwgMV0ubWF4KCkNCiAgICBsYWRvID0gKGRlciAtIGl6cSArIGFiYSAtIGFycikgLyAyICogRVNDQUxBX1JFQ09SVEUNCiAgICByZXR1cm4gbnAuYXJyYXkoWyhpenEgKyBkZXIpIC8gMiwgKGFyciArIGFiYSkgLyAyXSksIGZsb2F0KGxhZG8pDQoNCg0KZGVmIHJlY29ydGUoY2VudHJvLCBsYWRvKToNCiAgICAiIiJNYXRyaXogMsOXMyAoY3YyLndhcnBBZmZpbmUpIHF1ZSBsbGV2YSBlbCBjdWFkcmFkbyBkZSBgbGFkb2AgY2VudHJhZG8gZW4gYGNlbnRyb2AgYSAyMjTDlzIyNC4iIiINCiAgICBzID0gKFRBTUFOTyAtIDEpIC8gbGFkbw0KICAgIHJldHVybiBucC5hcnJheShbW3MsIDAsIFRBTUFOTyAvIDIgLSAwLjUgLSBzICogY2VudHJvWzBdXSwgWzAsIHMsIFRBTUFOTyAvIDIgLSAwLjUgLSBzICogY2VudHJvWzFdXV0sIG5wLmZsb2F0MzIpDQoNCg0KZGVmIHJlY29ydGFyKGltZ19iZ3IsIGNlbnRybywgbGFkbyk6DQogICAgIiIiSW1hZ2VuIDIyNMOXMjI0IFJHQiB1aW50OCBsaXN0YSBwYXJhIGxhIHJlZCAocGVybXV0ZSB5IC8yNTUgc2UgaGFjZW4gZW4gZWwgbG90ZSkuIiIiDQogICAgcmVjID0gY3YyLndhcnBBZmZpbmUoaW1nX2JnciwgcmVjb3J0ZShjZW50cm8sIGxhZG8pLCAoVEFNQU5PLCBUQU1BTk8pLCBmbGFncz1jdjIuSU5URVJfTElORUFSKQ0KICAgIHJldHVybiBjdjIuY3Z0Q29sb3IocmVjLCBjdjIuQ09MT1JfQkdSMlJHQikNCg0KDQpkZWYgZ3JhZG9zKHBvc2UpOg0KICAgICIiIkVqZS3DoW5ndWxvIEZMQU1FIFtOLCAzXSDihpIgZ3JhZG9zIChndWnDsWFkYSwgY2FiZWNlbywgYWxhYmVvKSBjb24gbGEgbWlzbWEgY29udmVuY2nDs24gcXVlDQogICAgbWVkaXJfZXhwcmVzaW9uLnBvc2VfZGVfbWF0cml6IChwYXJhIHF1ZSBlbCBtYXBhIGxpbmVhbCBhIGxhIHBvc2UgZGUgTWVkaWFQaXBlIHNlYSBzZW5jaWxsbykuIiIiDQogICAgb3V0ID0gbnAuemVyb3NfbGlrZShwb3NlLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgIGZvciBpLCB2IGluIGVudW1lcmF0ZShwb3NlKToNCiAgICAgICAgciwgXyA9IGN2Mi5Sb2RyaWd1ZXMobnAuYXNhcnJheSh2LCBucC5mbG9hdDY0KSkNCiAgICAgICAgY2FiZWNlbyA9IG5wLmRlZ3JlZXMobnAuYXJjdGFuMihyWzIsIDFdLCByWzIsIDJdKSkNCiAgICAgICAgZ3VpbmFkYSA9IG5wLmRlZ3JlZXMobnAuYXJjc2luKC1ucC5jbGlwKHJbMiwgMF0sIC0xLCAxKSkpDQogICAgICAgIGFsYWJlbyA9IG5wLmRlZ3JlZXMobnAuYXJjdGFuMihyWzEsIDBdLCByWzAsIDBdKSkNCiAgICAgICAgb3V0W2ldID0gKGd1aW5hZGEsIGNhYmVjZW8sIGFsYWJlbykNCiAgICByZXR1cm4gb3V0DQo=", "tools/cara_parlante/voz3/etiquetar_smirk.py": "IiIiRXRpcXVldGFzIG1lam9yZXMgcGFyYSBsYSBjYXJhOiBTTUlSSyAoRkxBTUUpIHBvciBmb3RvZ3JhbWEsIGVuIGxhIG1pc21hIHJlamlsbGEgYHRgIHF1ZSBlbA0KLm5weiBkZSBtZWRpZGFzIGRlIE1lZGlhUGlwZSAobWVkaXJfZXhwcmVzaW9uLnB5KSwgcGFyYSBwb2RlciBjb21wYXJhciB5IGVudHJlbmFyIGNvbiBhbWJhcy4NCg0KUG9yIHbDrWRlbzogZmZtcGVnIGRlY29kaWZpY2EgYSBsYSBmcmVjdWVuY2lhIGRlbCAubnB6ICgyNSBmcHMpIHkgYSAtLWFuY2hvIHDDrXhlbGVzIOKGkiBNZWRpYVBpcGUNCkZhY2VMYW5kbWFya2VyIChtb2RvIHbDrWRlbywgbGEgY2FyYSBtw6FzIGdyYW5kZSkgZGEgbG9zIDQ3OCBwdW50b3Mg4oaSIHJlY29ydGUgY3VhZHJhZG8gw5cxLDQgY29tbw0KZWwgZGVtbyBkZSBTTUlSSywgY29uIGVsIGNlbnRybyB5IGVsIGxhZG8gc3Vhdml6YWRvcyAobWVkaWEgZXhwb25lbmNpYWwpIHBhcmEgcXVlIGVsIHJlY29ydGUgbm8NCnRpZW1ibGUg4oaSIGxvdGVzIGRlIDIyNMOXMjI0IGFsIGNvZGlmaWNhZG9yIFNNSVJLIGVuIGxhIEdQVSDihpIgPHNhbGlkYT4vPG5vbWJyZT4ubnB6IGNvbg0KICB0IFtUXSwgb2sgW1RdICgxIHNpIGh1Ym8gY2FyYSksIGV4cHJlc2lvbiBbVCw1MF0sIG1hbmRpYnVsYSBbVCwzXSwgcGFycGFkb3MgW1QsMl0sDQogIHBvc2UgW1QsM10gZ3JhZG9zIChndWnDsWFkYSwgY2FiZWNlbywgYWxhYmVvKSwgcG9zZV9lamVhbmd1bG8gW1QsM10sIGNhbSBbVCwzXSwgZm9ybWEgWzMwMF0gKG1lZGlhKS4NClJlYW51ZGFibGUgKHNhbHRhIGxvIGhlY2hvOyBlc2NyaWJlIGEgLnRtcCB5IHJlbm9tYnJhKSB5IHJlcGFydGlibGU6IC0tcGFydGUgMi80IGhhY2UgMSBkZSBjYWRhIDQuDQpQcm9ncmVzbyBlbiBlbCBsb2c6IHVuYSBsw61uZWEgcG9yIHbDrWRlbyBhbCBlbXBlemFyLCBjYWRhIDEwICUgeSBhbCBhY2FiYXIgKGZwcyByZWFsZXMpLg0KDQpVc286IHB5dGhvbiBldGlxdWV0YXJfc21pcmsucHkgLS1tZWRpZGFzIERJUl9OUFogLS12aWRlb3MgRElSX01QNCAtLXNhbGlkYSBESVINCiAgICAgICAgLS1jaGVja3BvaW50IFNNSVJLX2VtMS5wdCAtLWxhbmRtYXJrZXIgZmFjZV9sYW5kbWFya2VyLnRhc2sgWy0tZGlzcG9zaXRpdm8gY3VkYV0NCiAgICAgICAgWy0tYW5jaG8gNjQwXSBbLS1sb3RlIDY0XSBbLS1wYXJ0ZSAxLzFdIFstLW1heF9zZWcgMF0gWy0tbGlzdGEgYS5ucHosYi5ucHpdDQpOZWNlc2l0YTogdGltbSwgbWVkaWFwaXBlLCBvcGVuY3YsIGltYWdlaW8tZmZtcGVnLiBOTyBuZWNlc2l0YSBlbCBtb2RlbG8gRkxBTUUgZGVsIE1QSS4NCiIiIg0KaW1wb3J0IGFyZ3BhcnNlDQppbXBvcnQgZ2xvYg0KaW1wb3J0IG9zDQppbXBvcnQgc3VicHJvY2Vzcw0KaW1wb3J0IHN5cw0KaW1wb3J0IHRpbWUNCg0KaW1wb3J0IGN2Mg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgdG9yY2gNCg0Kc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkNCmltcG9ydCBzbWlya19jb2RpZmljYWRvciBhcyBTICAjIG5vcWE6IEU0MDINCg0KDQpkZWYgZnBzX2RlKHQpOg0KICAgIGQgPSBucC5kaWZmKHQpDQogICAgcmV0dXJuIGZsb2F0KDEgLyBucC5tZWRpYW4oZCkpIGlmIGxlbihkKSBlbHNlIDI1LjANCg0KDQpjbGFzcyBMZWN0b3I6DQogICAgIiIiRm90b2dyYW1hcyBCR1IgY29uIGZmbXBlZyBhIGBmcHNgIHkgYGFuY2hvYCBmaWpvcyAoY29tbyBMZWN0b3JGZm1wZWcgZGUgbWVkaXJfZXhwcmVzaW9uLnB5KS4iIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCB2aWRlbywgZnBzLCBhbmNobywgZmZtcGVnKToNCiAgICAgICAgY2FwID0gY3YyLlZpZGVvQ2FwdHVyZSh2aWRlbykNCiAgICAgICAgdywgaCA9IGNhcC5nZXQoY3YyLkNBUF9QUk9QX0ZSQU1FX1dJRFRIKSBvciBhbmNobywgY2FwLmdldChjdjIuQ0FQX1BST1BfRlJBTUVfSEVJR0hUKSBvciBhbmNobw0KICAgICAgICBjYXAucmVsZWFzZSgpDQogICAgICAgIHNlbGYudyA9IGludChtaW4oYW5jaG8sIHcpKSAvLyAyICogMg0KICAgICAgICBzZWxmLmggPSBpbnQocm91bmQoaCAqIHNlbGYudyAvIHcgLyAyKSkgKiAyDQogICAgICAgIHNlbGYucCA9IHN1YnByb2Nlc3MuUG9wZW4oW2ZmbXBlZywgJy1sb2dsZXZlbCcsICdlcnJvcicsICctaScsIHZpZGVvLCAnLXZmJywgZidmcHM9e2Zwc30sc2NhbGU9e3NlbGYud306e3NlbGYuaH0nLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAnLWYnLCAncmF3dmlkZW8nLCAnLXBpeF9mbXQnLCAnYmdyMjQnLCAnLSddLCBzdGRvdXQ9c3VicHJvY2Vzcy5QSVBFKQ0KICAgICAgICBzZWxmLm4gPSBzZWxmLncgKiBzZWxmLmggKiAzDQoNCiAgICBkZWYgX19pdGVyX18oc2VsZik6DQogICAgICAgIHdoaWxlIFRydWU6DQogICAgICAgICAgICBiID0gc2VsZi5wLnN0ZG91dC5yZWFkKHNlbGYubikNCiAgICAgICAgICAgIGlmIGxlbihiKSA8IHNlbGYubjoNCiAgICAgICAgICAgICAgICBicmVhaw0KICAgICAgICAgICAgeWllbGQgbnAuZnJvbWJ1ZmZlcihiLCBucC51aW50OCkucmVzaGFwZShzZWxmLmgsIHNlbGYudywgMykNCiAgICAgICAgc2VsZi5jZXJyYXIoKQ0KDQogICAgZGVmIGNlcnJhcihzZWxmKToNCiAgICAgICAgaWYgc2VsZi5wLnBvbGwoKSBpcyBOb25lOg0KICAgICAgICAgICAgc2VsZi5wLnRlcm1pbmF0ZSgpDQogICAgICAgIHNlbGYucC5zdGRvdXQuY2xvc2UoKTsgc2VsZi5wLndhaXQoKQ0KDQoNCmRlZiBsYW5kbWFya2VyKHJ1dGEpOg0KICAgIGltcG9ydCBtZWRpYXBpcGUgYXMgbXANCiAgICBmcm9tIG1lZGlhcGlwZS50YXNrcyBpbXBvcnQgcHl0aG9uIGFzIG1wX3B5dGhvbg0KICAgIGZyb20gbWVkaWFwaXBlLnRhc2tzLnB5dGhvbiBpbXBvcnQgdmlzaW9uDQogICAgb3B0cyA9IHZpc2lvbi5GYWNlTGFuZG1hcmtlck9wdGlvbnMoYmFzZV9vcHRpb25zPW1wX3B5dGhvbi5CYXNlT3B0aW9ucyhtb2RlbF9hc3NldF9wYXRoPXJ1dGEpLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJ1bm5pbmdfbW9kZT12aXNpb24uUnVubmluZ01vZGUuVklERU8sIG51bV9mYWNlcz0yLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1pbl9mYWNlX2RldGVjdGlvbl9jb25maWRlbmNlPTAuMywgbWluX3RyYWNraW5nX2NvbmZpZGVuY2U9MC4zKQ0KICAgIHJldHVybiBtcCwgdmlzaW9uLkZhY2VMYW5kbWFya2VyLmNyZWF0ZV9mcm9tX29wdGlvbnMob3B0cykNCg0KDQpkZWYgcHVudG9zKG1wLCBsbWssIGltZ19iZ3IsIG1zKToNCiAgICByID0gbG1rLmRldGVjdF9mb3JfdmlkZW8obXAuSW1hZ2UoaW1hZ2VfZm9ybWF0PW1wLkltYWdlRm9ybWF0LlNSR0IsIGRhdGE9Y3YyLmN2dENvbG9yKGltZ19iZ3IsIGN2Mi5DT0xPUl9CR1IyUkdCKSksIG1zKQ0KICAgIGlmIG5vdCByLmZhY2VfbGFuZG1hcmtzOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIGgsIHcgPSBpbWdfYmdyLnNoYXBlWzoyXQ0KICAgIGNhcmFzID0gW25wLmFycmF5KFtbcC54ICogdywgcC55ICogaF0gZm9yIHAgaW4gY10sIG5wLmZsb2F0MzIpIGZvciBjIGluIHIuZmFjZV9sYW5kbWFya3NdDQogICAgcmV0dXJuIG1heChjYXJhcywga2V5PWxhbWJkYSBjOiBucC5wdHAoY1s6LCAwXSkpDQoNCg0KZGVmIGV0aXF1ZXRhcih2aWRlbywgdCwgcmVkLCBtcCwgbG1rLCBhLCBmZm1wZWcsIG5vbWJyZSk6DQogICAgZnBzID0gZnBzX2RlKHQpDQogICAgVCA9IGxlbih0KQ0KICAgIGV4cCwgbWFuZCwgcGFycCwgcG9zZSwgY2FtID0gKG5wLnplcm9zKChULCBuKSwgbnAuZmxvYXQzMikgZm9yIG4gaW4gKDUwLCAzLCAyLCAzLCAzKSkNCiAgICBvayA9IG5wLnplcm9zKFQsIG5wLnVpbnQ4KQ0KICAgIGZvcm1hcyA9IFtdDQogICAgbG90ZSwgaWR4ID0gW10sIFtdDQogICAgY2VudHJvID0gbGFkbyA9IE5vbmUNCiAgICBkZXYgPSBuZXh0KHJlZC5wYXJhbWV0ZXJzKCkpLmRldmljZQ0KICAgIHQwID0gdGltZS50aW1lKCk7IHVsdCA9IC0xDQogICAgdG9wZSA9IGludChhLm1heF9zZWcgKiBmcHMpIGlmIGEubWF4X3NlZyBlbHNlIFQNCiAgICBwcmludChmJ3tub21icmV9OiB7VH0gZm90b2dyYW1hcyBhIHtmcHM6LjFmfSBmcHMnLCBmbHVzaD1UcnVlKQ0KDQogICAgZGVmIHZhY2lhcigpOg0KICAgICAgICBpZiBub3QgbG90ZToNCiAgICAgICAgICAgIHJldHVybg0KICAgICAgICB4ID0gdG9yY2guZnJvbV9udW1weShucC5zdGFjayhsb3RlKSkudG8oZGV2KS5wZXJtdXRlKDAsIDMsIDEsIDIpLmZsb2F0KCkgLyAyNTUNCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCksIHRvcmNoLmF1dG9jYXN0KGRldi50eXBlLCBlbmFibGVkPWRldi50eXBlID09ICdjdWRhJyk6DQogICAgICAgICAgICBvID0gcmVkKHgpDQogICAgICAgIGlpID0gbnAuYXJyYXkoaWR4KQ0KICAgICAgICBleHBbaWldID0gb1snZXhwcmVzaW9uJ10uZmxvYXQoKS5jcHUoKS5udW1weSgpOyBtYW5kW2lpXSA9IG9bJ21hbmRpYnVsYSddLmZsb2F0KCkuY3B1KCkubnVtcHkoKQ0KICAgICAgICBwYXJwW2lpXSA9IG9bJ3BhcnBhZG9zJ10uZmxvYXQoKS5jcHUoKS5udW1weSgpOyBwb3NlW2lpXSA9IG9bJ3Bvc2UnXS5mbG9hdCgpLmNwdSgpLm51bXB5KCkNCiAgICAgICAgY2FtW2lpXSA9IG9bJ2NhbSddLmZsb2F0KCkuY3B1KCkubnVtcHkoKTsgZm9ybWFzLmFwcGVuZChvWydmb3JtYSddLmZsb2F0KCkuY3B1KCkubnVtcHkoKS5tZWFuKDApKQ0KICAgICAgICBva1tpaV0gPSAxDQogICAgICAgIGxvdGUuY2xlYXIoKTsgaWR4LmNsZWFyKCkNCg0KICAgIGxlY3RvciA9IExlY3Rvcih2aWRlbywgZnBzLCBhLmFuY2hvLCBmZm1wZWcpDQogICAgZm9yIGssIGltZyBpbiBlbnVtZXJhdGUobGVjdG9yKToNCiAgICAgICAgaWYgayA+PSBtaW4oVCwgdG9wZSk6DQogICAgICAgICAgICBsZWN0b3IuY2VycmFyKCkNCiAgICAgICAgICAgIGJyZWFrDQogICAgICAgIHB0cyA9IE5vbmUgaWYgKGEuY2FkYSA+IDEgYW5kIGsgJSBhLmNhZGEgYW5kIGNlbnRybyBpcyBub3QgTm9uZSkgZWxzZSBwdW50b3MobXAsIGxtaywgaW1nLCBpbnQoayAqIDEwMDAgLyBmcHMpKQ0KICAgICAgICBpZiBwdHMgaXMgbm90IE5vbmU6DQogICAgICAgICAgICBjLCBsID0gUy5jYWphKHB0cykNCiAgICAgICAgICAgIGNlbnRybywgbGFkbyA9IChjLCBsKSBpZiBjZW50cm8gaXMgTm9uZSBlbHNlICgwLjQgKiBjZW50cm8gKyAwLjYgKiBjLCAwLjQgKiBsYWRvICsgMC42ICogbCkNCiAgICAgICAgZWxpZiBjZW50cm8gaXMgTm9uZSBvciAoYS5jYWRhIDw9IDEpOg0KICAgICAgICAgICAgY2VudHJvID0gTm9uZQ0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgbG90ZS5hcHBlbmQoUy5yZWNvcnRhcihpbWcsIGNlbnRybywgbGFkbykpOyBpZHguYXBwZW5kKGspDQogICAgICAgIGlmIGxlbihsb3RlKSA+PSBhLmxvdGU6DQogICAgICAgICAgICB2YWNpYXIoKQ0KICAgICAgICBkZWMgPSAoMTAgKiBrKSAvLyBtYXgoMSwgbWluKFQsIHRvcGUpKQ0KICAgICAgICBpZiBkZWMgIT0gdWx0IGFuZCBrOg0KICAgICAgICAgICAgdWx0ID0gZGVjDQogICAgICAgICAgICBwcmludChmJyAge25vbWJyZX0ge2t9L3tUfSAoezEwICogZGVjfSAlKSB7ayAvICh0aW1lLnRpbWUoKSAtIHQwKTouMGZ9IGZwcycsIGZsdXNoPVRydWUpDQogICAgdmFjaWFyKCkNCiAgICBmb3JtYSA9IG5wLm1lYW4oZm9ybWFzLCAwKSBpZiBmb3JtYXMgZWxzZSBucC56ZXJvcygzMDAsIG5wLmZsb2F0MzIpDQogICAgcHJpbnQoZicgIHtub21icmV9IGxpc3RvOiBjYXJhIGVuIHtvay5tZWFuKCk6LjAlfSBkZSBsb3MgZm90b2dyYW1hcywge1QgLyAodGltZS50aW1lKCkgLSB0MCk6LjBmfSBmcHMnLCBmbHVzaD1UcnVlKQ0KICAgIHJldHVybiBkaWN0KHQ9dCwgb2s9b2ssIGV4cHJlc2lvbj1leHAsIG1hbmRpYnVsYT1tYW5kLCBwYXJwYWRvcz1wYXJwLCBwb3NlPVMuZ3JhZG9zKHBvc2UpLCBwb3NlX2VqZWFuZ3Vsbz1wb3NlLA0KICAgICAgICAgICAgICAgIGNhbT1jYW0sIGZvcm1hPWZvcm1hLmFzdHlwZShucC5mbG9hdDMyKSkNCg0KDQpkZWYgbWFpbigpOg0KICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgnLS1tZWRpZGFzJywgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tdmlkZW9zJywgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tc2FsaWRhJywgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tY2hlY2twb2ludCcsIHJlcXVpcmVkPVRydWUpDQogICAgYXAuYWRkX2FyZ3VtZW50KCctLWxhbmRtYXJrZXInLCByZXF1aXJlZD1UcnVlLCBoZWxwPSdmYWNlX2xhbmRtYXJrZXIudGFzayBkZSBNZWRpYVBpcGUnKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgnLS1kaXNwb3NpdGl2bycsIGRlZmF1bHQ9J2N1ZGEnIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAnY3B1JykNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tYW5jaG8nLCB0eXBlPWludCwgZGVmYXVsdD02NDApDQogICAgYXAuYWRkX2FyZ3VtZW50KCctLWxvdGUnLCB0eXBlPWludCwgZGVmYXVsdD02NCkNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tY2FkYScsIHR5cGU9aW50LCBkZWZhdWx0PTEsIGhlbHA9J01lZGlhUGlwZSBjYWRhIE4gZm90b2dyYW1hcyAoZWwgcmVjb3J0ZSBzZSBtYW50aWVuZSBlbnRyZSBtZWRpYXMpJykNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tcGFydGUnLCBkZWZhdWx0PScxLzEnLCBoZWxwPSdpL246IGVzdGUgcHJvY2VzbyBoYWNlIGxvcyB2w61kZW9zIGktMSBtw7NkdWxvIG4nKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgnLS1tYXhfc2VnJywgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLCBoZWxwPSdzb2xvIGxvcyBwcmltZXJvcyBzZWd1bmRvcyAocHJ1ZWJhcyknKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgnLS1saXN0YScsIGRlZmF1bHQ9JycsIGhlbHA9J3NvbG8gZXN0b3Mgbm9tYnJlcyAoc2VwYXJhZG9zIHBvciBjb21hKScpDQogICAgYSA9IGFwLnBhcnNlX2FyZ3MoKQ0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IGltYWdlaW9fZmZtcGVnDQogICAgICAgIGZmbXBlZyA9IGltYWdlaW9fZmZtcGVnLmdldF9mZm1wZWdfZXhlKCkNCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6DQogICAgICAgIGZmbXBlZyA9ICdmZm1wZWcnDQogICAgb3MubWFrZWRpcnMoYS5zYWxpZGEsIGV4aXN0X29rPVRydWUpDQogICAgaV9wYXJ0ZSwgbl9wYXJ0ZSA9IChpbnQoeCkgZm9yIHggaW4gYS5wYXJ0ZS5zcGxpdCgnLycpKQ0KICAgIHNvbG8gPSBzZXQoYS5saXN0YS5zcGxpdCgnLCcpKSBpZiBhLmxpc3RhIGVsc2UgTm9uZQ0KICAgIHZpZGVvcyA9IHtvcy5wYXRoLnNwbGl0ZXh0KG9zLnBhdGguYmFzZW5hbWUodikpWzBdOiB2IGZvciBleHQgaW4gKCdtcDQnLCAnbWt2JywgJ3dlYm0nKQ0KICAgICAgICAgICAgICBmb3IgdiBpbiBnbG9iLmdsb2Iob3MucGF0aC5qb2luKGEudmlkZW9zLCAnKionLCAnKi4nICsgZXh0KSwgcmVjdXJzaXZlPVRydWUpfQ0KICAgIHJlZCA9IFMuY2FyZ2FyKGEuY2hlY2twb2ludCwgYS5kaXNwb3NpdGl2bykNCiAgICB0b2RvcyA9IHNvcnRlZChnbG9iLmdsb2Iob3MucGF0aC5qb2luKGEubWVkaWRhcywgJyoqJywgJyoubnB6JyksIHJlY3Vyc2l2ZT1UcnVlKSkNCiAgICBoZWNob3MgPSBzYWx0YWRvcyA9IDANCiAgICBmb3IgaiwgbnB6IGluIGVudW1lcmF0ZSh0b2Rvcyk6DQogICAgICAgIG5vbWJyZSA9IG9zLnBhdGguc3BsaXRleHQob3MucGF0aC5iYXNlbmFtZShucHopKVswXQ0KICAgICAgICBpZiBqICUgbl9wYXJ0ZSAhPSBpX3BhcnRlIC0gMSBvciAoc29sbyBhbmQgbm9tYnJlIG5vdCBpbiBzb2xvKToNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIGRlc3QgPSBvcy5wYXRoLmpvaW4oYS5zYWxpZGEsIG5vbWJyZSArICcubnB6JykNCiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoZGVzdCk6DQogICAgICAgICAgICBzYWx0YWRvcyArPSAxDQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBpZiBub21icmUgbm90IGluIHZpZGVvczoNCiAgICAgICAgICAgIHByaW50KCdzaW4gdsOtZGVvJywgbm9tYnJlLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgdCA9IG5wLmxvYWQobnB6KVsndCddDQogICAgICAgIHByaW50KGYnW3tqICsgMX0ve2xlbih0b2Rvcyl9XScsIGVuZD0nICcsIGZsdXNoPVRydWUpDQogICAgICAgIHRyeToNCiAgICAgICAgICAgICMgTWVkaWFQaXBlIGVuIG1vZG8gdsOtZGVvIGV4aWdlIHRpZW1wb3MgY3JlY2llbnRlczogdW4gbGFuZG1hcmtlciBudWV2byBwb3IgdsOtZGVvDQogICAgICAgICAgICBtcCwgbG1rID0gbGFuZG1hcmtlcihhLmxhbmRtYXJrZXIpDQogICAgICAgICAgICBkID0gZXRpcXVldGFyKHZpZGVvc1tub21icmVdLCB0LCByZWQsIG1wLCBsbWssIGEsIGZmbXBlZywgbm9tYnJlKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6DQogICAgICAgICAgICBwcmludCgnRVJST1InLCBub21icmUsIHJlcHIoZSksIGZsdXNoPVRydWUpDQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBucC5zYXZlel9jb21wcmVzc2VkKGRlc3QgKyAnLnRtcC5ucHonLCAqKmQpDQogICAgICAgIG9zLnJlcGxhY2UoZGVzdCArICcudG1wLm5weicsIGRlc3QpDQogICAgICAgIGhlY2hvcyArPSAxDQogICAgcHJpbnQoZidsaXN0bzoge2hlY2hvc30gZXRpcXVldGFkb3MsIHtzYWx0YWRvc30geWEgZXN0YWJhbicsIGZsdXNoPVRydWUpDQoNCg0KaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoNCiAgICBtYWluKCkNCg==", "tools/cara_parlante/voz3/datos_v3.py": "IiIiTG8gcXVlIGNvbXBhcnRlbiBsb3MgZ3Vpb25lcyBkZSB2MzogZMOzbmRlIGVzdMOhbiBsb3MgZGF0b3MsIGPDs21vIHNlIGVtcGFyZWphbiB5IHF1w6kgY29sdW1uYXMgaGF5Lg0KDQotIEV0aXF1ZXRhcyBTTUlSSyAoZXRpcXVldGFyX3NtaXJrLnB5KTogcmFzZ29zKCkgYXBpbGEgZXhwcmVzacOzbiAoNTApLCBtYW5kw61idWxhICgzKSwgcMOhcnBhZG9zICgyKQ0KICB5IHBvc2UgZW4gZ3JhZG9zICgzKSDihpIgW1QsIDU4XSBlbiBlbCBvcmRlbiBkZSBFTlRSQURBU19TTUlSSy4NCi0gTWVkaWRhcyBNZWRpYVBpcGUgKG1lZGlyX2V4cHJlc2lvbi5weSk6IDUyIGJsZW5kc2hhcGVzICsgcG9zZSAoMykg4oaSIFtULCA1NV07IENPTFVNTkFTX0FSS0lULg0KLSBHUlVQT1M6IHF1w6kgY29sdW1uYXMgZm9ybWFuIGJvY2EsIHNvbnJpc2EsIG9qb3MsIGNlamFzIHkgY2FiZXphIChsb3MgbWlzbW9zIHF1ZSBsYSByZWQgZGUgZnVzacOzbikuDQotIHBhcmVzXyooKTogZW1wYXJlamFuIGFyY2hpdm9zIHBvciBub21icmUgKDxub21icmU+Lm5weiBlbiBjYWRhIGNhcnBldGEsIHJlY3Vyc2l2bykuDQoiIiINCmltcG9ydCBnbG9iDQppbXBvcnQgb3MNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQoNCkdSVVBPUyA9IHsNCiAgICAnYm9jYSc6IFsnamF3T3BlbicsICdtb3V0aENsb3NlJywgJ21vdXRoRnVubmVsJywgJ21vdXRoUHVja2VyJywgJ21vdXRoUHJlc3NMZWZ0JywgJ21vdXRoUHJlc3NSaWdodCcsDQogICAgICAgICAgICAgJ21vdXRoUm9sbExvd2VyJywgJ21vdXRoUm9sbFVwcGVyJywgJ21vdXRoTG93ZXJEb3duTGVmdCcsICdtb3V0aExvd2VyRG93blJpZ2h0JywNCiAgICAgICAgICAgICAnbW91dGhVcHBlclVwTGVmdCcsICdtb3V0aFVwcGVyVXBSaWdodCcsICdtb3V0aFN0cmV0Y2hMZWZ0JywgJ21vdXRoU3RyZXRjaFJpZ2h0J10sDQogICAgJ3NvbnJpc2EnOiBbJ21vdXRoU21pbGVMZWZ0JywgJ21vdXRoU21pbGVSaWdodCcsICdjaGVla1NxdWludExlZnQnLCAnY2hlZWtTcXVpbnRSaWdodCddLA0KICAgICdvam9zJzogWydleWVCbGlua0xlZnQnLCAnZXllQmxpbmtSaWdodCcsICdleWVTcXVpbnRMZWZ0JywgJ2V5ZVNxdWludFJpZ2h0JywgJ2V5ZVdpZGVMZWZ0JywgJ2V5ZVdpZGVSaWdodCddLA0KICAgICdjZWphcyc6IFsnYnJvd0lubmVyVXAnLCAnYnJvd091dGVyVXBMZWZ0JywgJ2Jyb3dPdXRlclVwUmlnaHQnLCAnYnJvd0Rvd25MZWZ0JywgJ2Jyb3dEb3duUmlnaHQnXSwNCiAgICAnY2FiZXphJzogWydjYWJlemFfZ3VpbmFkYScsICdjYWJlemFfY2FiZWNlbycsICdjYWJlemFfYWxhYmVvJ10sDQp9DQojIGxvIHF1ZSBtdWV2ZSBsYSBwYXJ0ZSBnZW5lcmF0aXZhIChubyB2ZXJiYWwpOyBsYSBib2NhIGVzIGxvIGRlIEdSVVBPU1snYm9jYSddDQpOT19WRVJCQUwgPSBHUlVQT1NbJ3NvbnJpc2EnXSArIEdSVVBPU1snb2pvcyddICsgR1JVUE9TWydjZWphcyddICsgR1JVUE9TWydjYWJlemEnXQ0KRU5UUkFEQVNfU01JUksgPSBbZidleHB7aTowMmR9JyBmb3IgaSBpbiByYW5nZSg1MCldICsgWydtYW5kX2FicmlyJywgJ21hbmRfbGFkbycsICdtYW5kX2dpcm8nLCAncGFycGFkb19penEnLCAncGFycGFkb19kZXInLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAncG9zZV9ndWluYWRhJywgJ3Bvc2VfY2FiZWNlbycsICdwb3NlX2FsYWJlbyddDQoNCg0KZGVmIG1hcGFfYXJjaGl2b3MoZCwgZXh0PSducHonKToNCiAgICByZXR1cm4ge29zLnBhdGguc3BsaXRleHQob3MucGF0aC5iYXNlbmFtZShmKSlbMF06IGYgZm9yIGYgaW4gZ2xvYi5nbG9iKG9zLnBhdGguam9pbihkLCAnKionLCAnKi4nICsgZXh0KSwgcmVjdXJzaXZlPVRydWUpfQ0KDQoNCmRlZiByYXNnb3Mocyk6DQogICAgIiIiW1QsIDU4XSBmbG9hdDMyIGEgcGFydGlyIGRlIHVuIG5weiBkZSBldGlxdWV0YXJfc21pcmsucHkgKG8gZGUgdW4gZGljdCBjb24gbGFzIG1pc21hcyBjbGF2ZXMpLiIiIg0KICAgIHJldHVybiBucC5jb25jYXRlbmF0ZShbc1snZXhwcmVzaW9uJ10sIHNbJ21hbmRpYnVsYSddLCBzWydwYXJwYWRvcyddLCBzWydwb3NlJ11dLCAxKS5hc3R5cGUobnAuZmxvYXQzMikNCg0KDQpkZWYgb2JqZXRpdm9fbWVkaWFwaXBlKG0pOg0KICAgICIiIltULCA1NV06IDUyIGJsZW5kc2hhcGVzICsgMyDDoW5ndWxvcywgeSBzdXMgbm9tYnJlcyAoQ09MVU1OQVNfQVJLSVQpLiIiIg0KICAgIHkgPSBucC5jb25jYXRlbmF0ZShbbVsnYnMnXS5hc3R5cGUobnAuZmxvYXQzMiksIG5wLm5hbl90b19udW0obVsncG9zZSddLmFzdHlwZShucC5mbG9hdDMyKSldLCAxKQ0KICAgIHJldHVybiB5LCBsaXN0KG1bJ2JzX25vbWJyZXMnXSkgKyBHUlVQT1NbJ2NhYmV6YSddDQoNCg0KZGVmIHBhcmVzX3NtaXJrKGRpcl9zbWlyaywgZGlyX21lZGlkYXMpOg0KICAgIHMsIG0gPSBtYXBhX2FyY2hpdm9zKGRpcl9zbWlyayksIG1hcGFfYXJjaGl2b3MoZGlyX21lZGlkYXMpDQogICAgcmV0dXJuIFsobiwgc1tuXSwgbVtuXSkgZm9yIG4gaW4gc29ydGVkKG0pIGlmIG4gaW4gc10NCg0KDQpkZWYgY2FyZ2FyX3BhcihmcywgZm0sIG1pbl9vaz0wLjUpOg0KICAgICIiIlggW1QsNThdLCBZIFtULDU1XSwgb2sgKGNhcmEgZW4gbG9zIGRvcyksIGNvbHVtbmFzOyBOb25lIHNpIGVsIGNsaXAgZXMgbWFsby4iIiINCiAgICBzLCBtID0gbnAubG9hZChmcyksIG5wLmxvYWQoZm0pDQogICAgayA9IG1pbihsZW4oc1sndCddKSwgbGVuKG1bJ3QnXSkpDQogICAgb2sgPSBzWydvayddWzprXS5hc3R5cGUoYm9vbCkgJiBtWydvayddWzprXS5hc3R5cGUoYm9vbCkNCiAgICBpZiBvay5tZWFuKCkgPCBtaW5fb2sgb3Igb2suc3VtKCkgPCA1MDoNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICB5LCBjb2xzID0gb2JqZXRpdm9fbWVkaWFwaXBlKG0pDQogICAgcmV0dXJuIHJhc2dvcyhzKVs6a10sIHlbOmtdLCBvaywgY29scw0KDQoNCmRlZiBhcGlsYXIocGFyZXMsIG1pbl9vaz0wLjUpOg0KICAgICIiIlRvZG9zIGxvcyBmb3RvZ3JhbWFzIHbDoWxpZG9zIGRlIGxvcyBwYXJlczogWCwgWSwgaWQgZGUgY2xpcCBwb3IgZm90b2dyYW1hLCBjb2x1bW5hcywgbm9tYnJlcy4iIiINCiAgICBYcywgWXMsIGlkcywgY29scywgbm9tYnJlcyA9IFtdLCBbXSwgW10sIE5vbmUsIFtdDQogICAgZm9yIG4sIGZzLCBmbSBpbiBwYXJlczoNCiAgICAgICAgciA9IGNhcmdhcl9wYXIoZnMsIGZtLCBtaW5fb2spDQogICAgICAgIGlmIHIgaXMgTm9uZToNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHgsIHksIG9rLCBjb2xzID0gcg0KICAgICAgICBYcy5hcHBlbmQoeFtva10pOyBZcy5hcHBlbmQoeVtva10pOyBpZHMuYXBwZW5kKG5wLmZ1bGwob2suc3VtKCksIGxlbihub21icmVzKSkpOyBub21icmVzLmFwcGVuZChuKQ0KICAgIHJldHVybiBucC5jb25jYXRlbmF0ZShYcyksIG5wLmNvbmNhdGVuYXRlKFlzKSwgbnAuY29uY2F0ZW5hdGUoaWRzKSwgY29scywgbm9tYnJlcw0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBkYXRvcyBkZSBlbnRyZW5hbWllbnRvIHYzDQpMID0gMjUwICAgICAgICAgICAgIyB2ZW50YW5hIChmb3RvZ3JhbWFzIGEgMjUgZnBzID0gMTAgcykNClBBU08gPSAxMjUgICAgICAgICAjIHNvbGFwZSBkZSBtZWRpYSB2ZW50YW5hDQpOX0ZPTiA9IDM1ICAgICAgICAgIyAzNCBmb25lbWFzIGRlbCBtb2RlbG8gYXJ0aWN1bGF0b3JpbyArIHNpbGVuY2lvICjDrW5kaWNlIDM0KQ0KDQoNCmRlZiBlc192YWwoaW5kaWNlLCBub21icmUsIHJ1dGEpOg0KICAgICIiIk1pc21vIHJlcGFydG8gcXVlIGxhIHJlZCBkZSBmdXNpw7NuOiBsYSBjYXJwZXRhIHZhbCwgMSBkZSBjYWRhIDggdsOtZGVvcyBkZSBZb3VUdWJlIChwb3INCiAgICBoYXNoIGRlbCB2w61kZW8sIGNvbiBzdXMgdHJvem9zKSB5IDEgZGUgY2FkYSA4IGRlIE9iYW1hL0Nhc2EgQmxhbmNhIChwb3Igw61uZGljZSkuIiIiDQogICAgaW1wb3J0IHpsaWINCiAgICByID0gcnV0YS5yZXBsYWNlKCcvJywgb3Muc2VwKQ0KICAgIGlmIG9zLnNlcCArICd2YWwnICsgb3Muc2VwIGluIHI6DQogICAgICAgIHJldHVybiBUcnVlDQogICAgaWYgYW55KGsgaW4gciBmb3IgayBpbiAoJ29iYW1hJywgJ2Nhc2FfYmxhbmNhJywgJ21pcmFsbGVzJykpOg0KICAgICAgICByZXR1cm4gaW5kaWNlICUgOCA9PSAwDQogICAgcmV0dXJuIHpsaWIuY3JjMzIobm9tYnJlLnJzcGxpdCgnXycsIDEpWzBdLmVuY29kZSgpKSAlIDggPT0gMA0KDQoNCmRlZiBwYXJlc19lbnRyZW5hbWllbnRvKGRpcl92b3osIGRpcl9hcnQsIGRpcl9zbWlyayk6DQogICAgIiIiKG5vbWJyZSwgdm96Lm5weiwgYXJ0Lm5weiwgc21pcmsubnB6KSBkZSBsb3MgY2xpcHMgcXVlIHRpZW5lbiBsYXMgdHJlcyBjb3Nhcy4iIiINCiAgICB2LCBhLCBzID0gbWFwYV9hcmNoaXZvcyhkaXJfdm96KSwgbWFwYV9hcmNoaXZvcyhkaXJfYXJ0KSwgbWFwYV9hcmNoaXZvcyhkaXJfc21pcmspDQogICAgcmV0dXJuIFsobiwgdltuXSwgYVtuXSwgc1tuXSkgZm9yIG4gaW4gc29ydGVkKHMpIGlmIG4gaW4gdiBhbmQgbiBpbiBhXQ0KDQoNCmRlZiBjYXJnYXJfY2xpcChmdiwgZmEsIGZzLCBtYXBhLCBjYXBhc191c2FyPU5vbmUsIG1pbl9vaz0wLjYpOg0KICAgICIiIlVuIGNsaXAgcGFyYSBlbnRyZW5hcjogdm96IFtULEssMTAyNF0gZjE2LCBhcnQgW1QsMTNdIGYxNiwgZm9uIFtUXSBpNjQsIG9iamV0aXZvIFtULDU1XSBmMzINCiAgICAoQVJLaXQgdsOtYSBlbCBtYXBhIFNNSVJL4oaSQVJLaXQsIG1lbm9zIGxhIG1lZGlhIGRlbCBjbGlwKSwgb2sgW1RdIGJvb2wsIGR1cmFjacOzbiAocykuIE5vbmUgc2kgZXMgbWFsby4iIiINCiAgICBpbXBvcnQgbWFwYV9hcmtpdA0KICAgIHMsIHYsIGEgPSBucC5sb2FkKGZzKSwgbnAubG9hZChmdiksIG5wLmxvYWQoZmEpDQogICAgb2sgPSBzWydvayddLmFzdHlwZShib29sKQ0KICAgIGsgPSBtaW4obGVuKG9rKSwgbGVuKHZbJ3QnXSksIGxlbihhWyd0J10pKQ0KICAgIG9rID0gb2tbOmtdDQogICAgaWYgb2subWVhbigpIDwgbWluX29rIG9yIGsgPCA3NToNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICB5ID0gbWFwYV9hcmtpdC5hcGxpY2FyKG1hcGEsIHJhc2dvcyhzKVs6a10pDQogICAgeSA9IHkgLSB5W29rXS5tZWFuKDApDQogICAgeVt+b2tdID0gMA0KICAgIHZveiA9IHZbJ3ZveiddWzprXQ0KICAgIGlmIGNhcGFzX3VzYXIgaXMgbm90IE5vbmU6DQogICAgICAgIGRpc3BvbmlibGVzID0gbGlzdCh2WydjYXBhcyddKQ0KICAgICAgICB2b3ogPSB2b3pbOiwgW2Rpc3BvbmlibGVzLmluZGV4KGMpIGZvciBjIGluIGNhcGFzX3VzYXJdXQ0KICAgIGZvbiA9IGFbJ2ZvbiddWzprXS5hc3R5cGUobnAuaW50NjQpDQogICAgZm9uW2ZvbiA8IDBdID0gTl9GT04gLSAxDQogICAgcmV0dXJuIGRpY3Qodm96PXZvei5hc3R5cGUobnAuZmxvYXQxNiksIGFydD1hWydhcnQnXVs6a10uYXN0eXBlKG5wLmZsb2F0MTYpLCBmb249Zm9uLCB5PXkuYXN0eXBlKG5wLmZsb2F0MzIpLCBvaz1vaywNCiAgICAgICAgICAgICAgICBkdXI9ZmxvYXQoc1sndCddW2sgLSAxXSAtIHNbJ3QnXVswXSkpDQoNCg0KY2xhc3MgQ29uanVudG86DQogICAgIiIiTG9zIGRhdG9zIGRlIGVudHJlbmFtaWVudG8gYXBpbGFkb3MgZW4gdmVudGFuYXMgZGUgTCBmb3RvZ3JhbWFzLCBsaXN0b3MgcGFyYSBpbmRleGFyLg0KDQogICAgTGEgdm96IChsYSBwYXJ0ZSBnb3JkYTogfjM2IEdCIGNvbiA3IGNhcGFzIGVuIDI4IGgpIHZhIGEgdW4gbWVtbWFwIGVuIGBkaXJfY2FjaGVgIChkaXNjbyBsb2NhbA0KICAgIGVuIENvbGFiOyBlbCBzaXN0ZW1hIGxvIG1hbnRpZW5lIGVuIGxhIGNhY2jDqSBkZSBSQU0gc2kgY2FiZSkgeSwgc2kgc2UgcGlkZSB5IGNhYmUsIGEgbGEgR1BVLg0KICAgIExvIGRlbcOhcyAoYXJ0aWN1bGFjacOzbiwgZm9uZW1hLCBvYmpldGl2bywgbcOhc2NhcmEpIHZhIGEgbGEgUkFNIGNvbW8gdGVuc29yZXMuDQogICAgTG9zIGNsaXBzIGRlIHZhbGlkYWNpw7NuIHNlIGd1YXJkYW4gZW50ZXJvcyBlbiBgdmFsYCAobGlzdGEgZGUgZGljdHMgZGUgY2FyZ2FyX2NsaXApLiIiIg0KDQogICAgZGVmIF9faW5pdF9fKHNlbGYsIHBhcmVzLCBtYXBhLCBkaXJfY2FjaGUsIGNhcGFzX3VzYXI9Tm9uZSwgbWF4X2hvcmFzPTAuMCwgbG9nPXByaW50KToNCiAgICAgICAgaW1wb3J0IHRvcmNoDQogICAgICAgIG9zLm1ha2VkaXJzKGRpcl9jYWNoZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICAgICAgZWxlZ2lkb3MsIHNlbGYudmFsLCBob3JhcyA9IFtdLCBbXSwgMC4wDQogICAgICAgIGZvciBpLCAobiwgZnYsIGZhLCBmcykgaW4gZW51bWVyYXRlKHBhcmVzKToNCiAgICAgICAgICAgIGlmIGVzX3ZhbChpLCBuLCBmcyk6DQogICAgICAgICAgICAgICAgYyA9IGNhcmdhcl9jbGlwKGZ2LCBmYSwgZnMsIG1hcGEsIGNhcGFzX3VzYXIpDQogICAgICAgICAgICAgICAgaWYgYyBpcyBub3QgTm9uZToNCiAgICAgICAgICAgICAgICAgICAgY1snbm9tYnJlJ10gPSBuDQogICAgICAgICAgICAgICAgICAgIHNlbGYudmFsLmFwcGVuZChjKQ0KICAgICAgICAgICAgZWxzZToNCiAgICAgICAgICAgICAgICBlbGVnaWRvcy5hcHBlbmQoKG4sIGZ2LCBmYSwgZnMpKQ0KICAgICAgICBpZiBub3Qgc2VsZi52YWwgYW5kIGxlbihlbGVnaWRvcykgPiAxOiAgICAgICAgICAgICMgcHJ1ZWJhcyBjb24gbXV5IHBvY29zIGNsaXBzOiBlbCDDumx0aW1vIGhhY2UgZGUgdmFsaWRhY2nDs24NCiAgICAgICAgICAgIG4sIGZ2LCBmYSwgZnMgPSBlbGVnaWRvcy5wb3AoKQ0KICAgICAgICAgICAgYyA9IGNhcmdhcl9jbGlwKGZ2LCBmYSwgZnMsIG1hcGEsIGNhcGFzX3VzYXIpDQogICAgICAgICAgICBpZiBjIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgICAgIGNbJ25vbWJyZSddID0gbjsgc2VsZi52YWwuYXBwZW5kKGMpDQogICAgICAgICMgdGFtYcOxbyB0b3RhbCBwYXJhIGVsIG1lbW1hcDogc29sbyBoYWNlIGZhbHRhIGxhIGxvbmdpdHVkIGRlIGNhZGEgY2xpcA0KICAgICAgICBsYXJnb3MgPSBbXQ0KICAgICAgICBmb3IgbiwgZnYsIGZhLCBmcyBpbiBlbGVnaWRvczoNCiAgICAgICAgICAgIHMgPSBucC5sb2FkKGZzKQ0KICAgICAgICAgICAgbGFyZ29zLmFwcGVuZChtaW4obGVuKHNbJ3QnXSksIGxlbihucC5sb2FkKGZ2KVsndCddKSkpDQogICAgICAgIHRvdGFsID0gc3VtKG1heChsZywgTCkgZm9yIGxnIGluIGxhcmdvcykNCiAgICAgICAgazAgPSBucC5sb2FkKGVsZWdpZG9zWzBdWzFdKVsndm96J10uc2hhcGVbMV0gaWYgY2FwYXNfdXNhciBpcyBOb25lIGVsc2UgbGVuKGNhcGFzX3VzYXIpDQogICAgICAgIHJ1dGEgPSBvcy5wYXRoLmpvaW4oZGlyX2NhY2hlLCBmJ3Zvel97dG90YWx9X3trMH0ubnB5JykNCiAgICAgICAgbnVldm8gPSBub3Qgb3MucGF0aC5leGlzdHMocnV0YSArICcub2snKSAgICAgICAgICAjIGxhIG1hcmNhIC5vayBzZSBlc2NyaWJlIGFsIGFjYWJhciBkZSBsbGVuYXJsbw0KICAgICAgICBzZWxmLnZveiA9IG5wLmxpYi5mb3JtYXQub3Blbl9tZW1tYXAocnV0YSwgbW9kZT0ndysnIGlmIG51ZXZvIGVsc2UgJ3InLCBkdHlwZT1ucC5mbG9hdDE2LCBzaGFwZT0odG90YWwsIGswLCAxMDI0KSkNCiAgICAgICAgWGEgPSBucC56ZXJvcygodG90YWwsIDEzKSwgbnAuZmxvYXQxNik7IFhmID0gbnAuZnVsbCh0b3RhbCwgTl9GT04gLSAxLCBucC5pbnQ2NCkNCiAgICAgICAgWXMsIE1zLCBpbmljaW9zLCBwb3MgPSBbXSwgW10sIFtdLCAwDQogICAgICAgIGZvciAobiwgZnYsIGZhLCBmcyksIGxnIGluIHppcChlbGVnaWRvcywgbGFyZ29zKToNCiAgICAgICAgICAgIGMgPSBjYXJnYXJfY2xpcChmdiwgZmEsIGZzLCBtYXBhLCBjYXBhc191c2FyKQ0KICAgICAgICAgICAgaWYgYyBpcyBOb25lOg0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICBrID0gbGVuKGNbJ29rJ10pDQogICAgICAgICAgICBpZiBudWV2bzoNCiAgICAgICAgICAgICAgICBzZWxmLnZveltwb3M6cG9zICsga10gPSBjWyd2b3onXQ0KICAgICAgICAgICAgWGFbcG9zOnBvcyArIGtdLCBYZltwb3M6cG9zICsga10gPSBjWydhcnQnXSwgY1snZm9uJ10NCiAgICAgICAgICAgIHBhZCA9IG1heCgwLCBMIC0gaykNCiAgICAgICAgICAgIFlzLmFwcGVuZChucC5wYWQoY1sneSddLCAoKDAsIHBhZCksICgwLCAwKSkpKTsgTXMuYXBwZW5kKG5wLnBhZChjWydvayddLCAoMCwgcGFkKSkpDQogICAgICAgICAgICBpbmljaW9zICs9IFtwb3MgKyBqIGZvciBqIGluIHJhbmdlKDAsIG1heChrLCBMKSAtIEwgKyAxLCBQQVNPKV0NCiAgICAgICAgICAgIHBvcyArPSBtYXgoaywgTCk7IGhvcmFzICs9IGNbJ2R1ciddIC8gMzYwMA0KICAgICAgICAgICAgaWYgbWF4X2hvcmFzIGFuZCBob3JhcyA+PSBtYXhfaG9yYXM6DQogICAgICAgICAgICAgICAgYnJlYWsNCiAgICAgICAgaWYgbnVldm86DQogICAgICAgICAgICBzZWxmLnZvei5mbHVzaCgpDQogICAgICAgICAgICBvcGVuKHJ1dGEgKyAnLm9rJywgJ3cnKS53cml0ZShzdHIocG9zKSkNCiAgICAgICAgc2VsZi5uID0gcG9zDQogICAgICAgIHNlbGYueSA9IG5wLmNvbmNhdGVuYXRlKFlzKVs6cG9zXTsgc2VsZi5tID0gbnAuY29uY2F0ZW5hdGUoTXMpWzpwb3NdDQogICAgICAgIHNlbGYuc2QgPSBzZWxmLnlbc2VsZi5tXS5zdGQoMCkgKyAxZS0zDQogICAgICAgIHNlbGYuYXJ0LCBzZWxmLmZvbiwgc2VsZi5pbmljaW9zLCBzZWxmLmhvcmFzID0gWGFbOnBvc10sIFhmWzpwb3NdLCBucC5hcnJheShpbmljaW9zKSwgaG9yYXMNCiAgICAgICAgbG9nKGYnZW50cmVuYW1pZW50bzoge2xlbihpbmljaW9zKX0gdmVudGFuYXMsIHtob3JhczouMWZ9IGgsIHtsZW4oc2VsZi52YWwpfSBjbGlwcyBkZSB2YWxpZGFjacOzbjsgdm96IGVuIHtydXRhfScpDQoNCiAgICBkZWYgYV90b3JjaChzZWxmLCBkaXNwb3NpdGl2bywgdm96X2VuX2dwdT1GYWxzZSk6DQogICAgICAgICIiIlRlbnNvcmVzIHBhcmEgaW5kZXhhciBwb3IgbG90ZXMuIExhIHZveiBzZSBxdWVkYSBlbiBlbCBtZW1tYXAgc2Fsdm8gcXVlIHNlIHBpZGEgKHkgcXVlcGEpIGVuIGxhIEdQVS4iIiINCiAgICAgICAgaW1wb3J0IHRvcmNoDQogICAgICAgIHNlbGYuYXJ0X3QgPSB0b3JjaC5mcm9tX251bXB5KHNlbGYuYXJ0LmFzdHlwZShucC5mbG9hdDE2KSkNCiAgICAgICAgc2VsZi5mb25fdCA9IHRvcmNoLmZyb21fbnVtcHkoc2VsZi5mb24pDQogICAgICAgIHNlbGYueV90ID0gdG9yY2guZnJvbV9udW1weSgoc2VsZi55IC8gc2VsZi5zZCkuYXN0eXBlKG5wLmZsb2F0MzIpKQ0KICAgICAgICBzZWxmLm1fdCA9IHRvcmNoLmZyb21fbnVtcHkoc2VsZi5tKQ0KICAgICAgICBzZWxmLnZvel90ID0gTm9uZQ0KICAgICAgICBpZiB2b3pfZW5fZ3B1Og0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIHNlbGYudm96X3QgPSB0b3JjaC5mcm9tX251bXB5KG5wLmFzY29udGlndW91c2FycmF5KHNlbGYudm96KSkudG8oZGlzcG9zaXRpdm8pDQogICAgICAgICAgICBleGNlcHQgKFJ1bnRpbWVFcnJvciwgTWVtb3J5RXJyb3IpOg0KICAgICAgICAgICAgICAgIHNlbGYudm96X3QgPSBOb25lDQogICAgICAgIHJldHVybiBzZWxmDQoNCiAgICBkZWYgbG90ZShzZWxmLCBpZHgsIGRpc3Bvc2l0aXZvKToNCiAgICAgICAgIiIiaWR4OiB0ZW5zb3IgW0JdIGRlIGluaWNpb3MgZGUgdmVudGFuYSDihpIgdm96IFtCLEwsSywxMDI0XSBmMzIsIGFydCwgZm9uLCB5LCBtIGVuIGBkaXNwb3NpdGl2b2AuIiIiDQogICAgICAgIGltcG9ydCB0b3JjaA0KICAgICAgICBpaSA9IGlkeFs6LCBOb25lXSArIHRvcmNoLmFyYW5nZShMKQ0KICAgICAgICBpZiBzZWxmLnZvel90IGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgdm96ID0gc2VsZi52b3pfdFtpaS50byhzZWxmLnZvel90LmRldmljZSldLmZsb2F0KCkNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIHZveiA9IHRvcmNoLmZyb21fbnVtcHkobnAuc3RhY2soW3NlbGYudm96W2k6aSArIExdIGZvciBpIGluIGlkeC50b2xpc3QoKV0pKS50byhkaXNwb3NpdGl2bykuZmxvYXQoKQ0KICAgICAgICByZXR1cm4gKHZveiwgc2VsZi5hcnRfdFtpaV0udG8oZGlzcG9zaXRpdm8pLmZsb2F0KCksIHNlbGYuZm9uX3RbaWldLnRvKGRpc3Bvc2l0aXZvKSwNCiAgICAgICAgICAgICAgICBzZWxmLnlfdFtpaV0udG8oZGlzcG9zaXRpdm8pLCBzZWxmLm1fdFtpaV0udG8oZGlzcG9zaXRpdm8pLnVuc3F1ZWV6ZSgtMSkuZmxvYXQoKSkNCg==", "tools/cara_parlante/voz3/mapa_arkit.py": "IiIiTWFwYSBsaW5lYWwgU01JUksgKEZMQU1FKSDihpIgbG9zIDUyIGJsZW5kc2hhcGVzIEFSS2l0IGRlIE1lZGlhUGlwZSArIDMgw6FuZ3Vsb3MgZGUgY2FiZXphLg0KDQpFb3MgKE1ldGFIdW1hbikgc29sbyBlbnRpZW5kZSBsb3MgNTIgbW9ycGhzIEFSS2l0OyBTTUlSSyBkYSBleHByZXNpw7NuIEZMQU1FICg1MCksIG1hbmTDrWJ1bGEgKDMpLA0KcMOhcnBhZG9zICgyKSB5IHBvc2UgKDMpLiBPcGNpb25lcyBjb25zaWRlcmFkYXM6DQogIGEpIGludmVydGlyIGxhIG1hdHJpeiBBUktpdOKGkkZMQU1FIGRlIG1lZGlhcGlwZS1ibGVuZHNoYXBlcy10by1mbGFtZSAoUGVpemhpWWFuKTogZXN0w6EgYWp1c3RhZGENCiAgICAgY29uIGNhcmFzIHNpbnTDqXRpY2FzLCA1MuKGkjUwIG5vIGVzIGludmVydGlibGUgc2luIHJlZ3VsYXJpemFyIHkgbm8gY3VicmUgbWFuZMOtYnVsYSBuaSBww6FycGFkb3M7DQogIGIpIHBhc2FyIHBvciBsYSBtYWxsYSBGTEFNRSB5IG1lZGlyIGxvcyBibGVuZHNoYXBlcyBjb24gTWVkaWFQaXBlIHNvYnJlIHVuIHJlbmRlcjogZXhpZ2UgZWwNCiAgICAgbW9kZWxvIEZMQU1FIGRlbCBNUEkgKHJlZ2lzdHJvKSB5IHJlbmRlcml6YXIgbWlsbG9uZXMgZGUgZm90b2dyYW1hczsNCiAgYykgcmlkZ2UgU01JUkvihpJNZWRpYVBpcGUgYWp1c3RhZG8gY29uIGxvcyBwYXJlcyByZWFsZXMgKG1pc21vIGZvdG9ncmFtYSBldGlxdWV0YWRvIHBvciBsb3MgZG9zKSwNCiAgICAgbWlsbG9uZXMgZGUgZm90b2dyYW1hcyBkZSBPYmFtYSB5IDFLSC4gRXMgbG8gcXVlIHNlIGhhY2UgYXF1w606IHNpbiByZWdpc3RybywgY2FsaWJyYWRvDQogICAgIGVuIGxhcyBtaXNtYXMgdW5pZGFkZXMgTWVkaWFQaXBlIHF1ZSB5YSB1c2EgZ2VuZXJhci5weSAoZ2FuYW5jaWFzIHkgZXNjYWxhX2JvY2FfMWtoLmpzb24pLA0KICAgICB5IGVsIHJ1aWRvIGRlIE1lZGlhUGlwZSBzZSBwcm9tZWRpYSBlbiBlbCBhanVzdGUgKGVsIG1hcGEgZXMgZGUgNTggcGFyw6FtZXRyb3MgYSA1NSBzYWxpZGFzKS4NCkxhIGxpbWl0YWNpw7NuOiBlbCBtYXBhIGhlcmVkYSBlbCBzZXNnbyBtZWRpbyBkZSBNZWRpYVBpcGUgKG5vIHN1IHRlbWJsb3IpLiBQYXJhIGxhIGJvY2ENCnBlZGFnw7NnaWNhIGVzbyBzZSBjb3JyaWdlIGVuIGdlbmVyYXIucHkgY29uIGxhIGdhcmFudMOtYSBkZSBCbGVuZGVyIChwL2IvbSwgdS9vL3cpLg0KDQpVc286ICBweXRob24gbWFwYV9hcmtpdC5weSAtLXNtaXJrIERJUiAtLW1lZGlkYXMgRElSIC0tc2FsaWRhIG1hcGFfYXJraXQuanNvbiBbLS1hbHBoYSAxMF0NCkRlamEgZWwgSlNPTiBjb24gbWVkaWEvc2QgZGUgZW50cmFkYSwgVyBbNTgsNTVdLCBiIFs1NV0sIGNvbHVtbmFzIHkgciBkZSB2YWxpZGFjacOzbiBjcnV6YWRhICgyIG1pdGFkZXMpLg0KRGVzZGUgb3Ryb3MgZ3Vpb25lczogbWFwYSA9IGNhcmdhcihqc29uKTsgWSA9IGFwbGljYXIobWFwYSwgWCkgIChYIFtULDU4XSBkZSByYXNnb3MoKSkuDQoiIiINCmltcG9ydCBhcmdwYXJzZQ0KaW1wb3J0IGpzb24NCmltcG9ydCBvcw0KaW1wb3J0IHN5cw0KaW1wb3J0IHpsaWINCg0KaW1wb3J0IG51bXB5IGFzIG5wDQoNCmlmIGhhc2F0dHIoc3lzLnN0ZG91dCwgJ3JlY29uZmlndXJlJyk6DQogICAgc3lzLnN0ZG91dC5yZWNvbmZpZ3VyZShlbmNvZGluZz0ndXRmLTgnLCBlcnJvcnM9J3JlcGxhY2UnKQ0Kc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkNCmltcG9ydCBkYXRvc192MyBhcyBEICAjIG5vcWE6IEU0MDINCg0KDQpkZWYgYWp1c3RhcihYLCBZLCBhbHBoYT0xMC4wKToNCiAgICAiIiJSaWRnZSBjb24gWCBlc3RhbmRhcml6YWRhLiBEZXZ1ZWx2ZSBlbCBtYXBhIGNvbW8gZGljdCBzZXJpYWxpemFibGUuIiIiDQogICAgbXgsIHN4ID0gWC5tZWFuKDApLCBYLnN0ZCgwKSArIDFlLTYNCiAgICBaID0gKFggLSBteCkgLyBzeA0KICAgIG15ID0gWS5tZWFuKDApDQogICAgQSA9IFouVCBAIFogKyBhbHBoYSAqIG5wLmV5ZShaLnNoYXBlWzFdKQ0KICAgIFcgPSBucC5saW5hbGcuc29sdmUoQSwgWi5UIEAgKFkgLSBteSkpICAgICAgICAgICAgICAgICAgICAjIFs1OCwgNTVdDQogICAgcmV0dXJuIHsnbWVkaWFfeCc6IG14LnRvbGlzdCgpLCAnc2RfeCc6IHN4LnRvbGlzdCgpLCAnVyc6IFcudG9saXN0KCksICdiJzogbXkudG9saXN0KCksICdhbHBoYSc6IGFscGhhfQ0KDQoNCmRlZiBhcGxpY2FyKG1hcGEsIFgpOg0KICAgIFogPSAobnAuYXNhcnJheShYLCBucC5mbG9hdDMyKSAtIG5wLmFycmF5KG1hcGFbJ21lZGlhX3gnXSwgbnAuZmxvYXQzMikpIC8gbnAuYXJyYXkobWFwYVsnc2RfeCddLCBucC5mbG9hdDMyKQ0KICAgIHJldHVybiBaIEAgbnAuYXJyYXkobWFwYVsnVyddLCBucC5mbG9hdDMyKSArIG5wLmFycmF5KG1hcGFbJ2InXSwgbnAuZmxvYXQzMikNCg0KDQpkZWYgY2FyZ2FyKHJ1dGEpOg0KICAgIHJldHVybiBqc29uLmxvYWQob3BlbihydXRhKSkNCg0KDQpkZWYgY29ycmVsYWNpb25lcyhQLCBZLCBjb2x1bW5hcywgY2xpcD1Ob25lKToNCiAgICAiIiJyIHBvciBjb2x1bW5hLiBTaSBgY2xpcGAgKGlkIHBvciBmb3RvZ3JhbWEpIHNlIGRhLCBzZSBxdWl0YSBsYSBtZWRpYSBkZSBjYWRhIGNsaXAgYW50ZXMNCiAgICAoZXMgbG8gcXVlIHZlIGVsIGVudHJlbmFtaWVudG86IGRpZmVyZW5jaWFzIHNvYnJlIGxhIG1lZGlhIGRlbCBjbGlwKS4iIiINCiAgICBpZiBjbGlwIGlzIG5vdCBOb25lOg0KICAgICAgICBQLCBZID0gUC5jb3B5KCksIFkuY29weSgpDQogICAgICAgIGZvciBjIGluIG5wLnVuaXF1ZShjbGlwKToNCiAgICAgICAgICAgIHMgPSBjbGlwID09IGMNCiAgICAgICAgICAgIFBbc10gLT0gUFtzXS5tZWFuKDApOyBZW3NdIC09IFlbc10ubWVhbigwKQ0KICAgIHIgPSB7fQ0KICAgIGZvciBrLCBjIGluIGVudW1lcmF0ZShjb2x1bW5hcyk6DQogICAgICAgIHJbY10gPSByb3VuZChmbG9hdChucC5jb3JyY29lZihQWzosIGtdLCBZWzosIGtdKVswLCAxXSksIDMpIGlmIFlbOiwga10uc3RkKCkgPiAxZS02IGFuZCBQWzosIGtdLnN0ZCgpID4gMWUtNiBlbHNlIE5vbmUNCiAgICByZXR1cm4gcg0KDQoNCmRlZiBwb3JfZ3J1cG8ocik6DQogICAgcmV0dXJuIHtnOiByb3VuZChmbG9hdChucC5uYW5tZWFuKFtyW2NdIGZvciBjIGluIGNzIGlmIHIuZ2V0KGMpIGlzIG5vdCBOb25lXSkpLCAzKSBpZiBhbnkoci5nZXQoYykgaXMgbm90IE5vbmUgZm9yIGMgaW4gY3MpIGVsc2UgTm9uZQ0KICAgICAgICAgICAgZm9yIGcsIGNzIGluIEQuR1JVUE9TLml0ZW1zKCl9DQoNCg0KZGVmIG1pdGFkKG5vbWJyZSk6DQogICAgIiIiUmVwYXJ0byBlc3RhYmxlIGVuIGRvcyBtaXRhZGVzIHBvciB2w61kZW8gKGxvcyB0cm96b3MgXzAwLCBfMDEgZGVsIG1pc21vIHbDrWRlbyB2YW4ganVudG9zKS4iIiINCiAgICByZXR1cm4gemxpYi5jcmMzMihub21icmUucnNwbGl0KCdfJywgMSlbMF0uZW5jb2RlKCkpICUgMg0KDQoNCmRlZiBtYWluKCk6DQogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpDQogICAgYXAuYWRkX2FyZ3VtZW50KCctLXNtaXJrJywgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tbWVkaWRhcycsIHJlcXVpcmVkPVRydWUpDQogICAgYXAuYWRkX2FyZ3VtZW50KCctLXNhbGlkYScsIHJlcXVpcmVkPVRydWUpDQogICAgYXAuYWRkX2FyZ3VtZW50KCctLWFscGhhJywgdHlwZT1mbG9hdCwgZGVmYXVsdD0xMC4wKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgnLS1tYXhfY2xpcHMnLCB0eXBlPWludCwgZGVmYXVsdD0wKQ0KICAgIGEgPSBhcC5wYXJzZV9hcmdzKCkNCiAgICBwYXJlcyA9IEQucGFyZXNfc21pcmsoYS5zbWlyaywgYS5tZWRpZGFzKQ0KICAgIGlmIGEubWF4X2NsaXBzOg0KICAgICAgICBwYXJlcyA9IHBhcmVzWzphLm1heF9jbGlwc10NCiAgICBYLCBZLCBjbGlwLCBjb2xzLCBub21icmVzID0gRC5hcGlsYXIocGFyZXMpDQogICAgcHJpbnQoZid7bGVuKG5vbWJyZXMpfSBjbGlwcywge2xlbihYKX0gZm90b2dyYW1hcyBjb24gY2FyYSBlbiBsb3MgZG9zIGV0aXF1ZXRhZG9yZXMnLCBmbHVzaD1UcnVlKQ0KICAgIG0gPSBucC5hcnJheShbbWl0YWQobikgZm9yIG4gaW4gbm9tYnJlc10pW2NsaXBdDQogICAgcnMgPSBbXQ0KICAgIGZvciBoIGluICgwLCAxKToNCiAgICAgICAgdHIsIHRlID0gbSAhPSBoLCBtID09IGgNCiAgICAgICAgaWYgdHIuc3VtKCkgPCAxMDAgb3IgdGUuc3VtKCkgPCAxMDA6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBtcCA9IGFqdXN0YXIoWFt0cl0sIFlbdHJdLCBhLmFscGhhKQ0KICAgICAgICBycy5hcHBlbmQoY29ycmVsYWNpb25lcyhhcGxpY2FyKG1wLCBYW3RlXSksIFlbdGVdLCBjb2xzLCBjbGlwW3RlXSkpDQogICAgcl9jdiA9IHtjOiAocm91bmQoZmxvYXQobnAubWVhbihbcltjXSBmb3IgciBpbiBycyBpZiByW2NdIGlzIG5vdCBOb25lXSkpLCAzKSBpZiBycyBhbmQgYW55KHJbY10gaXMgbm90IE5vbmUgZm9yIHIgaW4gcnMpIGVsc2UgTm9uZSkNCiAgICAgICAgICAgIGZvciBjIGluIGNvbHN9DQogICAgbWFwYSA9IGFqdXN0YXIoWCwgWSwgYS5hbHBoYSkNCiAgICBtYXBhLnVwZGF0ZShjb2x1bW5hc19lbnRyYWRhPUQuRU5UUkFEQVNfU01JUkssIGNvbHVtbmFzPWNvbHMsIGNsaXBzPWxlbihub21icmVzKSwgZm90b2dyYW1hcz1pbnQobGVuKFgpKSwNCiAgICAgICAgICAgICAgICByX2N2PXJfY3YsIHJfY3ZfZ3J1cG9zPXBvcl9ncnVwbyhyX2N2KSkNCiAgICBqc29uLmR1bXAobWFwYSwgb3BlbihhLnNhbGlkYSwgJ3cnKSwgaW5kZW50PTEpDQogICAgcHJpbnQoJ3IgdmFsaWRhY2nDs24gY3J1emFkYSBwb3IgZ3J1cG8gKHNpbiBtZWRpYSBkZWwgY2xpcCk6JywgbWFwYVsncl9jdl9ncnVwb3MnXSwgZmx1c2g9VHJ1ZSkNCg0KDQppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOg0KICAgIG1haW4oKQ0K", "tools/cara_parlante/voz3/fiabilidad.py": "IiIiUGlsb3RvIGRlIGZpYWJpbGlkYWQgZGUgbGFzIGV0aXF1ZXRhczogwr9jdcOhbnRvIGNvaW5jaWRlbiBTTUlSSyB5IE1lZGlhUGlwZSBzb2JyZSBsb3MgbWlzbW9zDQpmb3RvZ3JhbWFzLCBibGVuZHNoYXBlIGEgYmxlbmRzaGFwZT8NCg0KQ29uIGxvcyBjbGlwcyBxdWUgdGllbmVuIGxhcyBkb3MgZXRpcXVldGFzIHNlIGFqdXN0YSB1biBtYXBhIHJpZGdlIEZMQU1F4oaSY2FkYSBibGVuZHNoYXBlIEFSS2l0DQoobWFwYV9hcmtpdC5hanVzdGFyKSBlbiB1bmEgbWl0YWQgZGUgbG9zIGNsaXBzIChyZXBhcnRvIGVzdGFibGUgcG9yIHbDrWRlbykgeSBzZSBtaWRlIHIgZW4gbGENCm90cmEgbWl0YWQ7IGRlc3B1w6lzIGFsIHJldsOpcywgeSBzZSBwcm9tZWRpYW4uIFNlIGluZm9ybWEgciBicnV0byB5IHIgc2luIGxhIG1lZGlhIGRlIGNhZGEgY2xpcA0KKGxvIHF1ZSB2ZSBlbCBlbnRyZW5hbWllbnRvKSwgcG9yIGJsZW5kc2hhcGUgeSBwb3IgZ3J1cG8gKGJvY2EsIHNvbnJpc2EsIG9qb3MsIGNlamFzLCBjYWJlemEpLg0KDQpMZWN0dXJhOiBzaSBkb3MgZXN0aW1hZG9yZXMgaW5kZXBlbmRpZW50ZXMgY29pbmNpZGVuIGNvbiByID0gz4Egc29icmUgbGEgbWlzbWEgY2FyYSwgdW4gbW9kZWxvDQpwZXJmZWN0byBlbnRyZW5hZG8gY29uIHVubyBkZSBlbGxvcyBubyBwdWVkZSBwYXNhciBkZSDiiYgg4oiaz4EgZnJlbnRlIGFsIG90cm87IHkgdW5hIHIgYWx0YSBlbiBsYQ0KYm9jYSBkaWNlIHF1ZSBTTUlSSyB2ZSBsbyBxdWUgTWVkaWFQaXBlIHZlLCBjb24gbWVub3MgcnVpZG8gKFNNSVJLIGVzIDNEIHkgc2UgYWp1c3RhIGEgbGEgY2FyYQ0KZW50ZXJhKS4gVW4gciBiYWpvIGVuIHVuYSBjb2x1bW5hIGNvbmNyZXRhIHNlw7FhbGEgdW4gYmxlbmRzaGFwZSBxdWUgdW5vIGRlIGxvcyBkb3MgbWlkZSBtYWwuDQoNClVzbzogcHl0aG9uIGZpYWJpbGlkYWQucHkgLS1zbWlyayBESVIgLS1tZWRpZGFzIERJUiAtLXNhbGlkYSBpbmZvcm1lX2ZpYWJpbGlkYWQuanNvbiBbLS1hbHBoYSAxMF0NCkltcHJpbWUgbGEgdGFibGEgeSBkZWphIGVsIEpTT04gKHIgcG9yIGNvbHVtbmEgZW4gY2FkYSB2YXJpYW50ZSwgZ3J1cG9zLCBjbGlwcyB5IGZvdG9ncmFtYXMpLg0KIiIiDQppbXBvcnQgYXJncGFyc2UNCmltcG9ydCBqc29uDQppbXBvcnQgb3MNCmltcG9ydCBzeXMNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQoNCmlmIGhhc2F0dHIoc3lzLnN0ZG91dCwgJ3JlY29uZmlndXJlJyk6DQogICAgc3lzLnN0ZG91dC5yZWNvbmZpZ3VyZShlbmNvZGluZz0ndXRmLTgnLCBlcnJvcnM9J3JlcGxhY2UnKSAgICMgbGEgY29uc29sYSBkZSBXaW5kb3dzIGVzIGNwMTI1Mg0Kc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkNCmltcG9ydCBkYXRvc192MyBhcyBEICAgICAgICAgICMgbm9xYTogRTQwMg0KaW1wb3J0IG1hcGFfYXJraXQgYXMgTSAgICAgICAgIyBub3FhOiBFNDAyDQoNCg0KZGVmIG1haW4oKToNCiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tc21pcmsnLCByZXF1aXJlZD1UcnVlKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgnLS1tZWRpZGFzJywgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tc2FsaWRhJywgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tYWxwaGEnLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTEwLjApDQogICAgYSA9IGFwLnBhcnNlX2FyZ3MoKQ0KICAgIHBhcmVzID0gRC5wYXJlc19zbWlyayhhLnNtaXJrLCBhLm1lZGlkYXMpDQogICAgaWYgbGVuKHBhcmVzKSA8IDI6DQogICAgICAgIHN5cy5leGl0KGYnaGFjZW4gZmFsdGEgYWwgbWVub3MgMiBjbGlwcyBjb24gbGFzIGRvcyBldGlxdWV0YXMgKGhheSB7bGVuKHBhcmVzKX0pJykNCiAgICBYLCBZLCBjbGlwLCBjb2xzLCBub21icmVzID0gRC5hcGlsYXIocGFyZXMpDQogICAgbWl0ID0gbnAuYXJyYXkoW00ubWl0YWQobikgZm9yIG4gaW4gbm9tYnJlc10pDQogICAgaWYgbGVuKHNldChtaXQpKSA8IDI6ICAgICAgICAgICAgICAgICAgICAgICAjIHBvY29zIGNsaXBzOiByZXBhcnRvIGFsdGVybm8gcGFyYSBxdWUgaGF5YSBkb3MgbWl0YWRlcw0KICAgICAgICBtaXQgPSBucC5hcmFuZ2UobGVuKG5vbWJyZXMpKSAlIDINCiAgICBtID0gbWl0W2NsaXBdDQogICAgYnJ1dG8sIHNpbl9tZWRpYSwgbl90ZXN0ID0gW10sIFtdLCBbXQ0KICAgIGZvciBoIGluICgwLCAxKToNCiAgICAgICAgdHIsIHRlID0gbSAhPSBoLCBtID09IGgNCiAgICAgICAgbXAgPSBNLmFqdXN0YXIoWFt0cl0sIFlbdHJdLCBhLmFscGhhKQ0KICAgICAgICBQID0gTS5hcGxpY2FyKG1wLCBYW3RlXSkNCiAgICAgICAgYnJ1dG8uYXBwZW5kKE0uY29ycmVsYWNpb25lcyhQLCBZW3RlXSwgY29scykpDQogICAgICAgIHNpbl9tZWRpYS5hcHBlbmQoTS5jb3JyZWxhY2lvbmVzKFAsIFlbdGVdLCBjb2xzLCBjbGlwW3RlXSkpDQogICAgICAgIG5fdGVzdC5hcHBlbmQoaW50KHRlLnN1bSgpKSkNCg0KICAgIGRlZiBtZWRpYShycyk6DQogICAgICAgIHJldHVybiB7YzogKHJvdW5kKGZsb2F0KG5wLm1lYW4oW3JbY10gZm9yIHIgaW4gcnMgaWYgcltjXSBpcyBub3QgTm9uZV0pKSwgMykgaWYgYW55KHJbY10gaXMgbm90IE5vbmUgZm9yIHIgaW4gcnMpIGVsc2UgTm9uZSkNCiAgICAgICAgICAgICAgICBmb3IgYyBpbiBjb2xzfQ0KICAgIHJiLCBycyA9IG1lZGlhKGJydXRvKSwgbWVkaWEoc2luX21lZGlhKQ0KICAgIGdiLCBncyA9IE0ucG9yX2dydXBvKHJiKSwgTS5wb3JfZ3J1cG8ocnMpDQogICAgcHJpbnQoZid7bGVuKG5vbWJyZXMpfSBjbGlwcywge2xlbihYKX0gZm90b2dyYW1hczsgbWl0YWRlcyBkZSB7bl90ZXN0fSBmb3RvZ3JhbWFzJykNCiAgICBwcmludChmJ3siY29sdW1uYSI6MjJzfSB7InIgYnJ1dG8iOj44c30geyJyIHNpbiBtZWRpYSI6PjEyc30nKQ0KICAgIGZvciBjIGluIGNvbHM6DQogICAgICAgIHByaW50KGYne2M6MjJzfSB7c3RyKHJiW2NdKTo+OHN9IHtzdHIocnNbY10pOj4xMnN9JykNCiAgICBwcmludCgncG9yIGdydXBvIChicnV0byk6ICAgICcsIGdiKQ0KICAgIHByaW50KCdwb3IgZ3J1cG8gKHNpbiBtZWRpYSk6JywgZ3MpDQogICAgcHJpbnQoJ3RlY2hvIG9yaWVudGF0aXZvIGRlbCBtb2RlbG8gKHJhaXogZGUgciBzaW4gbWVkaWEpOicsIHtnOiAocm91bmQoZmxvYXQobnAuc3FydChtYXgodiwgMCkpKSwgMikgaWYgdiBpcyBub3QgTm9uZSBlbHNlIE5vbmUpIGZvciBnLCB2IGluIGdzLml0ZW1zKCl9KQ0KICAgIGpzb24uZHVtcCh7J2NsaXBzJzogbGVuKG5vbWJyZXMpLCAnZm90b2dyYW1hcyc6IGludChsZW4oWCkpLCAnbm9tYnJlcyc6IG5vbWJyZXMsICdhbHBoYSc6IGEuYWxwaGEsDQogICAgICAgICAgICAgICAncl9icnV0byc6IHJiLCAncl9zaW5fbWVkaWEnOiBycywgJ2dydXBvc19icnV0byc6IGdiLCAnZ3J1cG9zX3Npbl9tZWRpYSc6IGdzfSwNCiAgICAgICAgICAgICAgb3BlbihhLnNhbGlkYSwgJ3cnKSwgaW5kZW50PTEpDQoNCg0KaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoNCiAgICBtYWluKCkNCg==", "tools/cara_parlante/voz3/extraer_v3.py": "IiIiUmFzZ29zIGRlIHZveiB2MzogSHVCRVJULWxhcmdlIChvIFdhdkxNLWxhcmdlKSBjb25nZWxhZG8sIHZhcmlhcyBjYXBhcyBndWFyZGFkYXMgcGFyYSBxdWUgbGEKcmVkIGFwcmVuZGEgdW5hIHN1bWEgcG9uZGVyYWRhIGRlIGNhcGFzIChjb21vIGVuIFNVUEVSQiksIGVuIGxhIHJlamlsbGEgYHRgIGRlbCAubnB6IGRlIG1lZGlkYXMuCgpFbGVjY2nDs246IHBvciBkZWZlY3RvIGZhY2Vib29rL2h1YmVydC1sYXJnZS1sbDYwayAoQXBhY2hlLTIuMDsgZXMgZWwgY29kaWZpY2Fkb3IgZGUgRmFjZURpZmZ1c2VyLApDb2RlVGFsa2VyLCBEaWZmUG9zZVRhbGssIExlYXJuMlRhbGsgeSBTZWxmVGFsaywgcXVlIGxvIHByZWZpZXJlbiBhIHdhdjJ2ZWMyIHBhcmEgbGEgc2luY3JvbsOtYQpsYWJpYWwpLiBtaWNyb3NvZnQvd2F2bG0tbGFyZ2UgKC0tY29kaWZpY2Fkb3Igd2F2bG0tbGFyZ2UpIHB1bnTDumEgdW4gcG9jbyBtZWpvciBlbiBmb25lbWFzIGVuClNVUEVSQiB5IGVzIG3DoXMgcm9idXN0byBhbCBydWlkbywgcGVybyBzdSBsaWNlbmNpYSAocmVwb3NpdG9yaW8gVW5pU3BlZWNoKSBubyBlcyBBcGFjaGU6IHZhbGUKcGFyYSBlc3RlIHVzbyBhY2Fkw6ltaWNvLCBubyBjb21vIG9wY2nDs24gcG9yIGRlZmVjdG8uIExvcyBkb3M6IDI0IGNhcGFzLCAxMDI0IGRpbXMsIDUwIEh6LgoKRGlzY286IDI4IGggZGUgYXVkaW8gw5cgMjQgY2FwYXMgYSA1MCBIeiBzZXLDrWFuIH4yNTAgR0IuIFNlIGd1YXJkYW4gc29sbyAtLWNhcGFzICg3IHBvciBkZWZlY3RvOgo2LDksMTIsMTUsMTgsMjEsMjQpLCBhIDI1IEh6IChtZWRpYSBkZSBwYXJlcykgeSBlbiBmbG9hdDE2IOKGkiB+MSwzIEdCL2gsIH4zNiBHQiBlbiB0b3RhbCwgcXVlCmNhYmVuIGVuIERyaXZlICgyNTAgR0IgbGlicmVzKSB5IGVuIGxhIFJBTSBkZWwgQTEwMCBIaWdoLVJBTSAobyBlbiB1biBtZW1tYXAsIHZlciBkYXRvc192My5weSkuCjxzYWxpZGE+Lzxub21icmU+Lm5weiAoc2luIGNvbXByaW1pcjogZmxvYXQxNiBubyBjb21wcmltZSB5IHRhcmRhKSBjb24gYHZvemAgW1QsIEssIDEwMjRdIHkgYHRgLgoKVHJvem9zIGRlIDMwIHMgY29uIDEgcyBkZSBzb2xhcGUgKHNlIGRlc2NhcnRhIGVsIGJvcmRlKSBwYXJhIHF1ZSBlbCB0cmFuc2Zvcm1lciB0ZW5nYSBjb250ZXh0by4KUmVhbnVkYWJsZSB5IHJlcGFydGlibGUgKC0tcGFydGUgaS9uKS4KClVzbzogcHl0aG9uIGV4dHJhZXJfdjMucHkgLS1tZWRpZGFzIERJUl9OUFogLS1hdWRpbyBESVIgKG1wNC9tNGEvd2F2L2ZsYWMpIC0tc2FsaWRhIERJUgogICAgICAgIFstLWNvZGlmaWNhZG9yIGh1YmVydC1sYXJnZXx3YXZsbS1sYXJnZV0gWy0tY2FwYXMgNiw5LDEyLDE1LDE4LDIxLDI0XSBbLS1kaXNwb3NpdGl2byBjdWRhXQoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBnbG9iCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSwgJy4uJywgJ3ZvejJjYXJhJykpCmZyb20gZXh0cmFlciBpbXBvcnQgYXVkaW8gICAjIG5vcWE6IEU0MDIgIChmZm1wZWcg4oaSIGZsb2F0MzIgMTYga0h6IG1vbm8pCgpNT0RFTE9TID0geydodWJlcnQtbGFyZ2UnOiAnZmFjZWJvb2svaHViZXJ0LWxhcmdlLWxsNjBrJywgJ3dhdmxtLWxhcmdlJzogJ21pY3Jvc29mdC93YXZsbS1sYXJnZSd9CkNBUEFTID0gJzYsOSwxMiwxNSwxOCwyMSwyNCcKU1IgPSAxNjAwMAoKCmRlZiBjYXJnYXJfY29kaWZpY2Fkb3Iobm9tYnJlLCBkaXNwb3NpdGl2byk6CiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsCiAgICByZWQgPSBBdXRvTW9kZWwuZnJvbV9wcmV0cmFpbmVkKE1PREVMT1MuZ2V0KG5vbWJyZSwgbm9tYnJlKSkudG8oZGlzcG9zaXRpdm8pLmV2YWwoKQogICAgaWYgZGlzcG9zaXRpdm8uc3RhcnRzd2l0aCgnY3VkYScpOgogICAgICAgIHJlZCA9IHJlZC5oYWxmKCkKICAgIHJldHVybiByZWQKCgpkZWYgcmFzZ29zXzUwaHoocmVkLCB4LCBjYXBhcywgZGlzcG9zaXRpdm8sIHRyb3pvPTMwLjAsIG1hcmdlbj0xLjApOgogICAgIiIiW1Q1MCwgSywgMTAyNF0gZmxvYXQxNiBkZSB0b2RvIGVsIGF1ZGlvIHggKGZsb2F0MzIgMTYga0h6KSwgcG9yIHRyb3pvcyBjb24gc29sYXBlLiIiIgogICAgbiA9IGxlbih4KQogICAgTCwgTWcgPSBpbnQodHJvem8gKiBTUiksIGludChtYXJnZW4gKiBTUikKICAgIHNhbGlkYXMgPSBbXQogICAgaW5pID0gMAogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgd2hpbGUgaW5pIDwgbjoKICAgICAgICAgICAgYSwgYiA9IG1heCgwLCBpbmkgLSBNZyksIG1pbihuLCBpbmkgKyBMICsgTWcpCiAgICAgICAgICAgIHNlZyA9IHRvcmNoLmZyb21fbnVtcHkoeFthOmJdKVtOb25lXS50byhkaXNwb3NpdGl2bykKICAgICAgICAgICAgaWYgZGlzcG9zaXRpdm8uc3RhcnRzd2l0aCgnY3VkYScpOgogICAgICAgICAgICAgICAgc2VnID0gc2VnLmhhbGYoKQogICAgICAgICAgICBpZiBzZWcuc2hhcGVbMV0gPCA4MDA6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBoID0gcmVkKHNlZywgb3V0cHV0X2hpZGRlbl9zdGF0ZXM9VHJ1ZSkuaGlkZGVuX3N0YXRlcyAgICAgICAgICAjIDI1IHRlbnNvcmVzIFsxLCBULCAxMDI0XQogICAgICAgICAgICBocyA9IHRvcmNoLnN0YWNrKFtoW2NdWzBdIGZvciBjIGluIGNhcGFzXSwgMSkgICAgICAgICAgICAgICAgICAgICMgW1QsIEssIDEwMjRdCiAgICAgICAgICAgICMgNTAgSHo6IDMyMCBtdWVzdHJhcyBwb3IgcGFzbzsgc2UgcmVjb3J0YSBlbCBtYXJnZW4gYcOxYWRpZG8KICAgICAgICAgICAgaTAgPSAoaW5pIC0gYSkgLy8gMzIwCiAgICAgICAgICAgIGkxID0gaTAgKyAobWluKG4sIGluaSArIEwpIC0gaW5pKSAvLyAzMjAKICAgICAgICAgICAgc2FsaWRhcy5hcHBlbmQoaHNbaTA6aTFdLnRvKHRvcmNoLmZsb2F0MTYpLmNwdSgpKQogICAgICAgICAgICBpbmkgKz0gTAogICAgcmV0dXJuIHRvcmNoLmNhdChzYWxpZGFzKS5udW1weSgpIGlmIHNhbGlkYXMgZWxzZSBucC56ZXJvcygoMCwgbGVuKGNhcGFzKSwgMTAyNCksIG5wLmZsb2F0MTYpCgoKZGVmIGFfcmVqaWxsYShmNTAsIHQpOgogICAgIiIiNTAgSHog4oaSIDI1IEh6IChtZWRpYSBkZSBwYXJlcykg4oaSIGZpbGEgbcOhcyBjZXJjYW5hIGEgY2FkYSBpbnN0YW50ZSBkZSBsYSByZWppbGxhIHQuIiIiCiAgICBmMjUgPSBmNTBbOiBsZW4oZjUwKSAvLyAyICogMl0ucmVzaGFwZSgtMSwgMiwgKmY1MC5zaGFwZVsxOl0pLmFzdHlwZShucC5mbG9hdDMyKS5tZWFuKDEpCiAgICBnID0gbnAuYXJhbmdlKGxlbihmMjUpKSAvIDI1LjAKICAgIGlkeCA9IG5wLmNsaXAobnAuc2VhcmNoc29ydGVkKGcsIHQpLCAwLCBsZW4oZjI1KSAtIDEpCiAgICByZXR1cm4gZjI1W2lkeF0uYXN0eXBlKG5wLmZsb2F0MTYpCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLW1lZGlkYXMnLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLWF1ZGlvJywgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1zYWxpZGEnLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLWNvZGlmaWNhZG9yJywgZGVmYXVsdD0naHViZXJ0LWxhcmdlJykKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1jYXBhcycsIGRlZmF1bHQ9Q0FQQVMpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tZGlzcG9zaXRpdm8nLCBkZWZhdWx0PSdjdWRhJyBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgJ2NwdScpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tcGFydGUnLCBkZWZhdWx0PScxLzEnKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLW1heF9zZWcnLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAsIGhlbHA9J3NvbG8gbG9zIHByaW1lcm9zIHNlZ3VuZG9zIChwcnVlYmFzKScpCiAgICBhID0gYXAucGFyc2VfYXJncygpCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IGltYWdlaW9fZmZtcGVnCiAgICAgICAgZmZtcGVnID0gaW1hZ2Vpb19mZm1wZWcuZ2V0X2ZmbXBlZ19leGUoKQogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIGZmbXBlZyA9ICdmZm1wZWcnCiAgICBjYXBhcyA9IFtpbnQoYykgZm9yIGMgaW4gYS5jYXBhcy5zcGxpdCgnLCcpXQogICAgaV9wYXJ0ZSwgbl9wYXJ0ZSA9IChpbnQoeCkgZm9yIHggaW4gYS5wYXJ0ZS5zcGxpdCgnLycpKQogICAgb3MubWFrZWRpcnMoYS5zYWxpZGEsIGV4aXN0X29rPVRydWUpCiAgICByZWQgPSBjYXJnYXJfY29kaWZpY2Fkb3IoYS5jb2RpZmljYWRvciwgYS5kaXNwb3NpdGl2bykKICAgIGF1ZGlvcyA9IHtvcy5wYXRoLnNwbGl0ZXh0KG9zLnBhdGguYmFzZW5hbWUodikpWzBdOiB2IGZvciBleHQgaW4gKCdtcDQnLCAnbTRhJywgJ3dhdicsICdmbGFjJywgJ21wMycpCiAgICAgICAgICAgICAgZm9yIHYgaW4gZ2xvYi5nbG9iKG9zLnBhdGguam9pbihhLmF1ZGlvLCAnKionLCAnKi4nICsgZXh0KSwgcmVjdXJzaXZlPVRydWUpfQogICAgdG9kb3MgPSBzb3J0ZWQoZ2xvYi5nbG9iKG9zLnBhdGguam9pbihhLm1lZGlkYXMsICcqKicsICcqLm5weicpLCByZWN1cnNpdmU9VHJ1ZSkpCiAgICBoZWNob3MsIHQwID0gMCwgdGltZS50aW1lKCkKICAgIGZvciBqLCBucHogaW4gZW51bWVyYXRlKHRvZG9zKToKICAgICAgICBub21icmUgPSBvcy5wYXRoLnNwbGl0ZXh0KG9zLnBhdGguYmFzZW5hbWUobnB6KSlbMF0KICAgICAgICBkZXN0ID0gb3MucGF0aC5qb2luKGEuc2FsaWRhLCBub21icmUgKyAnLm5weicpCiAgICAgICAgaWYgaiAlIG5fcGFydGUgIT0gaV9wYXJ0ZSAtIDEgb3Igb3MucGF0aC5leGlzdHMoZGVzdCkgb3Igbm9tYnJlIG5vdCBpbiBhdWRpb3M6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdCA9IG5wLmxvYWQobnB6KVsndCddCiAgICAgICAgdHJ5OgogICAgICAgICAgICB4ID0gYXVkaW8oYXVkaW9zW25vbWJyZV0sIGZmbXBlZykKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIHByaW50KCdzaW4gYXVkaW8nLCBub21icmUsIGUsIGZsdXNoPVRydWUpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgYS5tYXhfc2VnOgogICAgICAgICAgICB4ID0geFs6IGludChhLm1heF9zZWcgKiBTUildCiAgICAgICAgZjUwID0gcmFzZ29zXzUwaHoocmVkLCB4LCBjYXBhcywgYS5kaXNwb3NpdGl2bykKICAgICAgICBpZiBsZW4oZjUwKSA8IDQ6CiAgICAgICAgICAgIHByaW50KCdhdWRpbyB2YWPDrW8nLCBub21icmUsIGZsdXNoPVRydWUpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbnAuc2F2ZXooZGVzdCArICcudG1wLm5weicsIHZvej1hX3JlamlsbGEoZjUwLCB0KSwgdD10LCBjYXBhcz1ucC5hcnJheShjYXBhcyksIGNvZGlmaWNhZG9yPWEuY29kaWZpY2Fkb3IpCiAgICAgICAgb3MucmVwbGFjZShkZXN0ICsgJy50bXAubnB6JywgZGVzdCkKICAgICAgICBoZWNob3MgKz0gMQogICAgICAgIGlmIGhlY2hvcyAlIDEwID09IDAgb3IgaiArIDEgPT0gbGVuKHRvZG9zKToKICAgICAgICAgICAgcHJpbnQoZidbe2ogKyAxfS97bGVuKHRvZG9zKX1dIHtoZWNob3N9IGhlY2hvcywge2xlbih0KSAvIDI1IC8gMzYwMDouMmZ9IGggZXN0ZSwgeyh0aW1lLnRpbWUoKSAtIHQwKSAvIGhlY2hvczouMGZ9IHMvY2xpcCcsIGZsdXNoPVRydWUpCiAgICBwcmludCgnbGlzdG8nLCBoZWNob3MsIGZsdXNoPVRydWUpCgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIG1haW4oKQo=", "tools/cara_parlante/voz3/red_v3.py": "IiIiTGFzIHJlZGVzIGRlIHYzIHkgbG8gcXVlIGNvbXBhcnRlbiBzdXMgZG9zIGVudHJlbmFtaWVudG9zIChib2NhIHkgbm8gdmVyYmFsKS4NCg0KLSBTdW1hQ2FwYXM6IHN1bWEgcG9uZGVyYWRhIGFwcmVuZGlkYSBkZSBsYXMgSyBjYXBhcyBndWFyZGFkYXMgZGUgSHVCRVJUL1dhdkxNIChzb2Z0bWF4IHNvYnJlIEsNCiAgcGVzb3MsIGNhZGEgY2FwYSBub3JtYWxpemFkYSBhbnRlcyksIGNvbW8gZW4gU1VQRVJCOyBkZXNwdcOpcyB1bmEgcHJveWVjY2nDs24gYSBkLg0KLSBUcm9uY286IGxhIHBhcnRlIGNvbcO6biBkZSBsYXMgZG9zIHJlZGVzOiB2b3ogKyBhcnRpY3VsYWNpw7NuIGRlIEJsZW5kZXIgKyBmb25lbWEgZW1iZWJpZG8g4oaSDQogIGNvbnYgbG9jYWwg4oaSIHRyYW5zZm9ybWVyIHByZS1ub3JtIChHRUxVKS4NCi0gQm9jYTogcmVncmVzacOzbjsgc2FsaWRhID0gYmFzZSBsaW5lYWwgZGVzZGUgbG9zIGNvbnRyb2xlcyBkZSBCbGVuZGVyICsgY29ycmVjY2nDs24gZGVsIHRyb25jbw0KICAoaWd1YWwgcXVlIGxhIHJlZCBkZSBmdXNpw7NuOiBsYSBhbmF0b23DrWEgbGEgcG9uZSBCbGVuZGVyLCBsYSByZWQgYcOxYWRlIGNvYXJ0aWN1bGFjacOzbiB5IHZpZGEpLg0KLSBOb1ZlcmJhbDogZmxvdyBtYXRjaGluZyAocmVjdGlmaWVkIGZsb3cpIDFEIGNvbiB0cmFuc2Zvcm1lciwgY29uZGljaW9uYWRvIHBvciBsYSB2b3osIMKraGFibGHCuw0KICB5IHVuIHZlY3RvciBkZSBlc3RpbG8gKHNkIGRlIGNhZGEgY2FuYWwgZW4gZWwgY2xpcCk7IG11ZXN0cmVvIEV1bGVyIGNvbiBzZW1pbGxhIHkgZ3XDrWEgc2luDQogIGNsYXNpZmljYWRvci4gRXMgbG8gcXVlIGhhY2UgcXVlIGNlamFzLCBvam9zLCBzb25yaXNhIGxlbnRhIHkgY2FiZXphIHNhbGdhbiB2aXZvcyB5IHZhcmlhZG9zDQogIGVuIHZleiBkZSBwcm9tZWRpYWRvcyAoRGlmZlBvc2VUYWxrLCBBUlRhbGspLg0KLSBFc3RhZG86IGd1YXJkYXIvY2FyZ2FyIGNoZWNrcG9pbnQgY29tcGxldG8gKHJlZCwgRU1BLCBvcHRpbWl6YWRvciwgcGxhbmlmaWNhZG9yLCDDqXBvY2EsIFJORykNCiAgcGFyYSByZWFudWRhciBzaSBDb2xhYiBzZSBjYWU7IHByb2dyZXNvLmpzb24gcG9yIMOpcG9jYTsgRU1BIGRlIHBlc29zLg0KIiIiDQppbXBvcnQganNvbg0KaW1wb3J0IG1hdGgNCmltcG9ydCBvcw0KaW1wb3J0IHJhbmRvbQ0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCB0b3JjaA0KaW1wb3J0IHRvcmNoLm5uIGFzIG5uDQoNCg0KY2xhc3MgU3VtYUNhcGFzKG5uLk1vZHVsZSk6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIGssIGQsIGRyb3A9MC4xKToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNlbGYubm9ybWEgPSBubi5MYXllck5vcm0oMTAyNCkNCiAgICAgICAgc2VsZi5wZXNvcyA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcyhrKSkNCiAgICAgICAgc2VsZi5zYWwgPSBubi5TZXF1ZW50aWFsKG5uLkRyb3BvdXQoZHJvcCksIG5uLkxpbmVhcigxMDI0LCBkKSkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHZveik6ICAgICAgICAgICAgICAgICAgICAgICAjIFtCLCBULCBLLCAxMDI0XQ0KICAgICAgICB3ID0gdG9yY2guc29mdG1heChzZWxmLnBlc29zLCAwKQ0KICAgICAgICByZXR1cm4gc2VsZi5zYWwoKHNlbGYubm9ybWEodm96KSAqIHdbTm9uZSwgTm9uZSwgOiwgTm9uZV0pLnN1bSgyKSkNCg0KICAgIGRlZiBwZXNvc19jYXBhcyhzZWxmKToNCiAgICAgICAgcmV0dXJuIHRvcmNoLnNvZnRtYXgoc2VsZi5wZXNvcy5kZXRhY2goKSwgMCkuY3B1KCkubnVtcHkoKS5yb3VuZCgzKS50b2xpc3QoKQ0KDQoNCmNsYXNzIFRyb25jbyhubi5Nb2R1bGUpOg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBrLCBkPTM4NCwgY2FwYXM9NiwgY2FiZXphcz02LCBkcm9wPTAuMiwgbl9hcnQ9MTMsIG5fZm9uPTM1KToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNlbGYudm96ID0gU3VtYUNhcGFzKGssIGQpDQogICAgICAgIHNlbGYuYXJ0ID0gbm4uTGluZWFyKG5fYXJ0LCBkKQ0KICAgICAgICBzZWxmLmZvbiA9IG5uLkVtYmVkZGluZyhuX2ZvbiwgZCkNCiAgICAgICAgc2VsZi5jb252ID0gbm4uQ29udjFkKGQsIGQsIDUsIHBhZGRpbmc9MiwgZ3JvdXBzPWQpDQogICAgICAgIGNhcGEgPSBubi5UcmFuc2Zvcm1lckVuY29kZXJMYXllcihkLCBjYWJlemFzLCA0ICogZCwgZHJvcG91dD1kcm9wLCBiYXRjaF9maXJzdD1UcnVlLCBub3JtX2ZpcnN0PVRydWUsIGFjdGl2YXRpb249J2dlbHUnKQ0KICAgICAgICBzZWxmLmN1ZXJwbyA9IG5uLlRyYW5zZm9ybWVyRW5jb2RlcihjYXBhLCBjYXBhcywgZW5hYmxlX25lc3RlZF90ZW5zb3I9RmFsc2UpDQogICAgICAgIHNlbGYubm9ybWEgPSBubi5MYXllck5vcm0oZCkNCg0KICAgIGRlZiBlbnRyYWRhKHNlbGYsIHZveiwgYXJ0LCBmb24pOg0KICAgICAgICBoID0gc2VsZi52b3oodm96KSArIHNlbGYuYXJ0KGFydCkgKyBzZWxmLmZvbihmb24pDQogICAgICAgIHJldHVybiBoICsgc2VsZi5jb252KGgudHJhbnNwb3NlKDEsIDIpKS50cmFuc3Bvc2UoMSwgMikNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHZveiwgYXJ0LCBmb24sIGV4dHJhPU5vbmUpOg0KICAgICAgICBoID0gc2VsZi5lbnRyYWRhKHZveiwgYXJ0LCBmb24pDQogICAgICAgIGlmIGV4dHJhIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgaCA9IGggKyBleHRyYQ0KICAgICAgICByZXR1cm4gc2VsZi5ub3JtYShzZWxmLmN1ZXJwbyhoKSkNCg0KDQpjbGFzcyBCb2NhKG5uLk1vZHVsZSk6DQogICAgIiIiVm96ICsgQmxlbmRlciArIGZvbmVtYSDihpIgY29sdW1uYXMgZGUgbGEgYm9jYSAoQVJLaXQgdsOtYSBtYXBhIFNNSVJLKSB5IG1hbmTDrWJ1bGEuIiIiDQoNCiAgICBkZWYgX19pbml0X18oc2VsZiwgaywgc2FsLCBkPTM4NCwgY2FwYXM9NiwgY2FiZXphcz02LCBkcm9wPTAuMiwgbl9hcnQ9MTMpOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgc2VsZi50cm9uY28gPSBUcm9uY28oaywgZCwgY2FwYXMsIGNhYmV6YXMsIGRyb3AsIG5fYXJ0KQ0KICAgICAgICBzZWxmLnNhbGlkYSA9IG5uLkxpbmVhcihkLCBzYWwpDQogICAgICAgIHNlbGYuYmFzZSA9IG5uLkxpbmVhcihuX2FydCwgc2FsKQ0KDQogICAgZGVmIGZvcndhcmQoc2VsZiwgdm96LCBhcnQsIGZvbik6DQogICAgICAgIHJldHVybiBzZWxmLmJhc2UoYXJ0KSArIHNlbGYuc2FsaWRhKHNlbGYudHJvbmNvKHZveiwgYXJ0LCBmb24pKQ0KDQoNCmRlZiB0aWVtcG9fc2ludXNvaWRhbCh0LCBkKToNCiAgICAiIiJ0IGVuIFswLDFdIFtCXSDihpIgW0IsIGRdIiIiDQogICAgZiA9IHRvcmNoLmV4cCgtbWF0aC5sb2coMTAwMDApICogdG9yY2guYXJhbmdlKDAsIGQsIDIsIGRldmljZT10LmRldmljZSkgLyBkKQ0KICAgIGFuZyA9IHRbOiwgTm9uZV0gKiAxMDAwICogZltOb25lXQ0KICAgIHJldHVybiB0b3JjaC5jYXQoW2FuZy5zaW4oKSwgYW5nLmNvcygpXSwgMSkNCg0KDQpjbGFzcyBOb1ZlcmJhbChubi5Nb2R1bGUpOg0KICAgICIiIkNhbXBvIGRlIHZlbG9jaWRhZCB2KHhfdCwgdCB8IHZveiwgaGFibGEsIGVzdGlsbykgcGFyYSBmbG93IG1hdGNoaW5nIHNvYnJlIFtCLCBMLCBDXS4iIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBrLCBjLCBkPTI1NiwgY2FwYXM9NCwgY2FiZXphcz00LCBkcm9wPTAuMSwgbl9hcnQ9MTMsIG5fZXN0aWxvPTApOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgc2VsZi5jLCBzZWxmLmQsIHNlbGYubl9lc3RpbG8gPSBjLCBkLCBuX2VzdGlsbw0KICAgICAgICBzZWxmLnRyb25jbyA9IFRyb25jbyhrLCBkLCBjYXBhcywgY2FiZXphcywgZHJvcCwgbl9hcnQpDQogICAgICAgIHNlbGYueCA9IG5uLkxpbmVhcihjLCBkKQ0KICAgICAgICBzZWxmLnQgPSBubi5TZXF1ZW50aWFsKG5uLkxpbmVhcihkLCBkKSwgbm4uR0VMVSgpLCBubi5MaW5lYXIoZCwgZCkpDQogICAgICAgIHNlbGYuZXN0aWxvID0gbm4uTGluZWFyKG5fZXN0aWxvLCBkKSBpZiBuX2VzdGlsbyBlbHNlIE5vbmUNCiAgICAgICAgc2VsZi5zYWxpZGEgPSBubi5MaW5lYXIoZCwgYykNCiAgICAgICAgbm4uaW5pdC56ZXJvc18oc2VsZi5zYWxpZGEud2VpZ2h0KTsgbm4uaW5pdC56ZXJvc18oc2VsZi5zYWxpZGEuYmlhcykNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHh0LCB0LCB2b3osIGFydCwgZm9uLCBlc3RpbG89Tm9uZSwgc2luX3Zvej1Ob25lKToNCiAgICAgICAgIiIic2luX3ZvejogbcOhc2NhcmEgW0JdIGJvb2w6IGVzYXMgbXVlc3RyYXMgdmFuIHNpbiBjb25kaWNpw7NuIGRlIHZveiAocGFyYSBsYSBndcOtYSkuIiIiDQogICAgICAgIGlmIHNpbl92b3ogaXMgbm90IE5vbmU6DQogICAgICAgICAgICB2b3ogPSB2b3ogKiAofnNpbl92b3opWzosIE5vbmUsIE5vbmUsIE5vbmVdLmZsb2F0KCkNCiAgICAgICAgZXh0cmEgPSBzZWxmLngoeHQpICsgc2VsZi50KHRpZW1wb19zaW51c29pZGFsKHQsIHNlbGYuZCkpWzosIE5vbmVdDQogICAgICAgIGlmIHNlbGYuZXN0aWxvIGlzIG5vdCBOb25lIGFuZCBlc3RpbG8gaXMgbm90IE5vbmU6DQogICAgICAgICAgICBleHRyYSA9IGV4dHJhICsgc2VsZi5lc3RpbG8oZXN0aWxvKVs6LCBOb25lXQ0KICAgICAgICByZXR1cm4gc2VsZi5zYWxpZGEoc2VsZi50cm9uY28odm96LCBhcnQsIGZvbiwgZXh0cmEpKQ0KDQogICAgQHRvcmNoLm5vX2dyYWQoKQ0KICAgIGRlZiBtdWVzdHJlYXIoc2VsZiwgdm96LCBhcnQsIGZvbiwgZXN0aWxvPU5vbmUsIHBhc29zPTMyLCBndWlhPTEuNSwgZ2VuZXJhZG9yPU5vbmUpOg0KICAgICAgICAiIiJFdWxlciBkZXNkZSBydWlkbyBOKDAsMSkgaGFzdGEgbGEgbXVlc3RyYTsgZ3XDrWEgc2luIGNsYXNpZmljYWRvciBzb2JyZSBsYSB2b3ouIiIiDQogICAgICAgIEIsIFQgPSB2b3ouc2hhcGVbOjJdDQogICAgICAgIHggPSB0b3JjaC5yYW5kbihCLCBULCBzZWxmLmMsIGdlbmVyYXRvcj1nZW5lcmFkb3IsIGRldmljZT12b3ouZGV2aWNlKQ0KICAgICAgICBmb3IgaSBpbiByYW5nZShwYXNvcyk6DQogICAgICAgICAgICB0ID0gdG9yY2guZnVsbCgoQiwpLCBpIC8gcGFzb3MsIGRldmljZT12b3ouZGV2aWNlKQ0KICAgICAgICAgICAgdiA9IHNlbGYoeCwgdCwgdm96LCBhcnQsIGZvbiwgZXN0aWxvKQ0KICAgICAgICAgICAgaWYgZ3VpYSAhPSAxLjA6DQogICAgICAgICAgICAgICAgdjAgPSBzZWxmKHgsIHQsIHZveiwgYXJ0LCBmb24sIGVzdGlsbywgc2luX3Zvej10b3JjaC5vbmVzKEIsIGR0eXBlPXRvcmNoLmJvb2wsIGRldmljZT12b3ouZGV2aWNlKSkNCiAgICAgICAgICAgICAgICB2ID0gdjAgKyBndWlhICogKHYgLSB2MCkNCiAgICAgICAgICAgIHggPSB4ICsgdiAvIHBhc29zDQogICAgICAgIHJldHVybiB4DQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGVudHJlbmFtaWVudG8gY29tw7puDQpkZWYgc2VtaWxsYXMocyk6DQogICAgdG9yY2gubWFudWFsX3NlZWQocyk7IG5wLnJhbmRvbS5zZWVkKHMpOyByYW5kb20uc2VlZChzKQ0KDQoNCmRlZiBlbWFfYWN0dWFsaXphcihlbWEsIHJlZCwgcGFzbyk6DQogICAgZGVjID0gbWluKDAuOTk5LCAoMSArIHBhc28pIC8gKDEwICsgcGFzbykpDQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6DQogICAgICAgIGZvciBwZSwgcHIgaW4gemlwKGVtYS5wYXJhbWV0ZXJzKCksIHJlZC5wYXJhbWV0ZXJzKCkpOg0KICAgICAgICAgICAgcGUubXVsXyhkZWMpLmFkZF8ocHIuZGV0YWNoKCksIGFscGhhPTEgLSBkZWMpDQoNCg0KZGVmIHBsYW5pZmljYWRvcihvcHQsIHBhc29zX2Vwb2NhLCBlcG9jYXMsIGNhbGVudGFtaWVudG89NSk6DQogICAgY2FsZW50LCB0b3RhbCA9IGNhbGVudGFtaWVudG8gKiBwYXNvc19lcG9jYSwgZXBvY2FzICogcGFzb3NfZXBvY2ENCiAgICBmID0gbGFtYmRhIHM6IG1pbigxLjAsIChzICsgMSkgLyBjYWxlbnQpICogMC41ICogKDEgKyBtYXRoLmNvcyhtYXRoLnBpICogbWluKDEuMCwgcyAvIHRvdGFsKSkpDQogICAgcmV0dXJuIHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlci5MYW1iZGFMUihvcHQsIGYpDQoNCg0KZGVmIGd1YXJkYXJfZXN0YWRvKHJ1dGEsIHJlZCwgZW1hLCBvcHQsIHNjaGVkLCBlc2NhbGEsICoqcmVzdG8pOg0KICAgIHRvcmNoLnNhdmUoeydyZWQnOiByZWQuc3RhdGVfZGljdCgpLCAnZW1hJzogZW1hLnN0YXRlX2RpY3QoKSwgJ29wdCc6IG9wdC5zdGF0ZV9kaWN0KCksICdzY2hlZCc6IHNjaGVkLnN0YXRlX2RpY3QoKSwNCiAgICAgICAgICAgICAgICAnZXNjYWxhJzogZXNjYWxhLnN0YXRlX2RpY3QoKSwgJ3JuZyc6IHRvcmNoLmdldF9ybmdfc3RhdGUoKSwgJ25wX3JuZyc6IG5wLnJhbmRvbS5nZXRfc3RhdGUoKSwgKipyZXN0b30sDQogICAgICAgICAgICAgICBydXRhICsgJy50bXAnKQ0KICAgIG9zLnJlcGxhY2UocnV0YSArICcudG1wJywgcnV0YSkNCg0KDQpkZWYgY2FyZ2FyX2VzdGFkbyhydXRhLCByZWQsIGVtYSwgb3B0LCBzY2hlZCwgZXNjYWxhLCBkaXNwb3NpdGl2byk6DQogICAgIiIiRGV2dWVsdmUgZWwgZGljdCBjb24gbG8gZGVtw6FzICjDqXBvY2EsIG1lam9yLCAuLi4pIG8gTm9uZSBzaSBubyBoYXkgY2hlY2twb2ludC4iIiINCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMocnV0YSk6DQogICAgICAgIHJldHVybiBOb25lDQogICAgZSA9IHRvcmNoLmxvYWQocnV0YSwgbWFwX2xvY2F0aW9uPWRpc3Bvc2l0aXZvLCB3ZWlnaHRzX29ubHk9RmFsc2UpDQogICAgcmVkLmxvYWRfc3RhdGVfZGljdChlWydyZWQnXSk7IGVtYS5sb2FkX3N0YXRlX2RpY3QoZVsnZW1hJ10pOyBvcHQubG9hZF9zdGF0ZV9kaWN0KGVbJ29wdCddKQ0KICAgIHNjaGVkLmxvYWRfc3RhdGVfZGljdChlWydzY2hlZCddKTsgZXNjYWxhLmxvYWRfc3RhdGVfZGljdChlWydlc2NhbGEnXSkNCiAgICB0b3JjaC5zZXRfcm5nX3N0YXRlKGVbJ3JuZyddLmNwdSgpKTsgbnAucmFuZG9tLnNldF9zdGF0ZShlWyducF9ybmcnXSkNCiAgICByZXR1cm4ge2s6IHYgZm9yIGssIHYgaW4gZS5pdGVtcygpIGlmIGsgbm90IGluICgncmVkJywgJ2VtYScsICdvcHQnLCAnc2NoZWQnLCAnZXNjYWxhJywgJ3JuZycsICducF9ybmcnKX0NCg0KDQpkZWYgYW5vdGFyKHNhbGlkYSwgbGluZWEpOg0KICAgICIiIlVuYSBsw61uZWEgSlNPTiBwb3Igw6lwb2NhIGVuIHByb2dyZXNvLmpzb24gKGHDsWFkaWVuZG8pIHkgcG9yIHBhbnRhbGxhLiIiIg0KICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4oc2FsaWRhLCAncHJvZ3Jlc28uanNvbicpLCAnYScpIGFzIGY6DQogICAgICAgIGYud3JpdGUoanNvbi5kdW1wcyhsaW5lYSkgKyAnXG4nKQ0KICAgIHByaW50KGpzb24uZHVtcHMobGluZWEpLCBmbHVzaD1UcnVlKQ0KDQoNCmRlZiBjb3JyZWxhY2lvbmVzKFAsIFQsIGNvbHVtbmFzKToNCiAgICByZXR1cm4ge2M6IChyb3VuZChmbG9hdChucC5jb3JyY29lZihQWzosIGtdLCBUWzosIGtdKVswLCAxXSksIDMpIGlmIFRbOiwga10uc3RkKCkgPiAxZS02IGFuZCBQWzosIGtdLnN0ZCgpID4gMWUtNiBlbHNlIE5vbmUpDQogICAgICAgICAgICBmb3IgaywgYyBpbiBlbnVtZXJhdGUoY29sdW1uYXMpfQ0KDQoNCmRlZiBtZWRpYV9ncnVwb3MociwgZ3J1cG9zLCBjb2x1bW5hcyk6DQogICAgb3V0ID0ge30NCiAgICBmb3IgZywgY3MgaW4gZ3J1cG9zLml0ZW1zKCk6DQogICAgICAgIHYgPSBbcltjXSBmb3IgYyBpbiBjcyBpZiBjIGluIGNvbHVtbmFzIGFuZCByLmdldChjKSBpcyBub3QgTm9uZV0NCiAgICAgICAgb3V0W2ddID0gcm91bmQoZmxvYXQobnAubWVhbih2KSksIDMpIGlmIHYgZWxzZSBOb25lDQogICAgcmV0dXJuIG91dA0KDQoNCmRlZiBhdW1lbnRhcl92b3oodm96LCBMLCBkZXYsIHJ1aWRvPTAuMDUsIG1hc2NhcmFzPTIsIHBfYXBhZ2FyPTAuMik6DQogICAgIiIiUnVpZG8sIG3DoXNjYXJhcyBkZSB0aWVtcG8geSwgYSB2ZWNlcywgbGEgdm96IGVudGVyYSBhcGFnYWRhIChvYmxpZ2EgYSB1c2FyIGxhIGFydGljdWxhY2nDs24pLiIiIg0KICAgIEIgPSB2b3ouc2hhcGVbMF0NCiAgICB2b3ogPSB2b3ogKyBydWlkbyAqIHRvcmNoLnJhbmRuX2xpa2Uodm96KQ0KICAgIHR0ID0gdG9yY2guYXJhbmdlKEwsIGRldmljZT1kZXYpW05vbmVdDQogICAgZm9yIF8gaW4gcmFuZ2UobWFzY2FyYXMpOg0KICAgICAgICBpbmkgPSB0b3JjaC5yYW5kaW50KDAsIEwsIChCLCAxKSwgZGV2aWNlPWRldik7IGxnID0gdG9yY2gucmFuZGludCgwLCBMIC8vIDEwLCAoQiwgMSksIGRldmljZT1kZXYpDQogICAgICAgIHZveiA9IHZveiAqIH4oKHR0ID49IGluaSkgJiAodHQgPCBpbmkgKyBsZykpWy4uLiwgTm9uZSwgTm9uZV0NCiAgICByZXR1cm4gdm96ICogKHRvcmNoLnJhbmQoQiwgMSwgMSwgMSwgZGV2aWNlPWRldikgPiBwX2FwYWdhcikNCg==", "tools/cara_parlante/voz3/entrenar_boca.py": "IiIiQm9jYSB2Mzogdm96IChIdUJFUlQtbGFyZ2UsIHN1bWEgZGUgY2FwYXMgYXByZW5kaWRhKSArIGFydGljdWxhY2nDs24gZGUgQmxlbmRlciArIGZvbmVtYSDihpIKbGFzIDE0IGNvbHVtbmFzIEFSS2l0IGRlIGxhIGJvY2EgKGV0aXF1ZXRhcyBTTUlSSyBwYXNhZGFzIHBvciBlbCBtYXBhIGEgQVJLaXQpLCBwb3IgcmVncmVzacOzbi4KClNhbGlkYSA9IGJhc2UgbGluZWFsIGRlc2RlIGxvcyAxMyBjb250cm9sZXMgZGUgQmxlbmRlciArIGNvcnJlY2Npw7NuIGRlbCB0cmFuc2Zvcm1lciwgY29tbyBsYQpyZWQgZGUgZnVzacOzbi4gUMOpcmRpZGFzOiBwb3NpY2nDs24gKE1TRSBjb24gbcOhc2NhcmEpLCB2ZWxvY2lkYWQgKGRpZmVyZW5jaWFzKSB5IHBlc28gcGVkYWfDs2dpY286CmVuIGxvcyBmb3RvZ3JhbWFzIGRlIHAvYi9tIGxhcyBjb2x1bW5hcyBkZSBjaWVycmUgKG1vdXRoQ2xvc2UsIG1vdXRoUHJlc3MqKSB5IGphd09wZW4gcGVzYW4KLS1wZXNvX3BlZGFnb2dpY28gdmVjZXMgbcOhcywgeSBlbiB1L28vdyBsYXMgZGUgcmVkb25kZW8gKG1vdXRoUHVja2VyLCBtb3V0aEZ1bm5lbCkuCgpWYWxpZGFjacOzbiBwb3Igw6lwb2NhIGNvbiBlbCBtb2RlbG8gRU1BOiByIHBvciBjb2x1bW5hIHkgZGUgbGEgYm9jYSwgcGVkYWdvZ8OtYSAoZGlmZXJlbmNpYSBlbiBzZAplbnRyZSBwL2IvbSB5IHZvY2FsZXMgYWJpZXJ0YXMgZW4gZWwgY2llcnJlOyBlbnRyZSB1L28vdyB5IGFiaWVydGFzIGVuIGVsIHJlZG9uZGVvOyByZWQgdnMgcmVhbCkuCkd1YXJkYSBlbiAtLXNhbGlkYTogbW9kZWxvLnB0IChtZWpvciDDqXBvY2EsIEVNQSksIG5vcm1hbGl6YWNpb24uanNvbiwgaW5mb3JtZS5qc29uIChzZQpyZWVzY3JpYmUgY2FkYSDDqXBvY2EpLCBwcm9ncmVzby5qc29uICh1bmEgbMOtbmVhIHBvciDDqXBvY2EpIHkgdWx0aW1vLnB0IChyZWQsIEVNQSwgb3B0aW1pemFkb3IsCnBsYW5pZmljYWRvciwgw6lwb2NhOiBjb24gw6lsIGVsIG1pc21vIG1hbmRhdG8gUkVBTlVEQSBzb2xvIHNpIENvbGFiIHNlIGNhZSkuCgpVc286IHB5dGhvbiBlbnRyZW5hcl9ib2NhLnB5IC0tdm96IERJUiAtLWFydCBESVIgLS1zbWlyayBESVIgLS1tYXBhIG1hcGFfYXJraXQuanNvbiAtLXNhbGlkYSBESVIKICAgICAgICAtLW1vZGVsb19hcnQgbW9kZWxvX2FydGljdWxhdG9yaW8uanNvbiBbLS1jYWNoZSAvY29udGVudC9kYXRvcy9jYWNoZV0gWy0tZXBvY2FzIDMwMF0KICAgICAgICBbLS1wYWNpZW5jaWEgMzBdIFstLWxvdGUgNDhdIFstLWxyIDRlLTRdIFstLWNhcGFzX3VzYXIgNiw5LDEyLDE1LDE4LDIxLDI0XSBbLS12b3pfZW5fZ3B1XQogICAgICAgIFstLW1heF9ob3JhcyAwXSBbLS1kIDM4NCAtLWNhcGFzIDZdIFstLWRpc3Bvc2l0aXZvIGN1ZGFdCiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNvcHkKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgppZiBoYXNhdHRyKHN5cy5zdGRvdXQsICdyZWNvbmZpZ3VyZScpOgogICAgc3lzLnN0ZG91dC5yZWNvbmZpZ3VyZShlbmNvZGluZz0ndXRmLTgnLCBlcnJvcnM9J3JlcGxhY2UnKQpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKQppbXBvcnQgZGF0b3NfdjMgYXMgRCAgICAgICMgbm9xYTogRTQwMgppbXBvcnQgbWFwYV9hcmtpdCAgICAgICAgICMgbm9xYTogRTQwMgppbXBvcnQgcmVkX3YzIGFzIFIgICAgICAgICMgbm9xYTogRTQwMgoKQ09MVU1OQVMgPSBELkdSVVBPU1snYm9jYSddCkNJRVJSRSwgUkVET05ERU8gPSBbJ21vdXRoQ2xvc2UnLCAnbW91dGhQcmVzc0xlZnQnLCAnbW91dGhQcmVzc1JpZ2h0JywgJ2phd09wZW4nXSwgWydtb3V0aFB1Y2tlcicsICdtb3V0aEZ1bm5lbCddCkFCSUVSVEFTID0gWydhYScsICdhZScsICdhaCcsICdhbyddCgoKZGVmIHBlZGFnb2dpYShQLCBULCBGLCBjb2x1bW5hcywgZm9uZW1hcyk6CiAgICAiIiJEaWZlcmVuY2lhIG1lZGlhIChlbiBzZCkgZW50cmUgbG9zIGZvbmVtYXMgZGUgY2llcnJlL3JlZG9uZGVvIHkgbGFzIHZvY2FsZXMgYWJpZXJ0YXMuIiIiCiAgICBpID0ge2M6IGsgZm9yIGssIGMgaW4gZW51bWVyYXRlKGNvbHVtbmFzKX0KICAgIGlkeCA9IHtmOiBrIGZvciBrLCBmIGluIGVudW1lcmF0ZShmb25lbWFzKX0KCiAgICBkZWYgbShBLCBjb25qLCBjb2xzKToKICAgICAgICBzZWwgPSBucC5pc2luKEYsIFtpZHhbZl0gZm9yIGYgaW4gY29uaiBpZiBmIGluIGlkeF0pCiAgICAgICAgcmV0dXJuIGZsb2F0KEFbc2VsXVs6LCBbaVtjXSBmb3IgYyBpbiBjb2xzXV0ubWVhbigpKSBpZiBzZWwuYW55KCkgZWxzZSBmbG9hdCgnbmFuJykKICAgIG91dCA9IHt9CiAgICBmb3Igbm9tYnJlLCBjb25qLCBjb2xzIGluIFsoJ2NpZXJyZV9wYm0nLCBbJ3AnLCAnYicsICdtJ10sIFsnbW91dGhDbG9zZScsICdtb3V0aFByZXNzTGVmdCcsICdtb3V0aFByZXNzUmlnaHQnXSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAoJ3JlZG9uZGVvX3VvJywgWyd1dycsICdvdycsICd3J10sIFJFRE9OREVPKV06CiAgICAgICAgb3V0W25vbWJyZV0gPSB7J3JlZCc6IHJvdW5kKG0oUCwgY29uaiwgY29scykgLSBtKFAsIEFCSUVSVEFTLCBjb2xzKSwgMyksICdyZWFsJzogcm91bmQobShULCBjb25qLCBjb2xzKSAtIG0oVCwgQUJJRVJUQVMsIGNvbHMpLCAzKX0KICAgIHJldHVybiBvdXQKCgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tdm96JywgcmVxdWlyZWQ9VHJ1ZSk7IGFwLmFkZF9hcmd1bWVudCgnLS1hcnQnLCByZXF1aXJlZD1UcnVlKTsgYXAuYWRkX2FyZ3VtZW50KCctLXNtaXJrJywgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1tYXBhJywgcmVxdWlyZWQ9VHJ1ZSk7IGFwLmFkZF9hcmd1bWVudCgnLS1zYWxpZGEnLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLW1vZGVsb19hcnQnLCByZXF1aXJlZD1UcnVlLCBoZWxwPSdtb2RlbG9fYXJ0aWN1bGF0b3Jpby5qc29uIChub21icmVzIGRlIGZvbmVtYSknKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLWNhY2hlJywgZGVmYXVsdD1Ob25lLCBoZWxwPSdjYXJwZXRhIGxvY2FsIHBhcmEgZWwgbWVtbWFwIGRlIGxhIHZveiAocG9yIGRlZmVjdG8gLS1zYWxpZGEvY2FjaGUpJykKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1lcG9jYXMnLCB0eXBlPWludCwgZGVmYXVsdD0zMDApOyBhcC5hZGRfYXJndW1lbnQoJy0tcGFjaWVuY2lhJywgdHlwZT1pbnQsIGRlZmF1bHQ9MzApCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tbG90ZScsIHR5cGU9aW50LCBkZWZhdWx0PTQ4KTsgYXAuYWRkX2FyZ3VtZW50KCctLWxyJywgdHlwZT1mbG9hdCwgZGVmYXVsdD00ZS00KQogICAgYXAuYWRkX2FyZ3VtZW50KCctLXBlc29fcGVkYWdvZ2ljbycsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9My4wKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLWNhcGFzX3VzYXInLCBkZWZhdWx0PScnLCBoZWxwPSdzdWJjb25qdW50byBkZSBsYXMgY2FwYXMgZ3VhcmRhZGFzIChUNDogMTIsMTgsMjQpJykKICAgIGFwLmFkZF9hcmd1bWVudCgnLS12b3pfZW5fZ3B1JywgYWN0aW9uPSdzdG9yZV90cnVlJyk7IGFwLmFkZF9hcmd1bWVudCgnLS1tYXhfaG9yYXMnLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTApCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tZCcsIHR5cGU9aW50LCBkZWZhdWx0PTM4NCk7IGFwLmFkZF9hcmd1bWVudCgnLS1jYXBhcycsIHR5cGU9aW50LCBkZWZhdWx0PTYpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tZGlzcG9zaXRpdm8nLCBkZWZhdWx0PSdjdWRhJyBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgJ2NwdScpCiAgICBhID0gYXAucGFyc2VfYXJncygpCiAgICBvcy5tYWtlZGlycyhhLnNhbGlkYSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIFIuc2VtaWxsYXMoMCkKICAgIGRldiA9IGEuZGlzcG9zaXRpdm87IGdwdSA9IGRldi5zdGFydHN3aXRoKCdjdWRhJykKICAgIGZvbmVtYXMgPSBqc29uLmxvYWQob3BlbihhLm1vZGVsb19hcnQpKVsnZm9uZW1hcyddCiAgICBtYXBhID0gbWFwYV9hcmtpdC5jYXJnYXIoYS5tYXBhKQogICAgc2VsID0gW21hcGFbJ2NvbHVtbmFzJ10uaW5kZXgoYykgZm9yIGMgaW4gQ09MVU1OQVNdCiAgICBjYXBhc191c2FyID0gW2ludChjKSBmb3IgYyBpbiBhLmNhcGFzX3VzYXIuc3BsaXQoJywnKV0gaWYgYS5jYXBhc191c2FyIGVsc2UgTm9uZQoKICAgIGRhdG9zID0gRC5Db25qdW50byhELnBhcmVzX2VudHJlbmFtaWVudG8oYS52b3osIGEuYXJ0LCBhLnNtaXJrKSwgbWFwYSwgYS5jYWNoZSBvciBvcy5wYXRoLmpvaW4oYS5zYWxpZGEsICdjYWNoZScpLAogICAgICAgICAgICAgICAgICAgICAgIGNhcGFzX3VzYXIsIGEubWF4X2hvcmFzKQogICAgZGF0b3MueSA9IGRhdG9zLnlbOiwgc2VsXTsgZGF0b3Muc2QgPSBkYXRvcy5zZFtzZWxdCiAgICBmb3IgYyBpbiBkYXRvcy52YWw6CiAgICAgICAgY1sneSddID0gY1sneSddWzosIHNlbF0KICAgIGRhdG9zLmFfdG9yY2goZGV2LCBhLnZvel9lbl9ncHUpCiAgICBzZCA9IGRhdG9zLnNkCiAgICBLID0gZGF0b3Mudm96LnNoYXBlWzFdCiAgICAjIHBlc29zIHBvciBjb2x1bW5hIHkgcG9yIGZvbmVtYSAocGVkYWdvZ8OtYSkKICAgIGlkeF9mb24gPSB7ZjogayBmb3IgaywgZiBpbiBlbnVtZXJhdGUoZm9uZW1hcyl9CiAgICBwZXNvX2NvbCA9IHRvcmNoLm9uZXMobGVuKENPTFVNTkFTKSwgZGV2aWNlPWRldikKICAgIGVzX2NpZXJyZSA9IHRvcmNoLnRlbnNvcihbYyBpbiBDSUVSUkUgZm9yIGMgaW4gQ09MVU1OQVNdLCBkZXZpY2U9ZGV2KQogICAgZXNfcmVkb25kZW8gPSB0b3JjaC50ZW5zb3IoW2MgaW4gUkVET05ERU8gZm9yIGMgaW4gQ09MVU1OQVNdLCBkZXZpY2U9ZGV2KQogICAgZm9uX2NpZXJyZSA9IHRvcmNoLnRlbnNvcihbaWR4X2ZvbltmXSBmb3IgZiBpbiAoJ3AnLCAnYicsICdtJykgaWYgZiBpbiBpZHhfZm9uXSwgZGV2aWNlPWRldikKICAgIGZvbl9yZWRvbmRlbyA9IHRvcmNoLnRlbnNvcihbaWR4X2ZvbltmXSBmb3IgZiBpbiAoJ3V3JywgJ293JywgJ3cnKSBpZiBmIGluIGlkeF9mb25dLCBkZXZpY2U9ZGV2KQoKICAgIHJlZCA9IFIuQm9jYShLLCBsZW4oQ09MVU1OQVMpLCBhLmQsIGEuY2FwYXMpLnRvKGRldikKICAgIGVtYSA9IGNvcHkuZGVlcGNvcHkocmVkKS5ldmFsKCkKICAgIGZvciBwIGluIGVtYS5wYXJhbWV0ZXJzKCk6CiAgICAgICAgcC5yZXF1aXJlc19ncmFkXyhGYWxzZSkKICAgIG9wdCA9IHRvcmNoLm9wdGltLkFkYW1XKHJlZC5wYXJhbWV0ZXJzKCksIGEubHIsIHdlaWdodF9kZWNheT0wLjA1KQogICAgcGFzb3NfZXAgPSAobGVuKGRhdG9zLmluaWNpb3MpICsgYS5sb3RlIC0gMSkgLy8gYS5sb3RlCiAgICBzY2hlZCA9IFIucGxhbmlmaWNhZG9yKG9wdCwgcGFzb3NfZXAsIGEuZXBvY2FzKQogICAgYmYxNiA9IGdwdSBhbmQgdG9yY2guY3VkYS5pc19iZjE2X3N1cHBvcnRlZCgpCiAgICB0aXBvID0gdG9yY2guYmZsb2F0MTYgaWYgYmYxNiBlbHNlIHRvcmNoLmZsb2F0MTYKICAgIGVzY2FsYSA9IHRvcmNoLmFtcC5HcmFkU2NhbGVyKGVuYWJsZWQ9Z3B1IGFuZCBub3QgYmYxNikKICAgIGVzdGFkbyA9IFIuY2FyZ2FyX2VzdGFkbyhvcy5wYXRoLmpvaW4oYS5zYWxpZGEsICd1bHRpbW8ucHQnKSwgcmVkLCBlbWEsIG9wdCwgc2NoZWQsIGVzY2FsYSwgZGV2KSBvciB7fQogICAgZXAwLCBtZWpvciwgbWVqb3JfZXAsIHNpbl9tZWpvcmEsIHBhc28gPSAoZXN0YWRvLmdldChrLCB2KSBmb3IgaywgdiBpbgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKCgnZXBvY2EnLCAwKSwgKCdtZWpvcicsIC0xLjApLCAoJ21lam9yX2VwJywgMCksICgnc2luX21lam9yYScsIDApLCAoJ3Bhc28nLCAwKSkpCiAgICBpZiBlc3RhZG86CiAgICAgICAgcHJpbnQoZidyZWFudWRhbmRvIGRlc2RlIGxhIMOpcG9jYSB7ZXAwfSAobWVqb3Ige21lam9yOi40Zn0gZW4gbGEge21lam9yX2VwfSknLCBmbHVzaD1UcnVlKQogICAganNvbi5kdW1wKHsnY29sdW1uYXMnOiBDT0xVTU5BUywgJ3NkJzogc2QudG9saXN0KCksICdjYXBhcyc6IFtpbnQoYykgZm9yIGMgaW4gKGNhcGFzX3VzYXIgb3IgbnAubG9hZChkYXRvc192M19wcmltZXJhKGEudm96KSlbJ2NhcGFzJ10pXSwKICAgICAgICAgICAgICAgJ2NvZGlmaWNhZG9yJzogc3RyKG5wLmxvYWQoZGF0b3NfdjNfcHJpbWVyYShhLnZveikpWydjb2RpZmljYWRvciddKSwgJ2FycXVpdGVjdHVyYSc6IGYnQm9jYSBkPXthLmR9IGNhcGFzPXthLmNhcGFzfScsCiAgICAgICAgICAgICAgICdkJzogYS5kLCAnY2FwYXNfdHJhbnNmb3JtZXInOiBhLmNhcGFzLCAnZm9uZW1hcyc6IGZvbmVtYXN9LAogICAgICAgICAgICAgIG9wZW4ob3MucGF0aC5qb2luKGEuc2FsaWRhLCAnbm9ybWFsaXphY2lvbi5qc29uJyksICd3JykpCiAgICBpbmljaW9zID0gdG9yY2guZnJvbV9udW1weShkYXRvcy5pbmljaW9zKQoKICAgIGRlZiB2YWxpZGFyKG1vZGVsbyk6CiAgICAgICAgUCwgVCwgRiA9IFtdLCBbXSwgW10KICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKSwgdG9yY2guYXV0b2Nhc3QoJ2N1ZGEnIGlmIGdwdSBlbHNlICdjcHUnLCBkdHlwZT10aXBvLCBlbmFibGVkPWdwdSk6CiAgICAgICAgICAgIGZvciBjIGluIGRhdG9zLnZhbDoKICAgICAgICAgICAgICAgIHRyb3pvcyA9IFtdCiAgICAgICAgICAgICAgICBmb3IgayBpbiByYW5nZSgwLCBsZW4oY1snb2snXSksIDEwMDApOgogICAgICAgICAgICAgICAgICAgIHR2ID0gdG9yY2guZnJvbV9udW1weShjWyd2b3onXVtrOmsgKyAxMDAwXS5hc3R5cGUobnAuZmxvYXQzMikpLnRvKGRldilbTm9uZV0KICAgICAgICAgICAgICAgICAgICB0YSA9IHRvcmNoLmZyb21fbnVtcHkoY1snYXJ0J11bazprICsgMTAwMF0uYXN0eXBlKG5wLmZsb2F0MzIpKS50byhkZXYpW05vbmVdCiAgICAgICAgICAgICAgICAgICAgdGYgPSB0b3JjaC5mcm9tX251bXB5KGNbJ2ZvbiddW2s6ayArIDEwMDBdKS50byhkZXYpW05vbmVdCiAgICAgICAgICAgICAgICAgICAgdHJvem9zLmFwcGVuZChtb2RlbG8odHYsIHRhLCB0ZilbMF0uZmxvYXQoKS5jcHUoKS5udW1weSgpKQogICAgICAgICAgICAgICAgcCA9IG5wLmNvbmNhdGVuYXRlKHRyb3pvcykgKiBzZAogICAgICAgICAgICAgICAgb2sgPSBjWydvayddCiAgICAgICAgICAgICAgICBQLmFwcGVuZChwW29rXSk7IFQuYXBwZW5kKGNbJ3knXVtva10pOyBGLmFwcGVuZChjWydmb24nXVtva10pCiAgICAgICAgcmV0dXJuIG5wLmNvbmNhdGVuYXRlKFApLCBucC5jb25jYXRlbmF0ZShUKSwgbnAuY29uY2F0ZW5hdGUoRikKCiAgICBkZWYgaW5mb3JtZShQLCBULCBGLCBlcCk6CiAgICAgICAgciA9IFIuY29ycmVsYWNpb25lcyhQLCBULCBDT0xVTU5BUykKICAgICAgICByZXR1cm4geydlcG9jYXNfaGVjaGFzJzogZXAsICdtZWpvcl9lcG9jYSc6IG1lam9yX2VwLCAndmVudGFuYXNfZW50cmVuYW1pZW50byc6IGludChsZW4oaW5pY2lvcykpLAogICAgICAgICAgICAgICAgJ2hvcmFzX2VudHJlbmFtaWVudG8nOiByb3VuZChkYXRvcy5ob3JhcywgMiksICdjbGlwc192YWxpZGFjaW9uJzogbGVuKGRhdG9zLnZhbCksCiAgICAgICAgICAgICAgICAncl9ib2NhJzogUi5tZWRpYV9ncnVwb3Mociwgeydib2NhJzogQ09MVU1OQVN9LCBDT0xVTU5BUylbJ2JvY2EnXSwgJ3InOiByLAogICAgICAgICAgICAgICAgJ3BlZGFnb2dpYSc6IHBlZGFnb2dpYShQIC8gc2QsIFQgLyBzZCwgRiwgQ09MVU1OQVMsIGZvbmVtYXMpLCAncGVzb3NfY2FwYXMnOiBlbWEudHJvbmNvLnZvei5wZXNvc19jYXBhcygpfQoKICAgIGZvciBlcCBpbiByYW5nZShlcDAsIGEuZXBvY2FzKToKICAgICAgICB0MCA9IHRpbWUudGltZSgpOyByZWQudHJhaW4oKTsgdG90ID0gMC4wCiAgICAgICAgcGVybSA9IHRvcmNoLnJhbmRwZXJtKGxlbihpbmljaW9zKSkKICAgICAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4ocGVybSksIGEubG90ZSk6CiAgICAgICAgICAgIHZveiwgYXJ0LCBmb24sIHksIG0gPSBkYXRvcy5sb3RlKGluaWNpb3NbcGVybVtpOmkgKyBhLmxvdGVdXSwgZGV2KQogICAgICAgICAgICB2b3ogPSBSLmF1bWVudGFyX3Zveih2b3osIEQuTCwgZGV2KQogICAgICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KCdjdWRhJyBpZiBncHUgZWxzZSAnY3B1JywgZHR5cGU9dGlwbywgZW5hYmxlZD1ncHUpOgogICAgICAgICAgICAgICAgcCA9IHJlZCh2b3osIGFydCwgZm9uKQogICAgICAgICAgICBwID0gcC5mbG9hdCgpCiAgICAgICAgICAgICMgcGVzbyBwZWRhZ8OzZ2ljbzogY2llcnJlIGVuIHAvYi9tIHkgcmVkb25kZW8gZW4gdS9vL3cKICAgICAgICAgICAgdyA9IHBlc29fY29sW05vbmUsIE5vbmUsIDpdLmV4cGFuZChwLnNoYXBlWzBdLCBELkwsIC0xKS5jbG9uZSgpCiAgICAgICAgICAgIHcgPSB3ICsgKGEucGVzb19wZWRhZ29naWNvIC0gMSkgKiAodG9yY2guaXNpbihmb24sIGZvbl9jaWVycmUpWy4uLiwgTm9uZV0gJiBlc19jaWVycmUpLmZsb2F0KCkKICAgICAgICAgICAgdyA9IHcgKyAoYS5wZXNvX3BlZGFnb2dpY28gLSAxKSAqICh0b3JjaC5pc2luKGZvbiwgZm9uX3JlZG9uZGVvKVsuLi4sIE5vbmVdICYgZXNfcmVkb25kZW8pLmZsb2F0KCkKICAgICAgICAgICAgd20gPSB3ICogbQogICAgICAgICAgICBsMSA9ICgocCAtIHkpICoqIDIgKiB3bSkuc3VtKCkgLyB3bS5zdW0oKS5jbGFtcChtaW49MSkKICAgICAgICAgICAgZHYgPSAocFs6LCAxOl0gLSBwWzosIDotMV0pIC0gKHlbOiwgMTpdIC0geVs6LCA6LTFdKQogICAgICAgICAgICB3diA9IHdtWzosIDE6XSAqIG1bOiwgOi0xXQogICAgICAgICAgICBsMiA9IChkdiAqKiAyICogd3YpLnN1bSgpIC8gd3Yuc3VtKCkuY2xhbXAobWluPTEpCiAgICAgICAgICAgIGxvc3MgPSBsMSArIDAuNSAqIGwyCiAgICAgICAgICAgIG9wdC56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICAgICAgZXNjYWxhLnNjYWxlKGxvc3MpLmJhY2t3YXJkKCk7IGVzY2FsYS51bnNjYWxlXyhvcHQpCiAgICAgICAgICAgIG5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyhyZWQucGFyYW1ldGVycygpLCAxLjApCiAgICAgICAgICAgIGVzY2FsYS5zdGVwKG9wdCk7IGVzY2FsYS51cGRhdGUoKTsgc2NoZWQuc3RlcCgpOyBwYXNvICs9IDEKICAgICAgICAgICAgUi5lbWFfYWN0dWFsaXphcihlbWEsIHJlZCwgcGFzbykKICAgICAgICAgICAgdG90ICs9IGxvc3MuaXRlbSgpICogcC5zaGFwZVswXQogICAgICAgIFAsIFQsIEYgPSB2YWxpZGFyKGVtYSkKICAgICAgICBpbmYgPSBpbmZvcm1lKFAsIFQsIEYsIGVwICsgMSkKICAgICAgICBub3RhID0gaW5mWydyX2JvY2EnXSBpZiBpbmZbJ3JfYm9jYSddIGlzIG5vdCBOb25lIGVsc2UgLTEuMAogICAgICAgIGxpbmVhID0geydlcG9jYSc6IGVwICsgMSwgJ3BlcmRpZGEnOiByb3VuZCh0b3QgLyBsZW4ocGVybSksIDQpLCAncl9ib2NhJzogaW5mWydyX2JvY2EnXSwKICAgICAgICAgICAgICAgICAncGVkYWdvZ2lhJzogaW5mWydwZWRhZ29naWEnXSwgJ2xyJzogcm91bmQoc2NoZWQuZ2V0X2xhc3RfbHIoKVswXSwgNiksICdzJzogcm91bmQodGltZS50aW1lKCkgLSB0MCwgMSl9CiAgICAgICAgaWYgbm90YSA+IG1lam9yOgogICAgICAgICAgICBtZWpvciwgbWVqb3JfZXAsIHNpbl9tZWpvcmEgPSBub3RhLCBlcCArIDEsIDAKICAgICAgICAgICAgdG9yY2guc2F2ZShlbWEuc3RhdGVfZGljdCgpLCBvcy5wYXRoLmpvaW4oYS5zYWxpZGEsICdtb2RlbG8ucHQnKSkKICAgICAgICAgICAgbGluZWFbJ21lam9yJ10gPSBUcnVlCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2luX21lam9yYSArPSAxCiAgICAgICAgaW5mWydtZWpvcl9lcG9jYSddID0gbWVqb3JfZXAKICAgICAgICBqc29uLmR1bXAoaW5mLCBvcGVuKG9zLnBhdGguam9pbihhLnNhbGlkYSwgJ2luZm9ybWUuanNvbicpLCAndycpLCBpbmRlbnQ9MSkKICAgICAgICBSLmd1YXJkYXJfZXN0YWRvKG9zLnBhdGguam9pbihhLnNhbGlkYSwgJ3VsdGltby5wdCcpLCByZWQsIGVtYSwgb3B0LCBzY2hlZCwgZXNjYWxhLAogICAgICAgICAgICAgICAgICAgICAgICAgZXBvY2E9ZXAgKyAxLCBtZWpvcj1tZWpvciwgbWVqb3JfZXA9bWVqb3JfZXAsIHNpbl9tZWpvcmE9c2luX21lam9yYSwgcGFzbz1wYXNvKQogICAgICAgIFIuYW5vdGFyKGEuc2FsaWRhLCBsaW5lYSkKICAgICAgICBpZiBzaW5fbWVqb3JhID49IGEucGFjaWVuY2lhOgogICAgICAgICAgICBwcmludChmJ3BhcmFkYSB0ZW1wcmFuYTogc2luIG1lam9yYXIgZGVzZGUgbGEgw6lwb2NhIHttZWpvcl9lcH0nLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBicmVhawogICAgcHJpbnQoJ21lam9yIMOpcG9jYScsIG1lam9yX2VwLCAnciBib2NhJywgcm91bmQobWVqb3IsIDMpLCBmbHVzaD1UcnVlKQogICAgcHJpbnQoJ0ZJTicsIGZsdXNoPVRydWUpCgoKZGVmIGRhdG9zX3YzX3ByaW1lcmEoZGlyX3Zveik6CiAgICByZXR1cm4gbmV4dChpdGVyKEQubWFwYV9hcmNoaXZvcyhkaXJfdm96KS52YWx1ZXMoKSkpCgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIG1haW4oKQo=", "tools/cara_parlante/voz3/entrenar_noverbal.py": "IiIiTm8gdmVyYmFsIHYzOiBjZWphcywgb2pvcyAocGFycGFkZW8sIGVudHJlY2VycmFyLCBhYnJpciksIHNvbnJpc2EgeSBjYWJlemEgY29uIHVuIG1vZGVsbwpHRU5FUkFUSVZPIGNvbmRpY2lvbmFkbyBwb3IgbGEgdm96OiBmbG93IG1hdGNoaW5nIChyZWN0aWZpZWQgZmxvdykgMUQgY29uIHRyYW5zZm9ybWVyLgoKUG9yIHF1w6kgZ2VuZXJhdGl2bzogbGEgdm96IG5vIGRldGVybWluYSBjZWphcywgcGFycGFkZW8gbmkgY2FiZXphOyB1bmEgcmVncmVzacOzbiBkYSBsYSBtZWRpYQooY2FyYSBwcm9tZWRpYWRhLCBzaW4gdmlkYSkuIEVsIGNhbXBvIGV2YWzDumEgZXN0byBjb24gRkRELCBkaXZlcnNpZGFkIHkgYWxpbmVhY2nDs24gY29uIGxvcyBnb2xwZXMKZGUgdm96LCBubyBjb24gciAoRGlmZlBvc2VUYWxrLCBBUlRhbGspLiBBcXXDrTogeDEgPSBsYXMgMTggY29sdW1uYXMgbm8gdmVyYmFsZXMgKEFSS2l0IHbDrWEgZWwKbWFwYSBTTUlSSywgbWVub3MgbGEgbWVkaWEgZGVsIGNsaXAsIGVuIHNkKSwgeDAgPSBydWlkbzsgbGEgcmVkIGFwcmVuZGUgdiA9IHgxIOKIkiB4MCBlbiB4X3QKY29uZGljaW9uYWRhIHBvciBsYSB2b3ogKEh1QkVSVCwgc3VtYSBkZSBjYXBhcyksIMKraGFibGHCuyArIGFydGljdWxhY2nDs24sIHkgdW4gdmVjdG9yIGRlIEVTVElMTwooc2QgZGUgY2FkYSBjYW5hbCBlbiBlbCBjbGlwOiBjdcOhbnRvIHNlIG11ZXZlIGVzZSBoYWJsYW50ZSksIHF1ZSBzZSBhcGFnYSBlbCAyMCAlIGRlIGxhcyB2ZWNlcwpwYXJhIHBvZGVyIG11ZXN0cmVhciBzaW4gw6lsLiBMYSB2b3ogc2UgYXBhZ2EgZWwgMTAgJSBwYXJhIGxhIGd1w61hIHNpbiBjbGFzaWZpY2Fkb3IuCgpWYWxpZGFjacOzbiAoRU1BLCBzZW1pbGxhIGZpamEpOiBww6lyZGlkYSBkZSBmbHVqbzsgeSBzb2JyZSB1bmEgbXVlc3RyYSBwb3IgY2xpcCAoRXVsZXIgMzIgcGFzb3MpOgpyIHBvciBncnVwbyAob3JpZW50YXRpdm8pLCBGREQgKGRpc3RhbmNpYSBkZSBGcsOpY2hldCwgY292YXJpYW56YSBkaWFnb25hbCwgZW50cmUgbGEgc2QgcG9yIGNsaXAKZGUgY2FkYSBjYW5hbCByZWFsIHkgZ2VuZXJhZGEpLCBkaXZlcnNpZGFkIChSTVMgZW50cmUgZG9zIHNlbWlsbGFzLCBlbiBzZCkgeSBhbGluZWFjacOzbiBjb24gbG9zCmdvbHBlcyBkZSB2b3ogKHBpY29zIGRlIHN1YmlkYSBkZSBkQiBkZWwgLm5weiBkZSBtZWRpZGFzIHZzIG3DrW5pbW9zIGRlIHZlbG9jaWRhZCBkZSBsYSBjYWJlemE7CnNlIGRhIHRhbWJpw6luIGxhIGRlIGxhcyBjYXJhcyByZWFsZXMgY29tbyByZWZlcmVuY2lhKS4gRWwgbWVqb3IgbW9kZWxvIGVzIGVsIGRlIG1lbm9yIHDDqXJkaWRhCmRlIHZhbGlkYWNpw7NuLiBHdWFyZGEgbW9kZWxvLnB0LCBub3JtYWxpemFjaW9uLmpzb24gKGNvbiBlc3RpbG9zIHJlYWxlcyBwYXJhIG11ZXN0cmVhciksCmluZm9ybWUuanNvbiwgcHJvZ3Jlc28uanNvbiB5IHVsdGltby5wdCAocmVhbnVkYSBzb2xvKS4KClVzbzogcHl0aG9uIGVudHJlbmFyX25vdmVyYmFsLnB5IC0tdm96IERJUiAtLWFydCBESVIgLS1zbWlyayBESVIgLS1tZWRpZGFzIERJUiAtLW1hcGEgbWFwYV9hcmtpdC5qc29uCiAgICAgICAgLS1zYWxpZGEgRElSIFstLWNhY2hlIERJUl0gWy0tZXBvY2FzIDMwMF0gWy0tcGFjaWVuY2lhIDMwXSBbLS1sb3RlIDQ4XSBbLS1sciAzZS00XQogICAgICAgIFstLWNhcGFzX3VzYXIgLi4uXSBbLS12b3pfZW5fZ3B1XSBbLS1tYXhfaG9yYXMgMF0gWy0tZCAyNTYgLS1jYXBhcyA0XSBbLS1wYXNvcyAzMl0gWy0tZ3VpYSAxLjVdCiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNvcHkKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgppZiBoYXNhdHRyKHN5cy5zdGRvdXQsICdyZWNvbmZpZ3VyZScpOgogICAgc3lzLnN0ZG91dC5yZWNvbmZpZ3VyZShlbmNvZGluZz0ndXRmLTgnLCBlcnJvcnM9J3JlcGxhY2UnKQpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKQppbXBvcnQgZGF0b3NfdjMgYXMgRCAgICAgICMgbm9xYTogRTQwMgppbXBvcnQgbWFwYV9hcmtpdCAgICAgICAgICMgbm9xYTogRTQwMgppbXBvcnQgcmVkX3YzIGFzIFIgICAgICAgICMgbm9xYTogRTQwMgoKQ09MVU1OQVMgPSBELk5PX1ZFUkJBTApHUlVQT1MgPSB7ZzogRC5HUlVQT1NbZ10gZm9yIGcgaW4gKCdzb25yaXNhJywgJ29qb3MnLCAnY2VqYXMnLCAnY2FiZXphJyl9CkNBQkVaQSA9IFtDT0xVTU5BUy5pbmRleChjKSBmb3IgYyBpbiBELkdSVVBPU1snY2FiZXphJ11dCgoKZGVmIHBpY29zKHgsIHVtYnJhbCwgc2VwPTUpOgogICAgIiIiw41uZGljZXMgZGUgbcOheGltb3MgbG9jYWxlcyBkZSB4IHBvciBlbmNpbWEgZGUgdW1icmFsLCBzZXBhcmFkb3MgYWwgbWVub3Mgc2VwLiIiIgogICAgaWR4ID0gW2kgZm9yIGkgaW4gcmFuZ2UoMSwgbGVuKHgpIC0gMSkgaWYgeFtpXSA+IHVtYnJhbCBhbmQgeFtpXSA+PSB4W2kgLSAxXSBhbmQgeFtpXSA+PSB4W2kgKyAxXV0KICAgIG91dCA9IFtdCiAgICBmb3IgaSBpbiBpZHg6CiAgICAgICAgaWYgbm90IG91dCBvciBpIC0gb3V0Wy0xXSA+PSBzZXA6CiAgICAgICAgICAgIG91dC5hcHBlbmQoaSkKICAgIHJldHVybiBucC5hcnJheShvdXQpCgoKZGVmIGdvbHBlc192b3ooZGIpOgogICAgIiIiR29scGVzIGRlIHZvejogcGljb3MgZGUgbGEgc3ViaWRhIGRlIGVuZXJnw61hIChkQikgcG9yIGVuY2ltYSBkZSAxIHNkLiIiIgogICAgZCA9IG5wLmRpZmYoZGIsIHByZXBlbmQ9ZGJbMF0pCiAgICBkID0gbnAuY29udm9sdmUoZCwgbnAub25lcygzKSAvIDMsIG1vZGU9J3NhbWUnKQogICAgcmV0dXJuIHBpY29zKGQsIGQubWVhbigpICsgZC5zdGQoKSkKCgpkZWYgYWxpbmVhY2lvbihtb3YsIGdvbHBlcywgc2lnbWE9My4wKToKICAgICIiIkJlYXQgYWxpZ24gc2NvcmUgKExpIGV0IGFsLiAyMDIxKTogcGFyYSBjYWRhIGdvbHBlIGRlIHZveiwgZXhwKOKIkmTCsi8yz4PCsikgY29uIGQgbGEgZGlzdGFuY2lhIGFsCiAgICBnb2xwZSBjaW5lbcOhdGljbyBtw6FzIGNlcmNhbm8gKG3DrW5pbW8gbG9jYWwgZGUgbGEgdmVsb2NpZGFkIGRlIGxhIGNhYmV6YSkuIiIiCiAgICBpZiBsZW4oZ29scGVzKSA9PSAwIG9yIGxlbihtb3YpIDwgMzoKICAgICAgICByZXR1cm4gZmxvYXQoJ25hbicpCiAgICB2ZWwgPSBucC5saW5hbGcubm9ybShucC5kaWZmKG1vdiwgYXhpcz0wKSwgYXhpcz0xKQogICAgY2luID0gcGljb3MoLXZlbCwgLW5wLmluZikgICAgICAgICAgICAgICAgICAgICAjIG3DrW5pbW9zIGRlIHZlbG9jaWRhZAogICAgaWYgbGVuKGNpbikgPT0gMDoKICAgICAgICByZXR1cm4gMC4wCiAgICBkID0gbnAuYWJzKGdvbHBlc1s6LCBOb25lXSAtIGNpbltOb25lLCA6XSkubWluKDEpCiAgICByZXR1cm4gZmxvYXQobnAubWVhbihucC5leHAoLWQgKiogMiAvICgyICogc2lnbWEgKiogMikpKSkKCgpkZWYgZnJlY2hldF9kaWFnb25hbChBLCBCKToKICAgICIiIkRpc3RhbmNpYSBkZSBGcsOpY2hldCBlbnRyZSBkb3MgbnViZXMgW04sIENdIGNvbiBnYXVzc2lhbmFzIGRlIGNvdmFyaWFuemEgZGlhZ29uYWwuIiIiCiAgICBtYSwgbWIgPSBBLm1lYW4oMCksIEIubWVhbigwKQogICAgdmEsIHZiID0gQS52YXIoMCkgKyAxZS02LCBCLnZhcigwKSArIDFlLTYKICAgIHJldHVybiBmbG9hdCgoKG1hIC0gbWIpICoqIDIpLnN1bSgpICsgKHZhICsgdmIgLSAyICogbnAuc3FydCh2YSAqIHZiKSkuc3VtKCkpCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLXZveicsIHJlcXVpcmVkPVRydWUpOyBhcC5hZGRfYXJndW1lbnQoJy0tYXJ0JywgcmVxdWlyZWQ9VHJ1ZSk7IGFwLmFkZF9hcmd1bWVudCgnLS1zbWlyaycsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tbWVkaWRhcycsIHJlcXVpcmVkPVRydWUsIGhlbHA9J3BhcmEgbG9zIGRCIChnb2xwZXMgZGUgdm96KSBkZSBsb3MgY2xpcHMgZGUgdmFsaWRhY2nDs24nKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLW1hcGEnLCByZXF1aXJlZD1UcnVlKTsgYXAuYWRkX2FyZ3VtZW50KCctLXNhbGlkYScsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tY2FjaGUnLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tZXBvY2FzJywgdHlwZT1pbnQsIGRlZmF1bHQ9MzAwKTsgYXAuYWRkX2FyZ3VtZW50KCctLXBhY2llbmNpYScsIHR5cGU9aW50LCBkZWZhdWx0PTMwKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLWxvdGUnLCB0eXBlPWludCwgZGVmYXVsdD00OCk7IGFwLmFkZF9hcmd1bWVudCgnLS1scicsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9M2UtNCkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1jYXBhc191c2FyJywgZGVmYXVsdD0nJyk7IGFwLmFkZF9hcmd1bWVudCgnLS12b3pfZW5fZ3B1JywgYWN0aW9uPSdzdG9yZV90cnVlJykKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1tYXhfaG9yYXMnLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTApCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tZCcsIHR5cGU9aW50LCBkZWZhdWx0PTI1Nik7IGFwLmFkZF9hcmd1bWVudCgnLS1jYXBhcycsIHR5cGU9aW50LCBkZWZhdWx0PTQpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tcGFzb3MnLCB0eXBlPWludCwgZGVmYXVsdD0zMik7IGFwLmFkZF9hcmd1bWVudCgnLS1ndWlhJywgdHlwZT1mbG9hdCwgZGVmYXVsdD0xLjUpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tZGlzcG9zaXRpdm8nLCBkZWZhdWx0PSdjdWRhJyBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgJ2NwdScpCiAgICBhID0gYXAucGFyc2VfYXJncygpCiAgICBvcy5tYWtlZGlycyhhLnNhbGlkYSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIFIuc2VtaWxsYXMoMCkKICAgIGRldiA9IGEuZGlzcG9zaXRpdm87IGdwdSA9IGRldi5zdGFydHN3aXRoKCdjdWRhJykKICAgIG1hcGEgPSBtYXBhX2Fya2l0LmNhcmdhcihhLm1hcGEpCiAgICBzZWwgPSBbbWFwYVsnY29sdW1uYXMnXS5pbmRleChjKSBmb3IgYyBpbiBDT0xVTU5BU10KICAgIGNhcGFzX3VzYXIgPSBbaW50KGMpIGZvciBjIGluIGEuY2FwYXNfdXNhci5zcGxpdCgnLCcpXSBpZiBhLmNhcGFzX3VzYXIgZWxzZSBOb25lCiAgICBtZWRpZGFzID0gRC5tYXBhX2FyY2hpdm9zKGEubWVkaWRhcykKCiAgICBkYXRvcyA9IEQuQ29uanVudG8oRC5wYXJlc19lbnRyZW5hbWllbnRvKGEudm96LCBhLmFydCwgYS5zbWlyayksIG1hcGEsIGEuY2FjaGUgb3Igb3MucGF0aC5qb2luKGEuc2FsaWRhLCAnY2FjaGUnKSwKICAgICAgICAgICAgICAgICAgICAgICBjYXBhc191c2FyLCBhLm1heF9ob3JhcykKICAgIGRhdG9zLnkgPSBkYXRvcy55WzosIHNlbF07IGRhdG9zLnNkID0gZGF0b3Muc2Rbc2VsXQogICAgZm9yIGMgaW4gZGF0b3MudmFsOgogICAgICAgIGNbJ3knXSA9IGNbJ3knXVs6LCBzZWxdCiAgICAgICAgY1snZGInXSA9IG5wLmxvYWQobWVkaWRhc1tjWydub21icmUnXV0pWydkYiddWzpsZW4oY1snb2snXSldIGlmIGNbJ25vbWJyZSddIGluIG1lZGlkYXMgZWxzZSBOb25lCiAgICBkYXRvcy5hX3RvcmNoKGRldiwgYS52b3pfZW5fZ3B1KQogICAgc2QgPSBkYXRvcy5zZDsgQyA9IGxlbihDT0xVTU5BUyk7IEsgPSBkYXRvcy52b3ouc2hhcGVbMV0KICAgICMgZXN0aWxvIHBvciB2ZW50YW5hOiBzZCBkZSBjYWRhIGNhbmFsIGVuIGxhIHZlbnRhbmEgKGVuIHNkIGdsb2JhbGVzKSwgZ3VhcmRhZG8gcG9yIHZlbnRhbmEKICAgIGVzdGlsb3MgPSBucC56ZXJvcygobGVuKGRhdG9zLmluaWNpb3MpLCBDKSwgbnAuZmxvYXQzMikKICAgIGZvciBqLCBpIGluIGVudW1lcmF0ZShkYXRvcy5pbmljaW9zKToKICAgICAgICBtID0gZGF0b3MubVtpOmkgKyBELkxdCiAgICAgICAgZXN0aWxvc1tqXSA9IChkYXRvcy55W2k6aSArIEQuTF1bbV0gLyBzZCkuc3RkKDApIGlmIG0uc3VtKCkgPiAxMCBlbHNlIDEuMAogICAgZXN0aWxvc190ID0gdG9yY2guZnJvbV9udW1weShlc3RpbG9zKQogICAgZXN0aWxvc192YWwgPSBbKChjWyd5J11bY1snb2snXV0gLyBzZCkuc3RkKDApKS5hc3R5cGUobnAuZmxvYXQzMikgZm9yIGMgaW4gZGF0b3MudmFsXQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDApCiAgICBtdWVzdHJhX2VzdGlsb3MgPSBlc3RpbG9zW3JuZy5jaG9pY2UobGVuKGVzdGlsb3MpLCBtaW4oMTYsIGxlbihlc3RpbG9zKSksIHJlcGxhY2U9RmFsc2UpXS50b2xpc3QoKQoKICAgIHJlZCA9IFIuTm9WZXJiYWwoSywgQywgYS5kLCBhLmNhcGFzLCBuX2VzdGlsbz1DKS50byhkZXYpCiAgICBlbWEgPSBjb3B5LmRlZXBjb3B5KHJlZCkuZXZhbCgpCiAgICBmb3IgcCBpbiBlbWEucGFyYW1ldGVycygpOgogICAgICAgIHAucmVxdWlyZXNfZ3JhZF8oRmFsc2UpCiAgICBvcHQgPSB0b3JjaC5vcHRpbS5BZGFtVyhyZWQucGFyYW1ldGVycygpLCBhLmxyLCB3ZWlnaHRfZGVjYXk9MC4wMikKICAgIHBhc29zX2VwID0gKGxlbihkYXRvcy5pbmljaW9zKSArIGEubG90ZSAtIDEpIC8vIGEubG90ZQogICAgc2NoZWQgPSBSLnBsYW5pZmljYWRvcihvcHQsIHBhc29zX2VwLCBhLmVwb2NhcykKICAgIGJmMTYgPSBncHUgYW5kIHRvcmNoLmN1ZGEuaXNfYmYxNl9zdXBwb3J0ZWQoKQogICAgdGlwbyA9IHRvcmNoLmJmbG9hdDE2IGlmIGJmMTYgZWxzZSB0b3JjaC5mbG9hdDE2CiAgICBlc2NhbGEgPSB0b3JjaC5hbXAuR3JhZFNjYWxlcihlbmFibGVkPWdwdSBhbmQgbm90IGJmMTYpCiAgICBlc3RhZG8gPSBSLmNhcmdhcl9lc3RhZG8ob3MucGF0aC5qb2luKGEuc2FsaWRhLCAndWx0aW1vLnB0JyksIHJlZCwgZW1hLCBvcHQsIHNjaGVkLCBlc2NhbGEsIGRldikgb3Ige30KICAgIGVwMCwgbWVqb3IsIG1lam9yX2VwLCBzaW5fbWVqb3JhLCBwYXNvID0gKGVzdGFkby5nZXQoaywgdikgZm9yIGssIHYgaW4KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICgoJ2Vwb2NhJywgMCksICgnbWVqb3InLCAxZTkpLCAoJ21lam9yX2VwJywgMCksICgnc2luX21lam9yYScsIDApLCAoJ3Bhc28nLCAwKSkpCiAgICBpZiBlc3RhZG86CiAgICAgICAgcHJpbnQoZidyZWFudWRhbmRvIGRlc2RlIGxhIMOpcG9jYSB7ZXAwfSAobWVqb3IgcMOpcmRpZGEgdmFsIHttZWpvcjouNGZ9IGVuIGxhIHttZWpvcl9lcH0pJywgZmx1c2g9VHJ1ZSkKICAgIHByaW1lcmEgPSBucC5sb2FkKG5leHQoaXRlcihELm1hcGFfYXJjaGl2b3MoYS52b3opLnZhbHVlcygpKSkpCiAgICBqc29uLmR1bXAoeydjb2x1bW5hcyc6IENPTFVNTkFTLCAnc2QnOiBzZC50b2xpc3QoKSwgJ2NhcGFzJzogW2ludChjKSBmb3IgYyBpbiAoY2FwYXNfdXNhciBvciBwcmltZXJhWydjYXBhcyddKV0sCiAgICAgICAgICAgICAgICdjb2RpZmljYWRvcic6IHN0cihwcmltZXJhWydjb2RpZmljYWRvciddKSwgJ2FycXVpdGVjdHVyYSc6IGYnTm9WZXJiYWwgZmxvdyBkPXthLmR9IGNhcGFzPXthLmNhcGFzfScsCiAgICAgICAgICAgICAgICdkJzogYS5kLCAnY2FwYXNfdHJhbnNmb3JtZXInOiBhLmNhcGFzLCAncGFzb3MnOiBhLnBhc29zLCAnZ3VpYSc6IGEuZ3VpYSwgJ2VzdGlsb3MnOiBtdWVzdHJhX2VzdGlsb3MsCiAgICAgICAgICAgICAgICdlc3RpbG9fbWVkaW8nOiBlc3RpbG9zLm1lYW4oMCkudG9saXN0KCl9LAogICAgICAgICAgICAgIG9wZW4ob3MucGF0aC5qb2luKGEuc2FsaWRhLCAnbm9ybWFsaXphY2lvbi5qc29uJyksICd3JykpCiAgICBpbmljaW9zID0gdG9yY2guZnJvbV9udW1weShkYXRvcy5pbmljaW9zKQoKICAgIGRlZiB0ZW5zb3Jlc19jbGlwKGMpOgogICAgICAgIHR2ID0gdG9yY2guZnJvbV9udW1weShjWyd2b3onXS5hc3R5cGUobnAuZmxvYXQzMikpLnRvKGRldilbTm9uZV0KICAgICAgICB0YSA9IHRvcmNoLmZyb21fbnVtcHkoY1snYXJ0J10uYXN0eXBlKG5wLmZsb2F0MzIpKS50byhkZXYpW05vbmVdCiAgICAgICAgcmV0dXJuIHR2LCB0YSwgdG9yY2guZnJvbV9udW1weShjWydmb24nXSkudG8oZGV2KVtOb25lXQoKICAgIGRlZiBwZXJkaWRhX3ZhbChtb2RlbG8pOgogICAgICAgICIiIlDDqXJkaWRhIGRlIGZsdWpvIGVuIGxhcyB2ZW50YW5hcyBkZSB2YWxpZGFjacOzbiBjb24gdCB5IHJ1aWRvIGZpam9zIChzZW1pbGxhKS4iIiIKICAgICAgICBnID0gdG9yY2guR2VuZXJhdG9yKGRldmljZT1kZXYpLm1hbnVhbF9zZWVkKDEpCiAgICAgICAgdG90LCBuID0gMC4wLCAwCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCksIHRvcmNoLmF1dG9jYXN0KCdjdWRhJyBpZiBncHUgZWxzZSAnY3B1JywgZHR5cGU9dGlwbywgZW5hYmxlZD1ncHUpOgogICAgICAgICAgICBmb3IgYywgZXN0IGluIHppcChkYXRvcy52YWwsIGVzdGlsb3NfdmFsKToKICAgICAgICAgICAgICAgIGZvciBrIGluIHJhbmdlKDAsIGxlbihjWydvayddKSAtIEQuTCArIDEsIEQuTCk6CiAgICAgICAgICAgICAgICAgICAgdHYsIHRhLCB0ZiA9ICh4WzosIGs6ayArIEQuTF0gZm9yIHggaW4gdGVuc29yZXNfY2xpcChjKSkKICAgICAgICAgICAgICAgICAgICB4MSA9IHRvcmNoLmZyb21fbnVtcHkoY1sneSddW2s6ayArIEQuTF0gLyBzZCkudG8oZGV2KVtOb25lXS5mbG9hdCgpCiAgICAgICAgICAgICAgICAgICAgbSA9IHRvcmNoLmZyb21fbnVtcHkoY1snb2snXVtrOmsgKyBELkxdKS50byhkZXYpW05vbmUsIDosIE5vbmVdLmZsb2F0KCkKICAgICAgICAgICAgICAgICAgICB4MCA9IHRvcmNoLnJhbmRuKHgxLnNoYXBlLCBnZW5lcmF0b3I9ZywgZGV2aWNlPWRldik7IHQgPSB0b3JjaC5yYW5kKDEsIGdlbmVyYXRvcj1nLCBkZXZpY2U9ZGV2KQogICAgICAgICAgICAgICAgICAgIHh0ID0gKDEgLSB0KVs6LCBOb25lLCBOb25lXSAqIHgwICsgdFs6LCBOb25lLCBOb25lXSAqIHgxCiAgICAgICAgICAgICAgICAgICAgdiA9IG1vZGVsbyh4dCwgdCwgdHYsIHRhLCB0ZiwgdG9yY2guZnJvbV9udW1weShlc3QpLnRvKGRldilbTm9uZV0pLmZsb2F0KCkKICAgICAgICAgICAgICAgICAgICB0b3QgKz0gKCgodiAtICh4MSAtIHgwKSkgKiogMikgKiBtKS5zdW0oKS5pdGVtKCkgLyBDOyBuICs9IG0uc3VtKCkuaXRlbSgpCiAgICAgICAgcmV0dXJuIHRvdCAvIG1heChuLCAxKQoKICAgIGRlZiBtdWVzdHJhcyhtb2RlbG8sIHNlbWlsbGEpOgogICAgICAgIGcgPSB0b3JjaC5HZW5lcmF0b3IoZGV2aWNlPWRldikubWFudWFsX3NlZWQoc2VtaWxsYSkKICAgICAgICBvdXQgPSBbXQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpLCB0b3JjaC5hdXRvY2FzdCgnY3VkYScgaWYgZ3B1IGVsc2UgJ2NwdScsIGR0eXBlPXRpcG8sIGVuYWJsZWQ9Z3B1KToKICAgICAgICAgICAgZm9yIGMsIGVzdCBpbiB6aXAoZGF0b3MudmFsLCBlc3RpbG9zX3ZhbCk6CiAgICAgICAgICAgICAgICB0diwgdGEsIHRmID0gdGVuc29yZXNfY2xpcChjKQogICAgICAgICAgICAgICAgeCA9IG1vZGVsby5tdWVzdHJlYXIodHYsIHRhLCB0ZiwgdG9yY2guZnJvbV9udW1weShlc3QpLnRvKGRldilbTm9uZV0sIGEucGFzb3MsIGEuZ3VpYSwgZykKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoeFswXS5mbG9hdCgpLmNwdSgpLm51bXB5KCkgKiBzZCkKICAgICAgICByZXR1cm4gb3V0CgogICAgZGVmIGV2YWx1YXIobW9kZWxvLCBlcCk6CiAgICAgICAgcHYgPSBwZXJkaWRhX3ZhbChtb2RlbG8pCiAgICAgICAgUzEsIFMyID0gbXVlc3RyYXMobW9kZWxvLCAxKSwgbXVlc3RyYXMobW9kZWxvLCAyKQogICAgICAgIFAgPSBucC5jb25jYXRlbmF0ZShbc1tjWydvayddXSBmb3IgcywgYyBpbiB6aXAoUzEsIGRhdG9zLnZhbCldKQogICAgICAgIFQgPSBucC5jb25jYXRlbmF0ZShbY1sneSddW2NbJ29rJ11dIGZvciBjIGluIGRhdG9zLnZhbF0pCiAgICAgICAgciA9IFIuY29ycmVsYWNpb25lcyhQLCBULCBDT0xVTU5BUykKICAgICAgICBkaW5fcmVhbCA9IG5wLnN0YWNrKFsoY1sneSddW2NbJ29rJ11dIC8gc2QpLnN0ZCgwKSBmb3IgYyBpbiBkYXRvcy52YWxdKQogICAgICAgIGRpbl9nZW4gPSBucC5zdGFjayhbKHNbY1snb2snXV0gLyBzZCkuc3RkKDApIGZvciBzLCBjIGluIHppcChTMSwgZGF0b3MudmFsKV0pCiAgICAgICAgZGl2ID0gZmxvYXQobnAubWVhbihbbnAuc3FydChucC5tZWFuKCgoczEgLSBzMikgLyBzZCkgKiogMikpIGZvciBzMSwgczIgaW4gemlwKFMxLCBTMildKSkKICAgICAgICBhbF9nZW4sIGFsX3JlYWwgPSBbXSwgW10KICAgICAgICBmb3IgcywgYyBpbiB6aXAoUzEsIGRhdG9zLnZhbCk6CiAgICAgICAgICAgIGlmIGNbJ2RiJ10gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBndiA9IGdvbHBlc192b3ooY1snZGInXSkKICAgICAgICAgICAgICAgIGFsX2dlbi5hcHBlbmQoYWxpbmVhY2lvbihzWzosIENBQkVaQV0sIGd2KSk7IGFsX3JlYWwuYXBwZW5kKGFsaW5lYWNpb24oY1sneSddWzosIENBQkVaQV0sIGd2KSkKICAgICAgICByZXR1cm4geydlcG9jYXNfaGVjaGFzJzogZXAsICdwZXJkaWRhX3ZhbCc6IHJvdW5kKHB2LCA0KSwgJ3JfZ3J1cG9zJzogUi5tZWRpYV9ncnVwb3MociwgR1JVUE9TLCBDT0xVTU5BUyksICdyJzogciwKICAgICAgICAgICAgICAgICdmZGQnOiByb3VuZChmcmVjaGV0X2RpYWdvbmFsKGRpbl9yZWFsLCBkaW5fZ2VuKSwgNCksICdkaXZlcnNpZGFkJzogcm91bmQoZGl2LCAzKSwKICAgICAgICAgICAgICAgICdhbGluZWFjaW9uX2dvbHBlcyc6IHsncmVkJzogcm91bmQoZmxvYXQobnAubmFubWVhbihhbF9nZW4pKSwgMykgaWYgYWxfZ2VuIGVsc2UgTm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAncmVhbCc6IHJvdW5kKGZsb2F0KG5wLm5hbm1lYW4oYWxfcmVhbCkpLCAzKSBpZiBhbF9yZWFsIGVsc2UgTm9uZX0sCiAgICAgICAgICAgICAgICAnY2xpcHNfdmFsaWRhY2lvbic6IGxlbihkYXRvcy52YWwpLCAndmVudGFuYXNfZW50cmVuYW1pZW50byc6IGludChsZW4oaW5pY2lvcykpLAogICAgICAgICAgICAgICAgJ2hvcmFzX2VudHJlbmFtaWVudG8nOiByb3VuZChkYXRvcy5ob3JhcywgMiksICdwZXNvc19jYXBhcyc6IG1vZGVsby50cm9uY28udm96LnBlc29zX2NhcGFzKCl9CgogICAgZm9yIGVwIGluIHJhbmdlKGVwMCwgYS5lcG9jYXMpOgogICAgICAgIHQwID0gdGltZS50aW1lKCk7IHJlZC50cmFpbigpOyB0b3QgPSAwLjAKICAgICAgICBwZXJtID0gdG9yY2gucmFuZHBlcm0obGVuKGluaWNpb3MpKQogICAgICAgIGZvciBpIGluIHJhbmdlKDAsIGxlbihwZXJtKSwgYS5sb3RlKToKICAgICAgICAgICAgaWRzID0gcGVybVtpOmkgKyBhLmxvdGVdCiAgICAgICAgICAgIHZveiwgYXJ0LCBmb24sIHgxLCBtID0gZGF0b3MubG90ZShpbmljaW9zW2lkc10sIGRldikKICAgICAgICAgICAgQiA9IHgxLnNoYXBlWzBdCiAgICAgICAgICAgIHZveiA9IFIuYXVtZW50YXJfdm96KHZveiwgRC5MLCBkZXYsIHBfYXBhZ2FyPTAuMCkKICAgICAgICAgICAgZXN0ID0gZXN0aWxvc190W2lkc10udG8oZGV2KSAqICh0b3JjaC5yYW5kKEIsIDEsIGRldmljZT1kZXYpID4gMC4yKSAgICAgICAgICAjIGVzdGlsbyBhcGFnYWRvIGEgdmVjZXMKICAgICAgICAgICAgc2luX3ZveiA9IHRvcmNoLnJhbmQoQiwgZGV2aWNlPWRldikgPCAwLjEgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIHBhcmEgbGEgZ3XDrWEgc2luIGNsYXNpZmljYWRvcgogICAgICAgICAgICB4MCA9IHRvcmNoLnJhbmRuX2xpa2UoeDEpOyB0ID0gdG9yY2gucmFuZChCLCBkZXZpY2U9ZGV2KQogICAgICAgICAgICB4dCA9ICgxIC0gdClbOiwgTm9uZSwgTm9uZV0gKiB4MCArIHRbOiwgTm9uZSwgTm9uZV0gKiB4MQogICAgICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KCdjdWRhJyBpZiBncHUgZWxzZSAnY3B1JywgZHR5cGU9dGlwbywgZW5hYmxlZD1ncHUpOgogICAgICAgICAgICAgICAgdiA9IHJlZCh4dCwgdCwgdm96LCBhcnQsIGZvbiwgZXN0LCBzaW5fdm96KQogICAgICAgICAgICBsb3NzID0gKCgodi5mbG9hdCgpIC0gKHgxIC0geDApKSAqKiAyKSAqIG0pLnN1bSgpIC8gKG0uc3VtKCkgKiBDKS5jbGFtcChtaW49MSkKICAgICAgICAgICAgb3B0Lnplcm9fZ3JhZChzZXRfdG9fbm9uZT1UcnVlKQogICAgICAgICAgICBlc2NhbGEuc2NhbGUobG9zcykuYmFja3dhcmQoKTsgZXNjYWxhLnVuc2NhbGVfKG9wdCkKICAgICAgICAgICAgbm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKHJlZC5wYXJhbWV0ZXJzKCksIDEuMCkKICAgICAgICAgICAgZXNjYWxhLnN0ZXAob3B0KTsgZXNjYWxhLnVwZGF0ZSgpOyBzY2hlZC5zdGVwKCk7IHBhc28gKz0gMQogICAgICAgICAgICBSLmVtYV9hY3R1YWxpemFyKGVtYSwgcmVkLCBwYXNvKQogICAgICAgICAgICB0b3QgKz0gbG9zcy5pdGVtKCkgKiBCCiAgICAgICAgaW5mID0gZXZhbHVhcihlbWEsIGVwICsgMSkKICAgICAgICBsaW5lYSA9IHsnZXBvY2EnOiBlcCArIDEsICdwZXJkaWRhJzogcm91bmQodG90IC8gbGVuKHBlcm0pLCA0KSwgJ3BlcmRpZGFfdmFsJzogaW5mWydwZXJkaWRhX3ZhbCddLCAncic6IGluZlsncl9ncnVwb3MnXSwKICAgICAgICAgICAgICAgICAnZmRkJzogaW5mWydmZGQnXSwgJ2RpdmVyc2lkYWQnOiBpbmZbJ2RpdmVyc2lkYWQnXSwgJ2FsaW5lYWNpb24nOiBpbmZbJ2FsaW5lYWNpb25fZ29scGVzJ10sCiAgICAgICAgICAgICAgICAgJ2xyJzogcm91bmQoc2NoZWQuZ2V0X2xhc3RfbHIoKVswXSwgNiksICdzJzogcm91bmQodGltZS50aW1lKCkgLSB0MCwgMSl9CiAgICAgICAgaWYgaW5mWydwZXJkaWRhX3ZhbCddIDwgbWVqb3I6CiAgICAgICAgICAgIG1lam9yLCBtZWpvcl9lcCwgc2luX21lam9yYSA9IGluZlsncGVyZGlkYV92YWwnXSwgZXAgKyAxLCAwCiAgICAgICAgICAgIHRvcmNoLnNhdmUoZW1hLnN0YXRlX2RpY3QoKSwgb3MucGF0aC5qb2luKGEuc2FsaWRhLCAnbW9kZWxvLnB0JykpCiAgICAgICAgICAgIGxpbmVhWydtZWpvciddID0gVHJ1ZQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNpbl9tZWpvcmEgKz0gMQogICAgICAgIGluZlsnbWVqb3JfZXBvY2EnXSA9IG1lam9yX2VwCiAgICAgICAganNvbi5kdW1wKGluZiwgb3Blbihvcy5wYXRoLmpvaW4oYS5zYWxpZGEsICdpbmZvcm1lLmpzb24nKSwgJ3cnKSwgaW5kZW50PTEpCiAgICAgICAgUi5ndWFyZGFyX2VzdGFkbyhvcy5wYXRoLmpvaW4oYS5zYWxpZGEsICd1bHRpbW8ucHQnKSwgcmVkLCBlbWEsIG9wdCwgc2NoZWQsIGVzY2FsYSwKICAgICAgICAgICAgICAgICAgICAgICAgIGVwb2NhPWVwICsgMSwgbWVqb3I9bWVqb3IsIG1lam9yX2VwPW1lam9yX2VwLCBzaW5fbWVqb3JhPXNpbl9tZWpvcmEsIHBhc289cGFzbykKICAgICAgICBSLmFub3RhcihhLnNhbGlkYSwgbGluZWEpCiAgICAgICAgaWYgc2luX21lam9yYSA+PSBhLnBhY2llbmNpYToKICAgICAgICAgICAgcHJpbnQoZidwYXJhZGEgdGVtcHJhbmE6IHNpbiBtZWpvcmFyIGRlc2RlIGxhIMOpcG9jYSB7bWVqb3JfZXB9JywgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgYnJlYWsKICAgIHByaW50KCdtZWpvciDDqXBvY2EnLCBtZWpvcl9lcCwgJ3DDqXJkaWRhIHZhbCcsIHJvdW5kKG1lam9yLCA0KSwgZmx1c2g9VHJ1ZSkKICAgIHByaW50KCdGSU4nLCBmbHVzaD1UcnVlKQoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICBtYWluKCkK", "tools/cara_parlante/voz3/estado_v3.py": "IiIiRXN0YWRvIGNvbXBhY3RvIGRlbCBjdWFkZXJubyB2MywgZXNjcml0byBjYWRhIG1pbnV0byBlbiBEcml2ZSAodm96MmNhcmEvdjMvZXN0YWRvLnR4dCkgcGFyYQpzZWd1aXJsbyBkZXNkZSBlbCB0ZWzDqWZvbm8gbyBkZXNkZSBvdHJhIHNlc2nDs24gc2luIGFicmlyIGVsIGxvZzogbMOtbmVhcyBjb3J0YXMsIHNpbiBsaXN0YXMuCgpMZWU6IGVsIGxvZyBkZWwgZ3Vpb24gKC9jb250ZW50L3YzLmxvZzogbGEgw7psdGltYSBsw61uZWEgwqsjIyMgcGFzb8K7IGRpY2UgZW4gcXXDqSBwYXNvIHZhKSwgbG9zCnJlY3VlbnRvcyBkZSBldGlxdWV0YXMgeSByYXNnb3MgaGVjaG9zIGZyZW50ZSBhIGxvcyBxdWUgaGF5IHF1ZSBoYWNlciwgbGEgw7psdGltYSBsw61uZWEgZGUKcHJvZ3Jlc28uanNvbiBkZSBjYWRhIGVudHJlbmFtaWVudG8geSBsYSBHUFUuIFNlIGxhbnphIGNvbiBzZXRzaWQgZGVzZGUgZWwgZ3Vpb247IG11ZXJlIHNvbG8KY3VhbmRvIGFwYXJlY2UgRklOIG8gRVJST1IgZW4gZWwgbG9nICh0cmFzIGVzY3JpYmlyIGVsIGVzdGFkbyBmaW5hbCkuCgpVc286IHB5dGhvbiBlc3RhZG9fdjMucHkgLS1sb2cgL2NvbnRlbnQvdjMubG9nIC0tdjMgL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9saXRlcmFsZXZlbF9jYXJhL3ZvejJjYXJhL3YzCiAgICAgICAgLS1tZWRpZGFzIC9jb250ZW50L2RhdG9zL21lZGlkYXMgWy0tY2FkYSA2MF0KIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgZ2xvYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRpbWUKCgpkZWYgdWx0aW1vKHBhdHJvbiwgcnV0YSk6CiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMocnV0YSk6CiAgICAgICAgcmV0dXJuICcnCiAgICBsaW5lYXMgPSBbbCBmb3IgbCBpbiBvcGVuKHJ1dGEsIGVycm9ycz0ncmVwbGFjZScpIGlmIHBhdHJvbiBpbiBsXQogICAgcmV0dXJuIGxpbmVhc1stMV0uc3RyaXAoKSBpZiBsaW5lYXMgZWxzZSAnJwoKCmRlZiBjdWVudGEoZCk6CiAgICByZXR1cm4gbGVuKGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4oZCwgJyoqJywgJyoubnB6JyksIHJlY3Vyc2l2ZT1UcnVlKSkKCgpkZWYgcHJvZ3Jlc28oZCk6CiAgICBwID0gb3MucGF0aC5qb2luKGQsICdwcm9ncmVzby5qc29uJykKICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhwKToKICAgICAgICByZXR1cm4gJ3NpbiBlbXBlemFyJwogICAgbGluZWFzID0gb3BlbihwKS5yZWFkKCkuc3RyaXAoKS5zcGxpdGxpbmVzKCkKICAgIGlmIG5vdCBsaW5lYXM6CiAgICAgICAgcmV0dXJuICdzaW4gZW1wZXphcicKICAgIHUgPSBqc29uLmxvYWRzKGxpbmVhc1stMV0pCiAgICBpbmYgPSBvcy5wYXRoLmpvaW4oZCwgJ2luZm9ybWUuanNvbicpCiAgICBtZWpvciA9IGpzb24ubG9hZChvcGVuKGluZikpLmdldCgnbWVqb3JfZXBvY2EnLCAnPycpIGlmIG9zLnBhdGguZXhpc3RzKGluZikgZWxzZSAnPycKICAgIGlmICdyX2JvY2EnIGluIHU6CiAgICAgICAgcmV0dXJuIGYiw6lwb2NhIHt1WydlcG9jYSddfSByIGJvY2Ege3VbJ3JfYm9jYSddfSAobWVqb3I6IMOpcG9jYSB7bWVqb3J9KSBwZWRhZ29nw61hIHt1WydwZWRhZ29naWEnXX0ge3VbJ3MnXX0gcy/DqXBvY2EiCiAgICByZXR1cm4gKGYiw6lwb2NhIHt1WydlcG9jYSddfSBww6lyZGlkYSB2YWwge3VbJ3BlcmRpZGFfdmFsJ119IChtZWpvcjogw6lwb2NhIHttZWpvcn0pIHIge3VbJ3InXX0gZmRkIHt1WydmZGQnXX0gIgogICAgICAgICAgICBmImRpdmVyc2lkYWQge3VbJ2RpdmVyc2lkYWQnXX0gYWxpbmVhY2nDs24ge3VbJ2FsaW5lYWNpb24nXX0ge3VbJ3MnXX0gcy/DqXBvY2EiKQoKCmRlZiBncHUoKToKICAgIHRyeToKICAgICAgICByID0gc3VicHJvY2Vzcy5ydW4oWydudmlkaWEtc21pJywgJy0tcXVlcnktZ3B1PXV0aWxpemF0aW9uLmdwdSxtZW1vcnkudXNlZCxtZW1vcnkudG90YWwnLCAnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyJ10sCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSwgdGltZW91dD0xMCkKICAgICAgICByZXR1cm4gci5zdGRvdXQuc3RyaXAoKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gJ3NpbiBudmlkaWEtc21pJwoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1sb2cnLCByZXF1aXJlZD1UcnVlKTsgYXAuYWRkX2FyZ3VtZW50KCctLXYzJywgcmVxdWlyZWQ9VHJ1ZSk7IGFwLmFkZF9hcmd1bWVudCgnLS1tZWRpZGFzJywgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1jYWRhJywgdHlwZT1pbnQsIGRlZmF1bHQ9NjApCiAgICBhID0gYXAucGFyc2VfYXJncygpCiAgICB3aGlsZSBUcnVlOgogICAgICAgIHBhc28gPSB1bHRpbW8oJyMjIyAnLCBhLmxvZykKICAgICAgICBuX21lZCA9IGN1ZW50YShhLm1lZGlkYXMpCiAgICAgICAgbGluZWFzID0gW3RpbWUuc3RyZnRpbWUoJyVIOiVNOiVTJykgKyAnICAnICsgKHBhc28gb3IgJ2FycmFuY2FuZG8nKSwKICAgICAgICAgICAgICAgICAgZidldGlxdWV0YXMgU01JUks6IHtjdWVudGEob3MucGF0aC5qb2luKGEudjMsICJzbWlyayIpKX0ve25fbWVkfSBjbGlwcycsCiAgICAgICAgICAgICAgICAgIGYncmFzZ29zIGRlIHZvejoge2N1ZW50YShvcy5wYXRoLmpvaW4oYS52MywgInZveiIpKX0ve25fbWVkfSBjbGlwcycsCiAgICAgICAgICAgICAgICAgIGYnYXJ0aWN1bGFjacOzbjoge2N1ZW50YShvcy5wYXRoLmpvaW4oYS52MywgImFydCIpKX0ve25fbWVkfSBjbGlwcycsCiAgICAgICAgICAgICAgICAgICdib2NhOiAnICsgcHJvZ3Jlc28ob3MucGF0aC5qb2luKGEudjMsICdib2NhJykpLAogICAgICAgICAgICAgICAgICAnbm8gdmVyYmFsOiAnICsgcHJvZ3Jlc28ob3MucGF0aC5qb2luKGEudjMsICdub3ZlcmJhbCcpKSwKICAgICAgICAgICAgICAgICAgJ0dQVTogJyArIGdwdSgpXQogICAgICAgIGVyciA9IHVsdGltbygnVHJhY2ViYWNrJywgYS5sb2cpIG9yIHVsdGltbygnS2lsbGVkJywgYS5sb2cpCiAgICAgICAgaWYgZXJyOgogICAgICAgICAgICBsaW5lYXMuYXBwZW5kKCdBVklTTyBlbiBlbCBsb2c6ICcgKyBlcnJbOjEyMF0pCiAgICAgICAgdHJ5OgogICAgICAgICAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKGEudjMsICdlc3RhZG8udHh0LnRtcCcpLCAndycpIGFzIGY6CiAgICAgICAgICAgICAgICBmLndyaXRlKCdcbicuam9pbihsaW5lYXMpICsgJ1xuJykKICAgICAgICAgICAgb3MucmVwbGFjZShvcy5wYXRoLmpvaW4oYS52MywgJ2VzdGFkby50eHQudG1wJyksIG9zLnBhdGguam9pbihhLnYzLCAnZXN0YWRvLnR4dCcpKQogICAgICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICBwYXNzCiAgICAgICAgaWYgcGFzby5zdGFydHN3aXRoKCcjIyMgRklOJykgb3IgcGFzby5zdGFydHN3aXRoKCcjIyMgRVJST1InKToKICAgICAgICAgICAgYnJlYWsKICAgICAgICB0aW1lLnNsZWVwKGEuY2FkYSkKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgbWFpbigpCg==", "tools/cara_parlante/voz2cara/extraer.py": "IiIiUmFzZ29zIGRlIHZveiBwYXJhIGVsIG1vZGVsbyB2b3rihpJjYXJhOiB3YXYydmVjMi1iYXNlIChwcmVlbnRyZW5hZG8sIEFwYWNoZS0yLjApIHNvYnJlIGVsCmF1ZGlvIGRlIGNhZGEgdsOtZGVvIG1lZGlkbywgZW4gbGEgbWlzbWEgcmVqaWxsYSBkZSAyNSBmcHMgcXVlIHN1IC5ucHogZGUgbWVkaWRhcy4KClBvciB2w61kZW86IGF1ZGlvIGEgMTYga0h6IG1vbm8gKGZmbXBlZykg4oaSIHdhdjJ2ZWMyLWJhc2Ug4oaSIG1lZGlhIGRlIGxhcyBjYXBhcyBvY3VsdGFzIDYtOQoobGFzIHF1ZSBtZWpvciBjb2RpZmljYW4gZm9uZW1hcyB5IHByb3NvZGlhKSDihpIgNTAgSHogYSAyNSBIeiBwcm9tZWRpYW5kbyBwYXJlcyDihpIgZmxvYXQxNi4KR3VhcmRhIDxzYWxpZGE+Lzxub21icmU+Lm5weiBjb24gYHZvemAgW1QsIDc2OF0geSBgdGAgW1RdLiBSZWFudWRhYmxlOyB0cm96b3MgZGUgMjAgcyBwYXJhCm5vIGFnb3RhciBsYSBtZW1vcmlhIGNvbiB2w61kZW9zIGxhcmdvcyAoZGlzY3Vyc29zIGRlIE9iYW1hKS4KClVzbzogIHB5dGhvbiBleHRyYWVyLnB5IC0tbWVkaWRhcyBESVJfTlBaIC0tdmlkZW9zIERJUl9NUDQgLS1zYWxpZGEgRElSIFstLWRpc3Bvc2l0aXZvIGN1ZGFdCkVtcGFyZWphIDxub21icmU+Lm5weiBjb24gPG5vbWJyZT4ubXA0IChidXNjYSByZWN1cnNpdmFtZW50ZSBlbiAtLXZpZGVvcykuCiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGdsb2IKaW1wb3J0IG9zCmltcG9ydCBzdWJwcm9jZXNzCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgoKZGVmIGF1ZGlvKG1wNCwgZmZtcGVnKToKICAgIHIgPSBzdWJwcm9jZXNzLnJ1bihbZmZtcGVnLCAnLWxvZ2xldmVsJywgJ2Vycm9yJywgJy1pJywgbXA0LCAnLWFjJywgJzEnLCAnLWFyJywgJzE2MDAwJywgJy1mJywgJ3MxNmxlJywgJy0nXSwKICAgICAgICAgICAgICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLCBjaGVjaz1UcnVlKQogICAgcmV0dXJuIG5wLmZyb21idWZmZXIoci5zdGRvdXQsIG5wLmludDE2KS5hc3R5cGUobnAuZmxvYXQzMikgLyAzMjc2OAoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1tZWRpZGFzJywgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS12aWRlb3MnLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLXNhbGlkYScsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tZGlzcG9zaXRpdm8nLCBkZWZhdWx0PSdjdWRhJyBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgJ2NwdScpCiAgICBhID0gYXAucGFyc2VfYXJncygpCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgV2F2MlZlYzJNb2RlbAogICAgdHJ5OgogICAgICAgIGltcG9ydCBpbWFnZWlvX2ZmbXBlZwogICAgICAgIGZmbXBlZyA9IGltYWdlaW9fZmZtcGVnLmdldF9mZm1wZWdfZXhlKCkKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBmZm1wZWcgPSAnZmZtcGVnJwogICAgb3MubWFrZWRpcnMoYS5zYWxpZGEsIGV4aXN0X29rPVRydWUpCiAgICByZWQgPSBXYXYyVmVjMk1vZGVsLmZyb21fcHJldHJhaW5lZCgnZmFjZWJvb2svd2F2MnZlYzItYmFzZScpLnRvKGEuZGlzcG9zaXRpdm8pLmV2YWwoKQogICAgdmlkZW9zID0ge29zLnBhdGguc3BsaXRleHQob3MucGF0aC5iYXNlbmFtZSh2KSlbMF06IHYKICAgICAgICAgICAgICBmb3IgdiBpbiBnbG9iLmdsb2Iob3MucGF0aC5qb2luKGEudmlkZW9zLCAnKionLCAnKi5tcDQnKSwgcmVjdXJzaXZlPVRydWUpfQogICAgaGVjaG9zID0gMAogICAgZm9yIG5weiBpbiBzb3J0ZWQoZ2xvYi5nbG9iKG9zLnBhdGguam9pbihhLm1lZGlkYXMsICcqKicsICcqLm5weicpLCByZWN1cnNpdmU9VHJ1ZSkpOgogICAgICAgIG5vbWJyZSA9IG9zLnBhdGguc3BsaXRleHQob3MucGF0aC5iYXNlbmFtZShucHopKVswXQogICAgICAgIGRlc3QgPSBvcy5wYXRoLmpvaW4oYS5zYWxpZGEsIG5vbWJyZSArICcubnB6JykKICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhkZXN0KSBvciBub21icmUgbm90IGluIHZpZGVvczoKICAgICAgICAgICAgY29udGludWUKICAgICAgICB0ID0gbnAubG9hZChucHopWyd0J10KICAgICAgICB0cnk6CiAgICAgICAgICAgIHggPSBhdWRpbyh2aWRlb3Nbbm9tYnJlXSwgZmZtcGVnKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcHJpbnQoJ3NpbiBhdWRpbycsIG5vbWJyZSwgZSkKICAgICAgICAgICAgY29udGludWUKICAgICAgICB0cm96b3MgPSBbXQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4oeCksIDE2MDAwICogMjApOgogICAgICAgICAgICAgICAgc2VnID0gdG9yY2gudGVuc29yKHhbaTppICsgMTYwMDAgKiAyMF0pW05vbmVdLnRvKGEuZGlzcG9zaXRpdm8pCiAgICAgICAgICAgICAgICBpZiBzZWcuc2hhcGVbMV0gPCA0MDA6CiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGggPSByZWQoc2VnLCBvdXRwdXRfaGlkZGVuX3N0YXRlcz1UcnVlKS5oaWRkZW5fc3RhdGVzCiAgICAgICAgICAgICAgICB0cm96b3MuYXBwZW5kKHRvcmNoLnN0YWNrKGhbNjoxMF0pLm1lYW4oMClbMF0uY3B1KCkubnVtcHkoKSkgICAgIyBbVDUwLCA3NjhdCiAgICAgICAgZjUwID0gbnAuY29uY2F0ZW5hdGUodHJvem9zKQogICAgICAgIGYyNSA9IGY1MFs6IGxlbihmNTApIC8vIDIgKiAyXS5yZXNoYXBlKC0xLCAyLCA3NjgpLm1lYW4oMSkgICAgICAgICAgICAgICAgIyAyNSBIegogICAgICAgIGcgPSBucC5hcmFuZ2UobGVuKGYyNSkpIC8gMjUuMAogICAgICAgIGlkeCA9IG5wLmNsaXAobnAuc2VhcmNoc29ydGVkKGcsIHQpLCAwLCBsZW4oZjI1KSAtIDEpICAgICAgICAgICAgICAgICAgICAgIyBhIGxhIHJlamlsbGEgZGVsIC5ucHoKICAgICAgICBucC5zYXZlel9jb21wcmVzc2VkKGRlc3QsIHZvej1mMjVbaWR4XS5hc3R5cGUobnAuZmxvYXQxNiksIHQ9dCkKICAgICAgICBoZWNob3MgKz0gMQogICAgICAgIGlmIGhlY2hvcyAlIDIwID09IDA6CiAgICAgICAgICAgIHByaW50KGhlY2hvcywgbm9tYnJlLCBmbHVzaD1UcnVlKQogICAgcHJpbnQoJ2xpc3RvJywgaGVjaG9zKQoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICBtYWluKCkK", "tools/cara_parlante/voz2cara/articulacion.py": "IiIiRWwgbW9kZWxvIGFydGljdWxhdG9yaW8gZGUgQmxlbmRlciAoYm9jYTNkKSBjb21vIGVudHJhZGEgZGUgbGEgcmVkIGRlIGZ1c2nDs24uDQoNCkVzIGVsIG1pc21vIGPDoWxjdWxvIHF1ZSBtdWV2ZSBsYSBib2NhIGRpZMOhY3RpY2EgZGUgUHJvbnVuY2lhcjogZm9uZW1hcyBBUlBBYmV0IChnMnBfZW4pDQpyZXBhcnRpZG9zIGRlbnRybyBkZSBjYWRhIHBhbGFicmEgeSBsYSBmw7NybXVsYSBkZSBkb21pbmFuY2lhIGRlIENvaGVuLU1hc3Nhcm8gZW50cmVuYWRhIGNvbg0KR1JJRCAobW9kZWxvX2FydGljdWxhdG9yaW8uanNvbikuIERhLCBwb3IgZm90b2dyYW1hLCBsb3MgMTIgY29udHJvbGVzIGRlbCBtb2RlbG8gKGVuIHopIHkgZWwNCmZvbmVtYSBxdWUgc3VlbmEuIGdlbmVyYXIucHkgKEVvcykgeSBwcmVwYXJhcl9hcnQucHkgKGRhdG9zIGRlIGVudHJlbmFtaWVudG8pIGxvIGNvbXBhcnRlbiwNCnBhcmEgcXVlIGxhIHJlZCB2ZWEgZW4gZWwgZW50cmVuYW1pZW50byBleGFjdGFtZW50ZSBsbyBtaXNtbyBxdWUgdmVyw6EgYWwgaGFibGFyIEVvcy4NCiIiIg0KaW1wb3J0IGpzb24NCmltcG9ydCBvcw0KaW1wb3J0IHJlDQoNCmltcG9ydCBudW1weSBhcyBucA0KDQpNT0RFTE8gPSBvcy5wYXRoLmFic3BhdGgob3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShfX2ZpbGVfXyksICcuLicsICcuLicsICcuLicsICdmcm9udCcsICdzcmMnLCAnYXBwJywNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ2NvbXBhcnRpZG8nLCAnYm9jYTNkJywgJ21vZGVsb19hcnRpY3VsYXRvcmlvLmpzb24nKSkNCl9nMnAgPSBOb25lDQoNCg0KZGVmIGZvbmVtYXMocGFsYWJyYXMpOg0KICAgICIiIihmb25lbWEgQVJQQWJldCBzaW4gYWNlbnRvLCB0MCwgdDEpIHJlcGFydGlkb3MgZGVudHJvIGRlIGNhZGEgcGFsYWJyYTsgbGFzIHZvY2FsZXMgcGVzYW4gZWwgZG9ibGUuIiIiDQogICAgZ2xvYmFsIF9nMnANCiAgICBpZiBfZzJwIGlzIE5vbmU6DQogICAgICAgIGZyb20gZzJwX2VuIGltcG9ydCBHMnANCiAgICAgICAgX2cycCA9IEcycCgpDQogICAgb3V0ID0gW10NCiAgICBmb3IgdGV4dG8sIHQwLCB0MSBpbiBwYWxhYnJhczoNCiAgICAgICAgZnMgPSBbcmUuc3ViKHInXGQnLCAnJywgcCkubG93ZXIoKSBmb3IgcCBpbiBfZzJwKHRleHRvKSBpZiByZS5tYXRjaChyJ1tBLVpdJywgcCldDQogICAgICAgIGlmIG5vdCBmczoNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHBlc28gPSBucC5hcnJheShbMi4wIGlmIGZbMF0gaW4gJ2FlaW91JyBlbHNlIDEuMCBmb3IgZiBpbiBmc10pDQogICAgICAgIGJvcmRlcyA9IHQwICsgKHQxIC0gdDApICogbnAuY29uY2F0ZW5hdGUoW1swXSwgbnAuY3Vtc3VtKHBlc28pIC8gcGVzby5zdW0oKV0pDQogICAgICAgIG91dCArPSBbKGYsIGJvcmRlc1tpXSwgYm9yZGVzW2kgKyAxXSkgZm9yIGksIGYgaW4gZW51bWVyYXRlKGZzKV0NCiAgICByZXR1cm4gb3V0DQoNCg0KZGVmIGJvY2EoZm9ucywgdCwgbW9kZWxvPU1PREVMTyk6DQogICAgIiIiQ29udHJvbGVzIGRlbCBtb2RlbG8gKHopIGVuIGxvcyBpbnN0YW50ZXMgdCAocykgY29uIGxhIGbDs3JtdWxhIGRlIGRvbWluYW5jaWEuIiIiDQogICAgTSA9IGpzb24ubG9hZChvcGVuKG1vZGVsbykpDQogICAgaWR4ID0ge2Y6IGkgZm9yIGksIGYgaW4gZW51bWVyYXRlKE1bJ2ZvbmVtYXMnXSl9DQogICAgVCwgbGEgPSBucC5hcnJheShNWydUJ10pLCBucC5hcnJheShNWydsb2dfYWxmYSddKQ0KICAgIHRhLCB0ciA9IG5wLmFycmF5KE1bJ2xvZ190aGV0YV9hbnQnXSksIG5wLmFycmF5KE1bJ2xvZ190aGV0YV9hcnInXSkNCiAgICBzZWdzID0gWyhpZHhbJ3NpbCddLCBmb25zWzBdWzFdIC0gMC4yNSldICsgWyhpZHguZ2V0KGYsIGlkeFsnYWgnXSksIChhICsgYikgLyAyKSBmb3IgZiwgYSwgYiBpbiBmb25zXSBcDQogICAgICAgICsgWyhpZHhbJ3NpbCddLCBmb25zWy0xXVsyXSArIDAuMjUpXQ0KICAgIGlkcyA9IG5wLmFycmF5KFtzWzBdIGZvciBzIGluIHNlZ3NdKTsgY3MgPSBucC5hcnJheShbc1sxXSBmb3IgcyBpbiBzZWdzXSkNCiAgICB5ID0gbnAuemVyb3MoKGxlbih0KSwgVC5zaGFwZVsxXSkpDQogICAgb3JkZW4gPSBucC5hcmdzb3J0KGNzKTsgaWRzLCBjcyA9IGlkc1tvcmRlbl0sIGNzW29yZGVuXQ0KICAgIGZvciBpMCBpbiByYW5nZSgwLCBsZW4odCksIDUwMCk6ICAgICAgICAgICAgICAgICAgICMgcG9yIGJsb3F1ZXM7IHNvbG8gcGVzYW4gbG9zIHNlZ21lbnRvcyBjZXJjYW5vcw0KICAgICAgICB0YiA9IHRbaTA6aTAgKyA1MDBdDQogICAgICAgIGEsIGIgPSBucC5zZWFyY2hzb3J0ZWQoY3MsIHRiWzBdIC0gMy4wKSwgbnAuc2VhcmNoc29ydGVkKGNzLCB0YlstMV0gKyAzLjApDQogICAgICAgIGEsIGIgPSBtYXgoMCwgYSAtIDEpLCBtaW4obGVuKGNzKSwgYiArIDEpDQogICAgICAgIGliLCBjYiA9IGlkc1thOmJdLCBjc1thOmJdDQogICAgICAgIGR0ID0gY2JbTm9uZSwgOl0gLSB0Yls6LCBOb25lXQ0KICAgICAgICBmb3IgayBpbiByYW5nZShULnNoYXBlWzFdKToNCiAgICAgICAgICAgIHRoZXRhID0gbnAud2hlcmUoZHQgPiAwLCBucC5leHAodGFbaWIsIGtdKSwgbnAuZXhwKHRyW2liLCBrXSkpDQogICAgICAgICAgICBsb2dkID0gbGFbaWIsIGtdIC0gdGhldGEgKiBucC5hYnMoZHQpDQogICAgICAgICAgICBsb2dkIC09IGxvZ2QubWF4KDEsIGtlZXBkaW1zPVRydWUpDQogICAgICAgICAgICBkID0gbnAuZXhwKGxvZ2QpOyBkIC89IGQuc3VtKDEsIGtlZXBkaW1zPVRydWUpDQogICAgICAgICAgICB5W2kwOmkwICsgNTAwLCBrXSA9IGQgQCBUW2liLCBrXQ0KICAgIHJldHVybiB7YzogeVs6LCBpXSBmb3IgaSwgYyBpbiBlbnVtZXJhdGUoTVsnY29udHJvbGVzJ10pfQ0KDQoNCmRlZiBlbnRyYWRhKHBhbGFicmFzLCB0LCBtb2RlbG89TU9ERUxPKToNCiAgICAiIiJFbnRyYWRhIGFydGljdWxhdG9yaWEgZGUgbGEgcmVkIGVuIGxvcyBpbnN0YW50ZXMgdDogW2xlbih0KSwgMTIgY29udHJvbGVzICsgMSBoYWJsYV0NCiAgICB5IGVsIMOtbmRpY2UgZGVsIGZvbmVtYSBxdWUgc3VlbmEgKC0xIGVuIHNpbGVuY2lvKS4gwqtoYWJsYcK7IGVzIDEgZGVudHJvIGRlIGxhcyBwYWxhYnJhcy4iIiINCiAgICBNID0ganNvbi5sb2FkKG9wZW4obW9kZWxvKSkNCiAgICBpZHggPSB7ZjogaSBmb3IgaSwgZiBpbiBlbnVtZXJhdGUoTVsnZm9uZW1hcyddKX0NCiAgICBmb25zID0gZm9uZW1hcyhwYWxhYnJhcykNCiAgICBhcnQgPSBucC56ZXJvcygobGVuKHQpLCBsZW4oTVsnY29udHJvbGVzJ10pICsgMSksIG5wLmZsb2F0MzIpDQogICAgZm9uID0gbnAuZnVsbChsZW4odCksIC0xLCBucC5pbnQxNikNCiAgICBpZiBub3QgZm9uczoNCiAgICAgICAgcmV0dXJuIGFydCwgZm9uDQogICAgYiA9IGJvY2EoZm9ucywgdCwgbW9kZWxvKQ0KICAgIGFydFs6LCA6LTFdID0gbnAuc3RhY2soW2JbY10gZm9yIGMgaW4gTVsnY29udHJvbGVzJ11dLCAxKQ0KICAgIGZvciBmLCBhLCB6IGluIGZvbnM6DQogICAgICAgIGksIGogPSBucC5zZWFyY2hzb3J0ZWQodCwgYSksIG5wLnNlYXJjaHNvcnRlZCh0LCB6KQ0KICAgICAgICBmb25baTpqXSA9IGlkeC5nZXQoZiwgaWR4WydhaCddKQ0KICAgIGZvciBfLCBhLCB6IGluIHBhbGFicmFzOg0KICAgICAgICBhcnRbbnAuc2VhcmNoc29ydGVkKHQsIGEpOm5wLnNlYXJjaHNvcnRlZCh0LCB6KSwgLTFdID0gMQ0KICAgIHJldHVybiBhcnQsIGZvbg0K", "tools/cara_parlante/voz2cara/preparar_art.py": "IiIiRW50cmFkYSBhcnRpY3VsYXRvcmlhIGRlIGxvcyB2w61kZW9zIGRlIGVudHJlbmFtaWVudG86IHF1w6kgZGljZW4geSBjdcOhbmRvIChXaGlzcGVyIGNvbg0KbWFyY2FzIGRlIHBhbGFicmEpLCB5IGxvIHF1ZSBoYXLDrWEgY29uIGVzbyBlbCBtb2RlbG8gYXJ0aWN1bGF0b3JpbyBkZSBCbGVuZGVyLCBmb3RvZ3JhbWEgYQ0KZm90b2dyYW1hIGVuIGxhIHJlamlsbGEgZGUgc3UgLm5weiBkZSBtZWRpZGFzLg0KDQpQb3IgdsOtZGVvOiBhdWRpbyAxNiBrSHog4oaSIGZhc3Rlci13aGlzcGVyIChwYWxhYnJhcyBjb24gdGllbXBvKSDihpIgYXJ0aWN1bGFjaW9uLmVudHJhZGEoKQ0K4oaSIDxzYWxpZGE+Lzxub21icmU+Lm5weiBjb24gYGFydGAgW1QsIDEzXSBmbG9hdDE2LCBgZm9uYCBbVF0gaW50MTYgeSBgdGA7IHkgbGFzIHBhbGFicmFzIGVuDQo8c2FsaWRhPi88bm9tYnJlPi5qc29uLiBSZWFudWRhYmxlLg0KDQpVc286IHB5dGhvbiBwcmVwYXJhcl9hcnQucHkgLS1tZWRpZGFzIERJUl9OUFogLS12aWRlb3MgRElSIChtcDQsIG00YSwgd2F2IG8gZmxhYykgLS1zYWxpZGEgRElSDQogICAgICAgICBbLS13aGlzcGVyIGxhcmdlLXYzXSBbLS1kaXNwb3NpdGl2byBjdWRhXSBbLS1tb2RlbG9fYXJ0IHJ1dGEvbW9kZWxvX2FydGljdWxhdG9yaW8uanNvbl0NCiIiIg0KaW1wb3J0IGFyZ3BhcnNlDQppbXBvcnQgZ2xvYg0KaW1wb3J0IGpzb24NCmltcG9ydCBvcw0KaW1wb3J0IHN5cw0KDQppbXBvcnQgbnVtcHkgYXMgbnANCg0Kc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkNCmZyb20gYXJ0aWN1bGFjaW9uIGltcG9ydCBNT0RFTE8sIGVudHJhZGEgICAjIG5vcWE6IEU0MDINCmZyb20gZXh0cmFlciBpbXBvcnQgYXVkaW8gICAgICAgICAgICAgICAgICAjIG5vcWE6IEU0MDINCg0KDQpkZWYgbWFpbigpOg0KICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgnLS1tZWRpZGFzJywgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tdmlkZW9zJywgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tc2FsaWRhJywgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0td2hpc3BlcicsIGRlZmF1bHQ9J2xhcmdlLXYzJykNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tZGlzcG9zaXRpdm8nLCBkZWZhdWx0PSdjdWRhJykNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tbW9kZWxvX2FydCcsIGRlZmF1bHQ9TU9ERUxPKQ0KICAgIGEgPSBhcC5wYXJzZV9hcmdzKCkNCiAgICBmcm9tIGZhc3Rlcl93aGlzcGVyIGltcG9ydCBCYXRjaGVkSW5mZXJlbmNlUGlwZWxpbmUsIFdoaXNwZXJNb2RlbA0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IGltYWdlaW9fZmZtcGVnDQogICAgICAgIGZmbXBlZyA9IGltYWdlaW9fZmZtcGVnLmdldF9mZm1wZWdfZXhlKCkNCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6DQogICAgICAgIGZmbXBlZyA9ICdmZm1wZWcnDQogICAgb3MubWFrZWRpcnMoYS5zYWxpZGEsIGV4aXN0X29rPVRydWUpDQogICAgdGlwbyA9ICdmbG9hdDE2JyBpZiBhLmRpc3Bvc2l0aXZvID09ICdjdWRhJyBlbHNlICdpbnQ4Jw0KICAgIGFzciA9IEJhdGNoZWRJbmZlcmVuY2VQaXBlbGluZShXaGlzcGVyTW9kZWwoYS53aGlzcGVyLCBkZXZpY2U9YS5kaXNwb3NpdGl2bywgY29tcHV0ZV90eXBlPXRpcG8pKQ0KICAgIHZpZGVvcyA9IHtvcy5wYXRoLnNwbGl0ZXh0KG9zLnBhdGguYmFzZW5hbWUodikpWzBdOiB2IGZvciBleHQgaW4gKCdtcDQnLCAnbTRhJywgJ3dhdicsICdmbGFjJykNCiAgICAgICAgICAgICAgZm9yIHYgaW4gZ2xvYi5nbG9iKG9zLnBhdGguam9pbihhLnZpZGVvcywgJyoqJywgJyouJyArIGV4dCksIHJlY3Vyc2l2ZT1UcnVlKX0NCiAgICBoZWNob3MgPSAwDQogICAgZm9yIG5weiBpbiBzb3J0ZWQoZ2xvYi5nbG9iKG9zLnBhdGguam9pbihhLm1lZGlkYXMsICcqKicsICcqLm5weicpLCByZWN1cnNpdmU9VHJ1ZSkpOg0KICAgICAgICBub21icmUgPSBvcy5wYXRoLnNwbGl0ZXh0KG9zLnBhdGguYmFzZW5hbWUobnB6KSlbMF0NCiAgICAgICAgZGVzdCA9IG9zLnBhdGguam9pbihhLnNhbGlkYSwgbm9tYnJlICsgJy5ucHonKQ0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhkZXN0KSBvciBub21icmUgbm90IGluIHZpZGVvczoNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHQgPSBucC5sb2FkKG5weilbJ3QnXQ0KICAgICAgICB0cnk6DQogICAgICAgICAgICB4ID0gYXVkaW8odmlkZW9zW25vbWJyZV0sIGZmbXBlZykNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOg0KICAgICAgICAgICAgcHJpbnQoJ3NpbiBhdWRpbycsIG5vbWJyZSwgZSkNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHNlZ3MsIF8gPSBhc3IudHJhbnNjcmliZSh4LCBsYW5ndWFnZT0nZW4nLCB3b3JkX3RpbWVzdGFtcHM9VHJ1ZSwgYmF0Y2hfc2l6ZT0xNikNCiAgICAgICAgcGFsYWJyYXMgPSBbKHcud29yZC5zdHJpcCgpLCBmbG9hdCh3LnN0YXJ0KSwgZmxvYXQody5lbmQpKSBmb3IgcyBpbiBzZWdzIGZvciB3IGluIChzLndvcmRzIG9yIFtdKQ0KICAgICAgICAgICAgICAgICAgICBpZiB3LndvcmQuc3RyaXAoKV0NCiAgICAgICAgYXJ0LCBmb24gPSBlbnRyYWRhKHBhbGFicmFzLCB0LCBhLm1vZGVsb19hcnQpDQogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQoZGVzdCwgYXJ0PWFydC5hc3R5cGUobnAuZmxvYXQxNiksIGZvbj1mb24sIHQ9dCkNCiAgICAgICAganNvbi5kdW1wKHBhbGFicmFzLCBvcGVuKG9zLnBhdGguam9pbihhLnNhbGlkYSwgbm9tYnJlICsgJy5qc29uJyksICd3JykpDQogICAgICAgIGhlY2hvcyArPSAxDQogICAgICAgIGlmIGhlY2hvcyAlIDIwID09IDA6DQogICAgICAgICAgICBwcmludChoZWNob3MsIG5vbWJyZSwgbGVuKHBhbGFicmFzKSwgJ3BhbGFicmFzJywgZmx1c2g9VHJ1ZSkNCiAgICBwcmludCgnbGlzdG8nLCBoZWNob3MpDQoNCg0KaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoNCiAgICBtYWluKCkNCg==", "modelo_articulatorio.json": "eyJ2ZXJzaW9uIjoxLCJtb2RlbG8iOiJkb21pbmFuY2lhIENvaGVuLU1hc3Nhcm8sIGVudHJlbmFkYSBjb24gR1JJRCAoQ0MgQlkgNC4wKSIsImZvcm11bGEiOiJ5KHQpID0gc3VtX3MgRF9zKHQpIFRbcF9zXSAvIHN1bV9zIERfcyh0KTsgRF9zID0gZXhwKGxvZ19hbGZhIC0gdGhldGEgKiB8Y19zIC0gdHwpLCB0aGV0YSA9IGV4cChsb2dfdGhldGFfYW50KSBzaSB0IDwgY19zLCBzaSBubyBleHAobG9nX3RoZXRhX2Fycik7IHNhbGlkYSBlbiB6IGRlbCBoYWJsYW50ZSIsImZvbmVtYXMiOlsiYWEiLCJhZSIsImFoIiwiYW8iLCJhdyIsImF5IiwiYiIsImNoIiwiZCIsImRoIiwiZWgiLCJleSIsImYiLCJnIiwiaGgiLCJpaCIsIml5IiwiamgiLCJrIiwibCIsIm0iLCJuIiwib3ciLCJwIiwiciIsInMiLCJzaWwiLCJ0IiwidGgiLCJ1dyIsInYiLCJ3IiwieSIsInoiXSwiY29udHJvbGVzIjpbImdlbzphcGVydHVyYV9pbnQiLCJnZW86YW5jaG8iLCJnZW86bWFuZGlidWxhIiwiZ2VvOnByb3RydXNpb24iLCJnZW86bWVqaWxsYV95IiwiYnM6amF3T3BlbiIsImJzOm1vdXRoUHVja2VyIiwiYnM6bW91dGhGdW5uZWwiLCJiczptb3V0aFJvbGxMb3dlciIsImJzOm1vdXRoUHJlc3MiLCJiczptb3V0aENsb3NlIiwiYnM6Y2hlZWtTcXVpbnQiXSwiVCI6W1sxLjQyMjIsLTAuNzY3OSwxLjkzNDMsMS41ODkzLC0wLjQ4NDcsMS41NTM4LC0wLjM1NywxLjAyMTMsLTAuNzM1NywtMC40Mzc1LC0wLjIxNTcsMC4xNDldLFsxLjkzNCwxLjA3NjksMS4zNjQ1LDAuNTMxNiwwLjIwOTEsMS43MDIzLC0xLjAzMDEsMS40NjA2LC0wLjk3OTIsMC44MDYsMC43OTM0LDAuMTY0NF0sWzEuNjI4LC0wLjQzODgsMC45Mjk3LDIuMTY5NCwtMS4xNTUyLDAuOTU4MSwxLjM3MDgsMy40NzQsLTAuNTEwOSwtMS4zNTM3LDAuOTM5OSwtMC4wMTU4XSxbLTAuNzAxOCwtMi42Mjc3LDAuNjg5NCwxLjMzMTYsLTAuMDM2OCwwLjczMjIsMi41Mzk5LDEuNjIxNiwwLjY3NywtMS44NTcxLDEuNDcxNywtMC4xNzZdLFsxLjU3OTUsLTAuNTEzOSwxLjYwOTgsMC4yMTk5LDAuNjc4NCwxLjcwMTEsMC41MTM3LDAuODg4OSwtMC42Nzg3LC0wLjU5MDIsMC42ODA0LC0wLjAyNl0sWzEuMzcxNSwwLjc5NDEsMS40NzA4LDAuOTYxNSwtMC4xOTcsMS40NjM4LC0wLjU2NjEsMy40NzU3LC0wLjYxNDksMC42OTQ0LC0wLjExMTYsMC4wMDAzXSxbLTEuMzQwMiwtMC43NzYxLC0wLjUzNTUsLTAuNDYsMS4wMywtMC41NDExLDEuMDg3MywtMS4yNDE0LDIuMjUzMywtMC42NTc0LDIuMDY2MywtMC4wOTUyXSxbMC45ODQ0LC0wLjEyMTksLTAuMTg3MywwLjQ3NjgsLTEuNDE2LDAuMTkxNiwtMC40MzM2LC0wLjI2MzEsLTAuNTcxNSwtMC45NDM4LC0wLjczMzUsMC4wNjgxXSxbMC42MjA5LDEuMjczNiwtMC40NTIxLC0wLjg2MDEsLTAuODE5OSwtMC43MzQ5LC0wLjQwMzQsLTAuMzE1NywtMC4zNzQ2LDAuODE0OCwtMC4zMTI3LDAuMjgzNl0sWy0xLjIzNTQsLTAuMTk4NywtMC41NzEzLDAuMTU5NywtMC41MDU5LC0xLjE5LC0wLjQzODcsLTAuMTIsLTAuMTUyOCwtMC43MDkxLC0wLjIyMTUsLTAuMDQ3MV0sWzEuODEwMSwxLjQ5MjUsMS45MzM5LC0wLjg4NTgsLTAuMjAwMywyLjE3NTcsLTAuOTA4MywxLjc3NDksLTAuNzkyMSwxLjU1NDUsMC4xMTE1LDAuMTIyOF0sWzEuODgyNywxLjMwODksMS42MTQ0LC0wLjg2NTEsLTAuMzQ4MiwyLjAzNTMsLTAuNjcwOCwwLjQyMzcsLTAuODM0MiwxLjEwODksMC4wMTU0LDAuMDcxOV0sWy0wLjgxMzEsLTAuMTAzMSwtMC41NzAyLC0xLjI4NjQsLTAuODE2NywtMS4wMDU4LC0wLjIzNDMsLTAuNzQ4MiwxLjIwODMsMC4xMjc4LC0wLjIwMjQsMC4wNDg4XSxbMC45OCwwLjY3MDIsMC4yNTk4LDAuMTczMywtMC4yNDcyLDAuNjM4NSwtMC4zOTksMC4yMzUsLTAuOTgxLC0wLjUzOTUsLTAuMzM1NiwtMC4wMDk5XSxbLTEuMzgxMywtMC45NzYzLC0wLjMyMDcsMC4zMTMsMC41ODQ5LC0wLjMxOTEsMC45NDAyLDAuMjk5MSwtMC4wNDcxLDAuMDI3NiwwLjM0OTksMC4wODcyXSxbMS40MTM5LDEuODgzMiwwLjg2NTMsMC41NzY4LC0xLjEwMzQsMS4wOSwtMC45MjM4LDEuNDU2MywtMC40MjQ1LDAuNzcwMiwtMC4yMzkyLDAuMDkyMV0sWzEuMDkyNywxLjMyNTUsMC42Mzg3LC0wLjAzMjksLTAuNjcwOSwwLjg3OSwtMC41NDYxLDAuNjEsLTAuNjE0NywwLjc0MjEsLTAuMzUzNywwLjE1NjJdLFsxLjAzMzcsLTAuNTY3MSwtMC4zMDEyLDAuNzQ0MiwtMC42NTY5LDAuMTEwOSwtMC4xODgzLDAuNDQxMiwtMC42MTIsLTAuNzc0LC0wLjc2MDUsLTAuMDc2XSxbMS4yMzgxLDEuNjE0LDAuNTc5NSwtMS4zMzQyLC0xLjA2NCwxLjEzODgsLTAuODkxNywtMC42MTk5LC0wLjY5NTgsMS4xNDgyLC0wLjM0MTIsMC4wNTU2XSxbMC41OTQ5LC0wLjMxNjUsMC4xMTU2LDAuNTQ4MywwLjM4MTQsMC4yNTA3LC0wLjQzNTMsMC4wODAxLDAuMDAyNiwtMC4yNDU4LC0wLjA2NTIsMC4wMTVdLFstMS40NDc3LC0xLjIyMzksLTAuNTQ0OSwtMS40NTk5LDEuMDY3OCwtMC45NjAyLDAuOTI1MywtMS4xNDk3LDIuMTkyNSwtMC41MDYsMi44MTEyLC0wLjE5NDldLFstMC4yNzUyLDAuNDAyMywtMC44NTIxLC0wLjQ3NjcsLTAuMzUsLTAuODE5LC0wLjYzNjEsLTAuODU2NSwtMC4wODMzLC0wLjIzNTUsLTAuNTM1MiwwLjAxODRdLFswLjE4MzcsLTEuODgwNCwwLjU1NjIsMS4zOTczLDAuNDgyNywwLjI0MDEsMi4yNDIxLDEuNjA0LC0wLjE1ODQsLTAuODY2NiwxLjExNTMsLTAuMDU4MV0sWy0xLjMyMTcsLTAuNzY3NiwtMC43NTUsMC4xMDM2LDEuNDg0MywtMC43NjM0LDEuMTY2NywtMS4wNTkyLDEuMTYyNiwtMC43NzIsMi4yMjQ4LC0wLjE1M10sWy0wLjg0NCwtMC44MDEsLTAuNDY0NCwwLjU5NjgsMC4zMDQ4LC0xLjE2ODQsMi4wNTk1LC0wLjg2NTQsMC43ODgyLC0wLjg1NzYsLTAuMDMzNywtMC4wNDY2XSxbMC4zNjc3LDAuNzQ2MSwtMC40NDk0LC0wLjgzNDEsLTAuNzYzOSwtMC4zNzcxLC0wLjQxNjgsLTAuMzQyMywtMC4zMDAyLC0wLjMxMTYsLTAuNTY0LDAuMTU3MV0sWy0wLjkyMDUsLTAuMzAxNCwtMC45MTQ5LDAuMzM5NywwLjMyMDUsLTAuODk1NCwwLjA5MDUsLTAuNDY2MiwwLjQyNDcsMC42Nzc4LC0wLjI0MDYsLTAuMDA1XSxbMC4xNzYsMS42MjgsLTAuNTI1NSwtMS43MDEsLTEuMDQ1LC0wLjM0ODMsLTAuNzYzNywtMC43MzUsLTAuNDMyMywwLjcwMzMsLTAuNTk5MSwwLjIzNTZdLFstMC40ODA0LDAuMTM4MSwtMC4xMjkyLC0wLjI3MDEsLTAuNTY4MywtMC40MDA2LC0wLjI3NDksLTAuMjQyOCwtMC4zNDk4LC0wLjU1MjUsMC4wMTI3LC0wLjEwOTddLFstMC41MTk2LC0xLjMwODEsLTAuMzEwOSwwLjg2MDUsMC41OTAyLC0wLjQxMjcsMS40NTg5LDAuOTY5NSwtMC4wMTMsLTAuODI4OSwwLjY5NzIsLTAuMDYwMl0sWy0xLjI3NDUsLTAuMDM4OSwtMC42NiwtMC45MzAyLC0wLjgxODIsLTEuNTY4NywtMC4yMzM1LC0wLjc4ODEsMS4wNzE3LC0wLjEwMjIsLTAuMzc2MywtMC4wMTY3XSxbLTEuNTU0NCwtNC4wNjc4LC0wLjI2ODEsMi4xMzgzLDEuNDc3MSwtMC40NDU4LDQuNTA3NCw0LjU5NzEsMS45OTE0LC0xLjI2MTEsNC4zMzAyLC0wLjExN10sWzAuNDczOSwtMC44MDk0LDAuNTUzNCwwLjk1NTksMC4xMTI0LDAuMjcxMiwtMC4yOTAyLDEuMTc0MywtMC44NzQ2LC0wLjg5MDMsLTAuMTgyMSwtMC4wMzQ0XSxbMC4zMzEyLDAuNjM1MiwtMC40NjksLTAuNzI5MywtMC41ODc3LC0wLjExMjksLTAuNDIzOSwtMC41ODcyLC0wLjI4MTgsMC4yNDI0LC0wLjQ4NTgsMC4yMzIxXV0sImxvZ19hbGZhIjpbWzIuNzIxLC0wLjE2ODMsMS4zNzY4LDAuOTIwNSwtMC4wNzUyLDIuNTQ4NywwLjMzNjYsMS40MDIxLDEuMDc3NCwwLjU0MzIsMC42OTU3LC0wLjgzMDJdLFstMC41NDI0LC0xLjYwNjcsLTAuMTU4LC0wLjg3ODUsLTEuNjcyLC0wLjMyODQsLTEuNjY4OCwtMC4zOTU3LC0wLjkwMTcsLTAuODMxOCwtMi40NTc3LC0xLjY0MzRdLFstMS4zNzk2LC0yLjUxNSwtMC4zMzYyLC0xLjM4NTYsLTEuNDAyMiwtMS4xNTQ4LC0yLjY4MDksLTAuOTI3OCwtMC40MjIxLC0xLjU1NCwtMS4wNjU0LC0wLjk3XSxbMS4xODU1LDIuMjUwNiwwLjQyMjksMS41MzIyLDAuNjY1NCwxLjE1NzUsMC44ODE4LDIuODk1MiwtMC4xMDQzLDAuODk5MywtMC4xMDEsMC4yNjM4XSxbMC4wMjQ4LDEuMzU2MSwwLjE2OTIsMC44NjIxLDAuNDE1OCwwLjA0OTMsMS45MDU4LDAuMjMyMywwLjQ1MTgsMi41MDkzLDAuNzE2MiwtMC41NDQ1XSxbMS42OTM4LC0wLjI0NSwxLjI4NSwtMC4wMDc4LC0wLjYxODksMS4zNDE1LDAuMjUzMiwtMS4xNDE2LDIuMDE1OCwtMC44OTksMC4wMTU0LC0wLjgwMV0sWzEuOTkzNSwwLjc5ODcsMS4wMTgzLDAuNTIzNCwxLjIzOCwyLjEzNzYsLTAuMzExNSwxLjAyNjcsMC4xMDA3LDEuMDE1OCwwLjU4ODMsMS4wNDM2XSxbMC4wNjY1LDAuNjQ0NiwwLjUyNzEsMC4yOTIsMC42NTI4LDAuMzc5OCwwLjQyMTYsMS40NDU2LDEuMjYxOCwwLjU0NzgsMC43NzEzLC0xLjAyNDRdLFstMC40MzUsMC4wNTEzLC0wLjIzMzQsMC42MjEzLDAuMjg3MSwtMC45NzExLDEuMzE4MSwwLjAwNTcsLTAuMzI4MSwtMC44MTU1LDAuMjE3OCwtMS41NTc0XSxbLTAuMjg0MiwwLjk2MjEsMC42Mjc0LDIuMjc1OSwwLjAwMjksMC44MzAyLDAuMjA1LDIuMzk1NiwtMC43NjU2LDAuNDk0NiwwLjY3NzEsMC4wMjk2XSxbMC44NzQyLC0wLjA5NjcsMC4yODg1LC0xLjEyMjUsLTAuMDg0OSwtMC4wNzA2LDAuNTU4OSwtMC42MTM3LDEuNzI5MywwLjMzODMsLTAuODkzMiwtMS4wNjc5XSxbMC44ODg0LDAuNTg2MywxLjA0NTMsMC42Njc1LDAuMzI0NCwwLjgxMDYsMC45OTM1LDEuMTY2NiwwLjMyMjUsMC40MDY5LDAuNTY4NSwtMC4wMzAxXSxbMi4wNTQ1LDEuMjgyNiwwLjkwMjYsMS4xNDA3LDAuODYzNCwyLjU5MywxLjM5MDEsMS44NzA4LDAuOTQ5NiwwLjkwNDYsMS4wMzQxLDAuNDc4NV0sWy0wLjU1ODcsLTEuMTQ4MiwwLjkyMzYsMC4xNTYzLC0wLjQ0MjcsLTAuMTA1OCwtMC42NDYsMC40MDIxLC0xLjE0MjcsLTAuNTY3MywwLjQ3NTQsLTAuNjMwMl0sWy0wLjYzNjIsMC4xMjk1LDAuNDY0LC0wLjE1MjUsMC42NzExLDAuMjUzNiwwLjU3MzcsMC4yODgzLC0wLjE0MjQsMC40MzI2LDAuODI2MSwtMC4yOTAyXSxbLTAuMzc4NiwtMC45NzYzLC0wLjAxMiwtMi4yNzM5LC0xLjIxODgsLTAuMjMxOCwtMC41MzE5LC0wLjkzMDcsMy4wMDgyLC0wLjQ1LC0wLjE1ODIsLTAuMzA1N10sWzEuMTU5NywwLjAzODEsMC42NTc0LC0wLjA1NywtMC4wMTE0LDAuNTQ1MiwwLjkzOTIsMC41NjUxLDEuNDA4MiwtMC41NzQzLDAuODE0NCwtMC42MzhdLFswLjk1OTYsMS41MywxLjQ1ODQsMi4wNjgzLDEuMDg2OCwwLjg2ODQsMC42NDU3LDEuMjIxNywwLjY4MDYsMS4yNzQxLDAuNzQyNCwtMC4xOTM3XSxbMC45NjM1LC0wLjYzOTQsMC42MTk4LC0wLjk1OCwtMC41MTgzLDAuNTg0MiwtMC4xMzY1LC0wLjg1NzQsMC40MzEyLC0wLjcwNDEsLTAuMjA4NywtMC44OTMxXSxbLTAuNjIyMywtMi4zMzQ0LC0wLjA4NjQsLTEuNjE3NCwtMS4xNDE5LC0wLjM2MjUsLTEuNjg4NCwtMC43ODg5LC0xLjIyODgsLTAuOTYxMywwLjAzNTgsLTAuMTEwOF0sWzIuMDk5NiwtMC41NzQ0LDAuMDc3MywwLjUwNTIsMC40ODE3LDEuNTA0NiwtMS4wMTYsMS4wMjAyLDAuMjcyMiwwLjk0MDMsLTAuNTM0OSwtMC4wOThdLFstMi4zMzQ5LC0xLjI0NTcsLTEuNjI2OCwtMC44NjQzLC0xLjAxMzUsLTEuODY3NCwtMS42Njc4LC0xLjMwNTcsLTIuNjgzLC0wLjg1NDIsLTEuNDY3MSwtMS4wNDE5XSxbMi41MDA0LDEuMzYxNSwwLjY2NzksMS40MzkxLDAuNzc0OCwxLjM4MzYsMC42MTc4LDAuNDI5OCwyLjUzNzcsMi4zMjcxLDAuMDA1NCwxLjY2MjVdLFsyLjA2NTEsMC40OTM5LDAuODU4NywxLjU5MDMsLTAuMjc1NCwxLjcxNjIsLTAuMjkyOCwxLjQzNDIsMC44MDExLDAuNTQzNSwwLjM5ODEsMC4zMzU0XSxbMC45OTk3LDEuMTY1NCwwLjg0NzksMC44MDk2LC0wLjI0NzksMS4zODQzLC0wLjQ2MjIsMC44MDY4LDAuNDc3MiwxLjE4ODEsLTIuMTU0NSwwLjkzMV0sWzAuMDI3MSwwLjExNTcsMS40NjAzLDAuMjU2OSwwLjY5MDcsMS4wMzY3LDAuNjQxNywxLjAzOTMsMS4wNDY5LC0xLjM1ODYsMS4yNjAxLC0wLjU5MTldLFstMy4yNDkxLC0zLjYwMzksLTIuNDg5MywtMy42MjYyLC0yLjI5MjcsLTMuMDEzMSwtMi43MjI2LC0yLjU0OSwtMy4xNjIxLC0zLjc0NzMsLTIuNjkzNSwwLjAwMDJdLFstMS4wMTc0LC0wLjU4NzgsMC4xMjIzLC0wLjEwMywtMC4yODc4LDAuMDQ0NywtMC40ODQ1LDAuNDM4NSwtMC4zMzUsLTAuNjY0LC0wLjY3MSwtMS4xMTIyXSxbMC4zNDc0LDEuMDYyLDEuNjg2OCwxLjI0MDIsLTAuMTI3MiwyLjI4MzQsMC43MDU0LDIuMjUzOSwtMC4yNzQyLC0wLjExOSwxLjI2MzIsLTAuMjkyNV0sWzIuNjQ3OCwwLjgyMTMsMC42MTExLDEuMjE4MiwwLjEzNTksMS44MDY3LDAuMTEwNSwwLjA2NTcsMC45OTk3LDIuMjE5OSwtMC4xMDc3LDEuNjgxNF0sWzAuOTA3MywxLjE3NjksMC41NDEzLDEuMTg2NCwwLjcxMSwxLjMwMjEsMC41NTM3LDIuMzY0NSwwLjc1NTksMS40NTI4LDEuOTQzNywwLjEzNzhdLFsxLjMxODEsMC43MDM3LC0wLjE0NTksMC45MjM2LDAuNzc3NCwwLjg2NDcsMC43Njc3LC0wLjM2MjEsLTAuMDQzLDEuODM0NSwwLjQ1MjgsMS40NDQ5XSxbMC40NzA1LDAuMjgzMiwwLjMxMiwxLjA2MTMsMC41NjYsMC41MTA4LC0wLjI5NzEsMC4wMDg3LC0wLjU0MTMsMS4wMTA5LDAuMTYxMSwwLjEzMV0sWy0wLjU5MDUsMi4xMTYsMC4xODYzLDEuODYwNCwxLjkxMjcsMC44MzUzLDEuOTUyLDAuMjgzOCwwLjA0OTgsMS44MjU4LDAuOTQzNywtMC42NDE1XV0sImxvZ190aGV0YV9hbnQiOltbMy42MjUzLDIuNTcyNCwzLjA2MzYsMi44MjA1LDIuNjE0NCwzLjQyMiwzLjUyNTksMi40OTM3LDEuNzM1NiwyLjMwNTUsMy42ODk3LDQuNDg5XSxbMi4xNjU5LDMuMzIxNywxLjgxMTgsMy41NDk4LDMuODU1NCwxLjkzNDksMy45Nzg0LDIuNDU2NCwxLjg5ODEsMi4xMTcsMi42MjgxLDMuMzY1M10sWzEuNTcyOCw0LjQxMTMsMi44MTM5LDMuNzIxMSwxLjI1NDQsMS43MjE4LDQuNDMyOSwzLjY1NywyLjEwMzUsMi42MTE0LDQuMTE1NiwzLjUyMTRdLFsyLjg1OTUsMy43MjE0LDMuOTE0Myw0LjcyNjUsMy45OTYyLDQuNTc0OSwzLjUwODYsNS4zNjg5LDIuMzYyNCwzLjI5MDksMy40Njg0LDIuNjI2OV0sWzEuNzM1OCw2LjYzNjMsMS41NjI4LDEuNzM3OSwzLjk0OTksMS42NjIsNy4xMzQzLDEuMjk3NSwxLjMzNzYsNy4xNzc2LDUuMTEzMyw0Ljc2MTRdLFszLjAwNzYsNC4wOCwyLjcwMSwxLjgxNzEsNC4wMTY2LDIuNzA0Niw0LjAwNDEsMS45ODE1LDMuMTUzNiw0Ljk0ODgsMy4wMzQsMS4zODkzXSxbMy4wNDEsMi40MjQzLDIuODUxMiwzLjA0MjQsMi41NzYyLDMuMDA5OSwyLjUwNSwyLjg4MTcsMy4xNjMzLDIuMDAxNSwyLjk1MzMsMi43MTM5XSxbMi45NzI0LDMuMTE5MiwzLjE2MDcsMS41NzA4LDMuMzI1NSwzLjAxNywyLjQ4MjMsMi45NzM3LDIuMDg4OCwzLjQ5OCwzLjE1ODgsMy4yNTkyXSxbMi44NjQ4LDIuOTc0OSwzLjQ1NzIsMy45NTA2LDIuOTM2NSwyLjk1NDcsMi44MzE0LDQuNDEwMywzLjYwNzcsMy44OTkyLDIuNzc5LDMuMTM5NF0sWzIuMzYwNiwyLjAyMywyLjUyOTQsMi40Mjc3LDEuNTg0MiwyLjI4OTcsMi40MTIxLDIuNzk5NSwxLjYwMjgsMS43NDI3LDEuOTk4OCwyLjE2MTJdLFszLjMyODUsMi45Njk2LDIuMDc0LDIuNTkwNSwyLjcyMTQsMi4wODE0LDIuOTI2NiwxLjgzNjcsMy40Njc0LDIuNjk3NiwxLjU0ODIsMi41NjA4XSxbMi40NjMyLDIuNzE3MiwyLjExOTksMy4wODM2LDIuNjkwNCwyLjIyMTksMi43NzAxLDIuMDA2NywyLjM0NDksMi40OTk0LDMuMTA4NiwxLjgzMThdLFszLjU4NDQsMy4xMDg3LDMuMzI3NywzLjE3NzcsMy4wMDUsMy41OTI3LDMuNDY5OCwyLjk0OTgsMy4xNDMsMi43NjAxLDIuODAzNiwyLjk1NDNdLFsxLjg3MzcsMy44NzA2LDIuMTk5Miw0LjA5OSwzLjYyMjYsMS43ODM1LDMuMTk4MiwxLjg0ODMsMi4wNDM5LDUuMjc0MiwyLjg3MDQsMS4yNzM3XSxbMi41Njk4LDIuMzUxLDEuNTExNiwxLjYzMiwwLjc1NDQsMS43NjI4LDIuOTI2NCwyLjIxOTQsMi42NzYyLDMuNDk5NCwyLjQ2NDksMi4zMTU1XSxbMS45NzI0LDIuMjIxNCwxLjc1ODQsMC4zMDE4LDIuNzAxMiwxLjg2MjIsMi4yOTE5LDEuNjQ2NywzLjk4NDUsMS45Mzg4LDIuNzA1MywyLjI1M10sWzMuMjY1NiwyLjU5NTksMi40MTI1LDMuMzcyOCwyLjExNDksMi40OTA3LDMuMDUzOCwyLjAzMDIsMi45OTgzLDIuMDc5OSwzLjM2MTUsMi41NTExXSxbMy45OTc3LDIuNjg2MSwyLjMzMTMsMi43ODMzLDMuNDQzOCwyLjU0NTIsMi42MTE0LDIuOTUxMiwzLjIyNTEsMi42NTY1LDIuNjMxNCwyLjM1OTddLFszLjQ4ODYsMi40Mzc4LDMuMjk3NywxLjcyMDIsMi41NzM2LDIuNzk4MywzLjA3MTQsMi41NDM0LDEuNzM2NiwyLjM2ODgsMy4yMDksMy4yMzY1XSxbMS43MzU1LDAuNzk1OSwxLjczNjYsMS4zODQ2LDAuODI0NiwxLjYyMjksMi45OTcxLDEuNjQ5NywyLjQ1OSwxLjE2OTQsMi4wMTQ4LDIuMTI0OF0sWzMuNTY1NCwyLjg4ODksMy42NTY1LDMuMTc3NSwyLjM0NzEsMy40ODg2LDMuMDA5MywzLjI1ODIsMy4yNzc5LDIuOTIxLDIuODcxNiwxLjk5OTJdLFszLjE5MTcsMy4wODUzLDIuODY0NSwyLjY4MTMsMy40MjU1LDIuOTE1NSwzLjcyODIsMS41ODksMi45MTA5LDIuMTExOCwyLjI3ODMsMS4zNDg0XSxbMi42OTI0LDIuOTI0NiwyLjI0MTgsMi44MzQ0LDMuOTM2NSwyLjAzMTYsMy4zMDI5LDMuMDE3MywyLjI0MjUsMy4xNDQ2LDMuNzkyNywzLjg3MjNdLFszLjEwODksMi40ODUyLDIuNzgyMiwzLjI0MjksMi40NDkxLDIuODk0NSwyLjg0OSwyLjg5MjgsMi44ODQ1LDIuNDI0NywyLjc3NjYsMi41NzY0XSxbMi44MDM1LDIuMzQyMiwyLjQzOTMsMS44MzkyLDAuODc5NSwyLjc1ODUsMi41ODM1LDIuNjA3MiwyLjcyOSwyLjIyMDMsMC4xNzQ5LDIuNjA5Ml0sWzEuOTMzNCwzLjU1ODIsMi42ODM0LDMuMzgwNywzLjM4ODMsMi42NDkyLDMuNDYwOCwyLjQ2NSwyLjY0NTEsMC41NDExLDIuODg5MSwyLjEwNzZdLFswLjAzOTgsMi4wNjI2LDAuMDkwNSw0LjEzMDUsMS44NTA3LDAuMTY1NiwyLjEzMzIsLTAuMDc0NiwwLjE4NDEsMS45MzkzLDMuMDU1MSwyLjUyMDldLFsyLjU2MjMsMi41OTUxLDMuMDk1NywyLjkzMzIsMy4wMjYzLDIuOTgxNSwyLjY0NTEsMi45NTI4LDEuODk1LDIuNzA2MywyLjM3MzMsMi40MDEyXSxbMy4yOTA0LDIuMDkzMSwyLjYxMTksMi4yMTIxLDIuMjUzOCwyLjk3MDUsMS44NTQ0LDIuNTI0OSwxLjU3MjksMy43MTY5LDEuODY5MiwyLjU5NzVdLFs0LjQ3NjcsMi4zMDE0LDIuMzIxMywyLjA1MDUsMy41ODI5LDQuMjA0NywyLjU2MTUsMS44NDU4LDIuNDYyOSwyLjYxMTMsNC4xODIyLDIuNDk4MV0sWzIuOTQzNiwzLjM4NTcsMi41MjA4LDMuOTczMywyLjQ3MywzLjA2MywyLjc1NTksMy44ODQzLDIuOTk2MywyLjc0MjcsMy43MjY3LDMuODA2Nl0sWzIuODc0NywyLjYxODgsMi41OTYxLDIuNzkzMywyLjM5MDksMi42MzI5LDIuODYzMSw2LjE4ODUsMi45MDQzLDIuNjA4MSwzLjAzNDMsMi41ODY1XSxbMi42NzI2LDIuMjg1LDIuNDU1LDEuOTM2OCwxLjY4OCwyLjE5NTQsMy4wMjc3LDIuODkwNywxLjk2MjYsMi4xODQyLDMuMTAyLDIuNjE5OF0sWzEuMzczMiwzLjYyNDcsMi4zMDg0LDIuNzY3MywzLjAxNDIsMy4zMzI4LDMuODE5MiwxLjg0MzUsMS41MDg4LDMuMzI5NywyLjk0NjQsMS42NDg1XV0sImxvZ190aGV0YV9hcnIiOltbMi44MjU5LDMuMjk3OCwyLjM5MTMsMi43OTI1LDEuODA2NywyLjY0MzQsMi4wMzIsMi4yMjYyLDIuNjc3LDIuNDU4MywyLjQ0NTQsMS4xODMxXSxbMi4xOTMzLDMuMzU5NywxLjY5OTcsMi4zMTM3LDIuMzkxNywxLjk4ODksMy4yNDY3LDIuNTQ3NCwzLjI3NDYsMy4yOTkzLDMuMzU0MSwxLjQ2NzhdLFszLjMyNDQsMy44NDE1LDIuNTM1NCwyLjE4NzIsMy4xMzIzLDIuNjg4NSw0LjQ2NCwyLjc4NCwyLjUyMTIsMi40MjA3LDEuOTA5OCwwLjgzMDJdLFszLjY4MjEsMi42NzA3LDIuMTM0MSwyLjQwNjIsMS4yNDYyLDIuMzUyMywyLjg4MTYsMi41ODY5LDMuOTQ4MiwyLjg5MSwyLjA3MTksMi41MjkzXSxbNC4xNzIxLDIuMTcxNSwzLjE0MiwyLjE1NDIsMS44MzM2LDMuOTQ2NywyLjM3NTUsMy40MTU5LDQuMTI1MywyLjYzMzYsMi4zNTYxLDEuMTQ5OV0sWzIuOTc0LDEuMjk3MywyLjY5OTUsMy4xMjI4LDEuOTI0MywyLjc2MTEsMS44MzE5LDMuODQ1MiwzLjIwMTYsMC44MDM3LDIuNjY3NSwwLjkzMTNdLFszLjYzNjQsMi41MzMyLDMuNDMzMiwzLjQzNzUsMi4xNTg4LDMuNTE5OSwyLjU2NTQsNC40MTg0LDQuOTY1MywyLjYwMzEsMi44MTExLDIuMjE4Nl0sWzIuNDkyNSwxLjYxMjMsMS44NzU2LDEuNzIsMS43MjI5LDIuMDY0MSwyLjE1NSwyLjY1NzEsMi40OTYyLDIuMTIzMiwyLjA0MzQsMi41OTc3XSxbMy4xODQsMS44ODcyLDEuNjQ1NywxLjY4ODIsMS4zMzY3LDIuMjM2OSwyLjY4NjMsMS43MjU2LDIuMTg4NywwLjc4NTcsMi4xMzAzLDEuNDI5OF0sWzMuNDMzOCwyLjM3NTUsMy44NTAzLDIuNzg2MiwwLjYwOTEsMy40OTg1LDIuNTk0NSwzLjc5ODQsMy4xMzcyLDIuNjUzMiwyLjUxNDksMi4wNDA3XSxbMy4zMzUyLDIuMzE4NSwyLjgxMDcsMS4xMTQ3LDIuNzEyNywyLjk3ODksMi44MjkyLDMuNTU1OCwzLjUyMjcsMi43OTU5LDMuNzE0NywyLjQ0NTddLFszLjI0MjIsMS45OTQ0LDIuOTE4OSwxLjQ1NTIsMS42NzE2LDMuMTI0OSwyLjQ2NDQsMi43NTQ5LDMuMDE3OSwxLjc3ODQsMy4xNDYxLDEuNTUxNF0sWzMuMDc4NiwxLjgzNywyLjQ5ODIsMy4xNjEzLDIuMTU5MiwzLjA1NTEsMi43OTA2LDMuMjc2NiwzLjE1ODcsMi4xMTI1LDIuNjA4NSwxLjMxNDZdLFsyLjY1NzEsMC43ODIsMi41NzI5LDIuNTczNCwwLjcyMzQsMi43Mzk5LDAuOTIzNCwyLjQzNjUsMy44NzY4LDIuMDcyNSwyLjQwMzUsMS45NDE0XSxbNC4xMjksMS44NjQ1LDMuMjY4MywzLjE4NDcsMS4zODQ4LDMuNzA0MSwxLjUwMjIsNC4xOTQ0LDIuNDQ2MywxLjgxNDgsMS44NDA0LDEuMTEzM10sWzIuNjUxMiwxLjQ0NTcsMi4yMTIxLDQuMTY3MSwyLjE4OTIsMi4yMzMyLDIuMzcyMywzLjIyNiw0LjM4NTEsMS4xMTQxLDIuOTEyMywxLjM0Nl0sWzMuMjE5NSwxLjg0MDksMi43NDIzLDIuMTA1NywyLjAxMDEsMy4xMjE4LDIuNDQxNywyLjQyMDYsMy4wNTQ0LDEuMDQwNCwyLjM2NzUsMS45MTY3XSxbMi4zOTExLDEuOTg5NCwyLjYzNTIsMi4xNjA4LDEuNjU1NywzLjU5MzgsMi4yMTMxLDEuODY0MSwyLjE0MTksMS44NTAyLDIuMDU2NiwyLjA2NTRdLFszLjMwMjYsMS44NTAxLDMuMzk0MiwxLjQ5ODIsMS42MzgzLDIuODA4OCwxLjk0MzQsMC43NzYyLDIuNzc0NCwxLjQzNSwxLjk5OTQsMi44MzIyXSxbMi45MDkxLDMuNjQ4LDIuNDgzNiwyLjMxODEsMS40OTE0LDIuNzUwNywzLjc5NCwyLjk0NzQsNC4xNzkzLDQuNzU5NywyLjkxOTksMS40NjE0XSxbMy4wNjE3LDEuNzQyNSwzLjIyNjgsMy41MjQ2LDEuNTY3NywzLjAzMTMsMi41ODM2LDMuMzk5NywzLjU2NTksMi4xNDEyLDIuNDIwNCwxLjc5MDVdLFszLjI1MTYsMC44NzMyLDIuMTYxNiwwLjk3MzUsMC42Njk2LDIuNDg3OSwxLjAyMTMsMS4xMDQ0LDMuMzMxLDIuNTQ4OCwxLjA5MDUsMS42ODMyXSxbMi45NTgzLDIuMDk4NiwyLjQxMTksMi4xMjg4LDEuNzU5MywyLjUzMTYsMi4yMjU3LDEuOTksMy4zNzc2LDIuNDY0OSwxLjkyODMsMi4yOTAyXSxbNC4wMzEyLDIuMjY2MSwzLjMwOTgsMi4xMTcxLDEuMzQ2NCwzLjcwNTUsMi45MTE1LDUuMDUxNyw1Ljg2ODgsMi4zODY3LDMuMDA3MSwxLjkxMjhdLFszLjM5MDgsMi4yNTksMy40MTQ0LDIuMDg5NiwxLjg0NTksMy40MDk3LDIuNzQyNCw0Ljg0NzIsMy41Mjg0LDIuNjU4MSwzLjM2MDMsMS44OTg1XSxbMi43OTc4LDEuNzYzOSwyLjQ4MDgsMS44MjkyLDIuMDQzNCwyLjYzODUsMi4xMDU0LDIuODk3OSwyLjcyMzYsMS4zMDAyLDIuMzc0LDIuMjY5NV0sWzMuMDMzOSwzLjI1NSwxLjk2ODksNC44NDczLC0wLjAwMjUsMi41NDE4LDEuNDQ0NSwxLjUxMDEsMi4zMTU0LDIuMjg4LDEuODU5LDAuMDMwN10sWzIuMzg4NywxLjU3MTEsMi4wMjc0LDEuNjkzMiwxLjQ3MiwyLjEwNzYsMS43MTc5LDIuNTU2MSwyLjE0NjQsMS4yNjE0LDEuMzU1NCwxLjYwMDldLFszLjU4MTYsMi40MDgxLDMuODY2NCwzLjAwNzEsMS44MTE1LDMuOTc1MywxLjk3MTQsMy44OTcxLDIuMjY1NywyLjU3MzUsMy40MjYzLDIuMTQzXSxbMy4xODMxLDIuMDkyOSwyLjY3MDksMi4wMDIyLDEuNjg1MSwzLjA1NDIsMi4wODM1LDIuMTk1MSwyLjg3NzEsMi41MTkzLDIuMTYzNywxLjg2OTJdLFsyLjk4NywyLjA3MywyLjI2MzUsMy4yNTY3LDIuMTY5NywyLjkwMDcsMi4zMjI4LDMuNDIxMSwzLjIwMzIsMi41NjExLDIuMzc3NSwxLjQxMzNdLFszLjE1MjksMi4zMjgsNC4yMTE5LDIuMDcxNiwyLjA4MzEsMy45MzM2LDIuNjExNiwyLjAzNjgsMi41MTksMi41MzI2LDIuOTU1MywyLjM5MjhdLFsyLjk5NTcsMi4wODg2LDIuMjIwNCwyLjQ3NTUsMS41MDQ1LDMuMDQ3LDMuNTQxNCwxLjg1MjYsMy40ODM0LDIuMjk3NSwzLjE1MDgsMS41ODU0XSxbMy4wMzc3LDEuNTIwMiwyLjkzNDksMS4yODAyLDEuNDQzNiwzLjQ4OTQsMS41NDQ0LDAuOTgwNiwyLjkwNDUsMi40NiwxLjU5ODgsMi43NTk1XV19"}
GUION = "#!/bin/bash\n# Eos v3 en Colab: SMIRK → fiabilidad → etiquetas → voz → articulación → boca + no verbal. Reanudable.\n# Variables: SOLO_PILOTO=1 (parar tras el piloto), N_PROC (procesos de etiquetado), CODIFICADOR (hubert-large|wavlm-large).\nset -x\nCB=/content/drive/MyDrive/literalevel_cara/casa_blanca\nV2C=/content/drive/MyDrive/literalevel_cara/voz2cara\nV3=$V2C/v3\nD=/content/datos\nS=/content/tools/cara_parlante\nN_PROC=${N_PROC:-4}\nCODIFICADOR=${CODIFICADOR:-hubert-large}\nmkdir -p $D/medidas/obama $D/cache $V3/smirk/obama $V3/smirk/1kh $V3/voz/obama $V3/voz/1kh $V3/art $V3/boca $V3/noverbal $CB/audio\npaso() { echo \"### $1\"; }\nfalla() { paso \"ERROR en $1\"; exit 1; }\n\npaso \"0 preparar\"\npip -q install timm mediapipe gdown imageio-ffmpeg faster-whisper g2p_en 2>&1 | tail -n 2\npython -c \"import nltk; [nltk.download(p, quiet=True) for p in ('averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger', 'cmudict')]\"\n[ -f $V3/SMIRK_em1.pt ] || gdown 1T65uEd9dVLHgVw5KiUYL66NUee-MCzoE -O $V3/SMIRK_em1.pt || falla \"descarga SMIRK\"\n[ -f $V3/face_landmarker.task ] || wget -q -O $V3/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task\ncp -n $CB/medidas/*.npz $D/medidas/obama/\ncd $D && unzip -q -n $V2C/1kh.zip && unzip -q -n $V2C/audio_1kh.zip && cd /content\n[ -f $V2C/1kh_videos.zip ] && { mkdir -p $D/1kh && cd $D/1kh && unzip -q -n $V2C/1kh_videos.zip && cd /content; }\nls $CB/videos/*.mp4 | xargs -P 8 -I{} bash -c 'o=$0/$(basename \"{}\" .mp4).m4a; [ -f \"$o\" ] || ffmpeg -loglevel error -y -i \"{}\" -vn -ac 1 -ar 16000 -c:a aac -b:a 48k \"$o\"' $CB/audio\n# GPU pequeña (T4): menos capas, lotes más cortos\nMEM=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n 1)\nif [ \"${MEM:-0}\" -lt 20000 ]; then EXTRA=\"--lote 16 --capas_usar 12,18,24\"; BOCA_EXTRA=\"--d 256 --capas 4\"; else EXTRA=\"--lote 48\"; BOCA_EXTRA=\"\"; fi\nSM=\"--checkpoint $V3/SMIRK_em1.pt --landmarker $V3/face_landmarker.task\"\n\npaso \"1 piloto SMIRK (1 h de Obama)\"\nLISTA=$(python - <<'EOF'\nimport glob, numpy as np, os\ns, out = 0.0, []\nfor f in sorted(glob.glob('/content/datos/medidas/obama/*.npz')):\n    t = np.load(f)['t']; out.append(os.path.splitext(os.path.basename(f))[0]); s += t[-1]\n    if s >= 3600: break\nprint(','.join(out))\nEOF\n)\npython -u $S/voz3/etiquetar_smirk.py --medidas $D/medidas/obama --videos $CB/videos --salida $V3/smirk/obama $SM --lista \"$LISTA\" || falla \"piloto SMIRK\"\npython -u $S/voz3/fiabilidad.py --smirk $V3/smirk/obama --medidas $D/medidas/obama --salida $V3/fiabilidad_piloto.json | tail -n 8\n[ \"${SOLO_PILOTO:-0}\" = \"1\" ] && { paso \"FIN (solo piloto)\"; exit 0; }\n\npaso \"2 etiquetado SMIRK completo\"\nfor i in $(seq 1 $N_PROC); do\n  python -u $S/voz3/etiquetar_smirk.py --medidas $D/medidas/obama --videos $CB/videos --salida $V3/smirk/obama $SM --parte $i/$N_PROC > /content/smirk_obama_$i.log 2>&1 &\ndone; wait\nif [ -d $D/1kh ]; then\n  for i in $(seq 1 $N_PROC); do\n    python -u $S/voz3/etiquetar_smirk.py --medidas $D/medidas --videos $D/1kh --salida $V3/smirk/1kh $SM --parte $i/$N_PROC > /content/smirk_1kh_$i.log 2>&1 &\n  done; wait\nfi\ngrep -h \"listo:\" /content/smirk_*.log; grep -h ERROR /content/smirk_*.log | head -n 5\n\npaso \"3 mapa SMIRK→ARKit\"\npython -u $S/voz3/mapa_arkit.py --smirk $V3/smirk --medidas $D/medidas --salida $V3/mapa_arkit.json || falla \"mapa\"\n\npaso \"4 rasgos de voz ($CODIFICADOR)\"\npython -u $S/voz3/extraer_v3.py --medidas $D/medidas/obama --audio $CB/audio --salida $V3/voz/obama --codificador $CODIFICADOR || falla \"voz obama\"\npython -u $S/voz3/extraer_v3.py --medidas $D/medidas --audio $D/audio_1kh --salida $V3/voz/1kh --codificador $CODIFICADOR || falla \"voz 1kh\"\n\npaso \"5 articulación de Blender (Whisper)\"\ncp -n $V2C/art_obama/*.npz $V2C/art_obama/*.json $V3/art/ 2>/dev/null\npython -u $S/voz2cara/preparar_art.py --medidas $D/medidas/obama --videos $CB/audio --salida $V3/art --whisper large-v3-turbo --modelo_art /content/modelo_articulatorio.json\npython -u $S/voz2cara/preparar_art.py --medidas $D/medidas --videos $D/audio_1kh --salida $V3/art --whisper large-v3-turbo --modelo_art /content/modelo_articulatorio.json\n\npaso \"6a entrenar boca\"\npython -u $S/voz3/entrenar_boca.py --voz $V3/voz --art $V3/art --smirk $V3/smirk --mapa $V3/mapa_arkit.json --salida $V3/boca \\\n  --modelo_art /content/modelo_articulatorio.json --cache $D/cache --epocas 300 --paciencia 30 $EXTRA $BOCA_EXTRA || falla \"boca\"\npaso \"6b entrenar no verbal\"\npython -u $S/voz3/entrenar_noverbal.py --voz $V3/voz --art $V3/art --smirk $V3/smirk --medidas $D/medidas --mapa $V3/mapa_arkit.json \\\n  --salida $V3/noverbal --cache $D/cache --epocas 300 --paciencia 30 $EXTRA || falla \"no verbal\"\npaso \"FIN\"\n"
# --- 2. Escribir los guiones y lanzar todo
SOLO_PILOTO = 0      # 1: parar tras el piloto de fiabilidad (1 h de Obama) y mirar fiabilidad_piloto.json
N_PROC = 4           # procesos de etiquetado SMIRK a la vez (MediaPipe va en CPU)
CODIFICADOR = 'hubert-large'   # o 'wavlm-large'
import base64, os, subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA')
for nombre, b64 in ARCHIVOS.items():
    os.makedirs(os.path.dirname('/content/' + nombre) or '/content', exist_ok=True)
    open('/content/' + nombre, 'wb').write(base64.b64decode(b64))
open('/content/v3.sh', 'w').write(GUION)
env = dict(os.environ, SOLO_PILOTO=str(SOLO_PILOTO), N_PROC=str(N_PROC), CODIFICADOR=CODIFICADOR)
subprocess.Popen(['setsid', 'bash', '-c', 'bash /content/v3.sh > /content/v3.log 2>&1'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True, env=env)
V3 = '/content/drive/MyDrive/literalevel_cara/voz2cara/v3'
os.makedirs(V3, exist_ok=True)
subprocess.Popen(['setsid', 'python', '/content/tools/cara_parlante/voz3/estado_v3.py', '--log', '/content/v3.log',
                  '--v3', V3, '--medidas', '/content/datos/medidas'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
print('lanzado: /content/v3.log y', V3 + '/estado.txt')

In [ ]:
# --- 3. Estado (vuelve a ejecutar esta celda con Ctrl+Enter)
!cat /content/drive/MyDrive/literalevel_cara/voz2cara/v3/estado.txt 2>/dev/null
!grep -a -E '^### |listo|por grupo|parada|Error|Traceback|Killed' /content/v3.log | tail -n 8